# Final Validation Notebook — Synchronized Surface ECG and Intracardiac Electrogram Classification

This notebook is a **standalone, resumable analysis** for the remaining validation questions in the study. It does **not** require outputs, checkpoints, caches, or notebooks from any earlier analysis version.

It starts from the raw Leipzig Heart Center WFDB dataset and writes only to:

`outputs_mechxformer_av_v3_final_validation/`

### Analyses included

1. **Nested patient-level reference model** with patient-level temperature scaling and decision-threshold selection.
2. **Source clinical-diagnosis concordance audit** when a diagnosis-bearing subject metadata file is available.
3. **Strict ECG-only and strict IEGM-only models**, followed by paired patient-level comparison with the matched multimodal reference model (permutation + exact McNemar, Holm-adjusted within the two-comparison family).
4. **Automatic ECG-based R-peak localization rebuilt from raw signals**, including XQRS/fallback provenance, 50/100/150-ms matching sensitivity, and patient-level localization summaries.
5. **Automatic-localization classification with and without annotation-derived pacing features**, using matched initialization and paired inference.
6. **Repeated controlled self-supervised-initialization versus random-initialization experiments** across multiple matched seed schedules.
7. **External-validation readiness check**. External performance is produced only when a genuinely independent synchronized cohort is supplied; otherwise the notebook records that external validation is unavailable rather than synthesizing a result.
8. Consolidated **CSV + Excel + Markdown** exports for manuscript reporting.

### Run instructions

- Set `MECHSSL_DATA_DIR` to the raw Leipzig dataset root if the default path is not correct.
- A CUDA GPU is strongly recommended for the neural experiments.
- The notebook is resumable: completed fold artifacts are reused unless `V3_FORCE_RERUN=True`.
- To restart a partially completed run, reopen this notebook and run all cells; completed fold outputs are detected automatically.
- No result in this notebook should be reported until the corresponding block finishes successfully.


## Core environment, dataset processing, model architecture, and reusable training utilities


In [1]:
# ============================================================
# 1) Environment setup
# ============================================================
from __future__ import annotations

import os
import re
import gc
import sys
import json
import time
import math
import copy
import random
import hashlib
import warnings
import platform
import subprocess
import shutil
import datetime as dt
from dataclasses import dataclass, asdict
from pathlib import Path
from functools import lru_cache
from fractions import Fraction
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.signal as sps
from scipy import stats
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

try:
    import wfdb
    from wfdb import processing
except Exception:
    wfdb = None
    processing = None
    print('WARNING: wfdb is not installed. Install with: pip install wfdb')

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    roc_curve,
    precision_recall_curve,
)

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
except Exception:
    pass

print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('WFDB:', getattr(wfdb, '__version__', 'not installed'))


Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
NumPy: 1.26.4
Pandas: 2.3.3
Torch: 2.12.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
WFDB: 4.3.1


In [2]:
# ============================================================
# 2) Configuration
# ============================================================
ECG_LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
IEGM_CHANNELS = ['CS12', 'CS34', 'CS56', 'CS78', 'CS90', 'RVA12', 'ABL12']
BINARY_LABELS = ['AVNRT', 'AVRT']  # AVNRT=0, AVRT=1
OTHER_RHYTHMS = ['AFIB', 'AFL', 'EAT', 'AT', 'VT', 'IVR', 'N', 'A', 'B', 'J', '/A', '/V']

@dataclass
class CFG:
    # Required local dataset path.
    data_dir: str = os.environ.get('MECHSSL_DATA_DIR', r'C:\Users\hasee\Desktop/leipzig-heart-centerwith-congenital-heart-disease-1.0.0')

    # Output folders.
    out_dir: str = './outputs_mechxformer_av_v3_final_validation'
    cache_dir: str = './outputs_mechxformer_av_v3_final_validation/cache'
    model_dir: str = './outputs_mechxformer_av_v3_final_validation/models'
    figure_dir: str = './outputs_mechxformer_av_v3_final_validation/figures'
    table_dir: str = './outputs_mechxformer_av_v3_final_validation/tables'
    log_dir: str = './outputs_mechxformer_av_v3_final_validation/logs'

    # Signal/window settings.
    target_fs: int = 500
    window_sec: float = 10.0
    stride_sec: float = 5.0
    min_label_fraction: float = 0.50
    pre_r_sec: float = 0.20
    post_r_sec: float = 0.40
    beat_len: int = 160
    max_beats: int = 40
    rpeak_source: str = 'annotation_first'
    use_bandpass: bool = True
    bandpass_low_hz: float = 0.5
    bandpass_high_hz: float = 100.0
    notch_hz: Optional[float] = None

    # Cross-validation.
    seed: int = 42
    n_outer_splits: int = 5
    n_inner_splits: int = 4
    max_records: Optional[int] = None
    max_all_windows: Optional[int] = None      # set for smoke tests only
    max_binary_windows: Optional[int] = None   # set for smoke tests only

    # Transformer size deliberately small for small-N data.
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    d_model: int = 96
    n_heads: int = 4
    n_layers: int = 1
    dropout: float = 0.40
    stochastic_depth: float = 0.05

    # Window model training.
    batch_size: int = 8
    num_workers: int = 0
    epochs_window: int = 50
    epochs_mil: int = 80
    patience: int = 15
    lr_head: float = 3e-4
    lr_encoder: float = 1e-5
    lr_scratch: float = 1e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    use_amp: bool = True

    # Self-supervised pretraining.
    use_ssl_pretraining: bool = True
    ssl_epochs: int = 25
    ssl_lr: float = 1e-4
    ssl_batch_size: int = 8
    ssl_mask_patch_p: float = 0.35
    ssl_mask_channel_p: float = 0.12
    ssl_patch_len: int = 16
    lambda_ssl_recon: float = 1.0
    lambda_ssl_cross_recon: float = 0.25
    lambda_ssl_modal_contrast: float = 0.05
    ssl_overwrite: bool = False

    # Record-level MIL.
    max_windows_per_record_train: int = 48
    max_windows_per_record_eval: int = 96
    record_batch_size: int = 2
    window_aux_lambda: float = 0.20
    supcon_lambda: float = 0.08
    prototype_lambda: float = 0.06
    contrastive_temperature: float = 0.20
    prototype_warmup_epochs: int = 3

    # Loss/calibration.
    label_smoothing: float = 0.05
    use_focal_loss: bool = True
    focal_gamma: float = 1.5
    tune_threshold: bool = True
    use_temperature_scaling: bool = True
    use_tta: bool = True
    n_tta: int = 4

    # Augmentation.
    use_augmentation: bool = True
    aug_noise_std: float = 0.015
    aug_scale_std: float = 0.10
    aug_time_shift_ms: int = 20
    aug_lead_dropout_p: float = 0.04
    aug_beat_dropout_p: float = 0.03

    # Reporting/statistics.
    n_bootstrap: int = 1000
    n_permutations: int = 5000
    confidence_level: float = 0.95
    primary_metric: str = 'record_macro_f1'
    save_bootstrap_distributions: bool = False


# Runtime switches. Edit these before running the notebook.
SMOKE_TEST = False
RUN_MAIN_EXPERIMENT = False
RUN_REPEATED_CV = False
RUN_CORE_SENSITIVITY = False
RUN_FULL_SENSITIVITY = False
RUN_RPEAK_DEPLOYABILITY_EXPERIMENT = False
RUN_INTERPRETABILITY_EXPORTS = False

cfg = CFG()
# Resolve the raw Leipzig dataset from an environment variable or common notebook locations.
_data_candidates = [
    os.environ.get('MECHSSL_DATA_DIR', ''),
    cfg.data_dir,
    '/content/leipzig-heart-centerwith-congenital-heart-disease-1.0.0',
    '/content/leipzig-heart-center-ecg-1.0.0',
    '/kaggle/input/leipzig-heart-centerwith-congenital-heart-disease-1.0.0',
]
for _cand in _data_candidates:
    if _cand and Path(_cand).expanduser().exists():
        cfg.data_dir = str(Path(_cand).expanduser())
        break
if SMOKE_TEST:
    cfg.max_records = 6
    cfg.max_all_windows = 300
    cfg.max_binary_windows = 200
    cfg.n_outer_splits = 2
    cfg.n_inner_splits = 2
    cfg.ssl_epochs = 1
    cfg.epochs_window = 1
    cfg.epochs_mil = 1
    cfg.n_bootstrap = 50
    cfg.n_permutations = 100
    cfg.use_tta = False
for p in [cfg.out_dir, cfg.cache_dir, cfg.model_dir, cfg.figure_dir, cfg.table_dir, cfg.log_dir]:
    Path(p).mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2))


{
  "data_dir": "C:\\Users\\hasee\\Desktop\\leipzig-heart-centerwith-congenital-heart-disease-1.0.0",
  "out_dir": "./outputs_mechxformer_av_v3_final_validation",
  "cache_dir": "./outputs_mechxformer_av_v3_final_validation/cache",
  "model_dir": "./outputs_mechxformer_av_v3_final_validation/models",
  "figure_dir": "./outputs_mechxformer_av_v3_final_validation/figures",
  "table_dir": "./outputs_mechxformer_av_v3_final_validation/tables",
  "log_dir": "./outputs_mechxformer_av_v3_final_validation/logs",
  "target_fs": 500,
  "window_sec": 10.0,
  "stride_sec": 5.0,
  "min_label_fraction": 0.5,
  "pre_r_sec": 0.2,
  "post_r_sec": 0.4,
  "beat_len": 160,
  "max_beats": 40,
  "rpeak_source": "annotation_first",
  "use_bandpass": true,
  "bandpass_low_hz": 0.5,
  "bandpass_high_hz": 100.0,
  "notch_hz": null,
  "seed": 42,
  "n_outer_splits": 5,
  "n_inner_splits": 4,
  "max_records": null,
  "max_all_windows": null,
  "max_binary_windows": null,
  "device": "cuda",
  "d_model": 96,
  "n_

In [3]:
# ============================================================
# 3) Reproducibility helpers
# ============================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

seed_everything(cfg.seed)

def stable_hash(obj: Any) -> str:
    s = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))

save_json(asdict(cfg), Path(cfg.log_dir) / 'configuration.json')


def environment_snapshot() -> Dict[str, Any]:
    """Collect a lightweight reproducibility snapshot for the run manifest."""
    snap = {
        'timestamp_utc': dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'python': sys.version,
        'platform': platform.platform(),
        'processor': platform.processor(),
        'cpu_count': os.cpu_count(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'torch': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'wfdb': getattr(wfdb, '__version__', None),
    }
    try:
        snap['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        snap['git_commit'] = None
    return snap

def configuration_hash(cfg_obj: CFG) -> str:
    return stable_hash(asdict(cfg_obj))

save_json(environment_snapshot(), Path(cfg.log_dir) / 'environment_summary.json')
(Path(cfg.log_dir) / 'configuration_hash.txt').write_text(configuration_hash(cfg), encoding='utf-8')


16

In [4]:
# ============================================================
# 4) Dataset discovery and annotation parsing
# ============================================================
def ensure_dataset(data_dir: str) -> Tuple[Path, List[str]]:
    if wfdb is None:
        raise ImportError('wfdb is required. Install with: pip install wfdb')
    data_path = Path(data_dir).expanduser().resolve()
    if not data_path.exists():
        raise FileNotFoundError(
            f'Dataset folder not found: {data_path}\n'
            'Set cfg.data_dir to the Leipzig WFDB root folder containing RECORDS, .hea, .dat, and .atr files.'
        )
    records_file = data_path / 'RECORDS'
    if records_file.exists():
        record_ids = [r.strip() for r in records_file.read_text().splitlines() if r.strip()]
    else:
        hea_files = sorted(data_path.rglob('*.hea'))
        if not hea_files:
            raise FileNotFoundError(f'No RECORDS file and no .hea files found under {data_path}')
        record_ids = [str(p.relative_to(data_path).with_suffix('')).replace(os.sep, '/') for p in hea_files]
        records_file.write_text('\n'.join(record_ids) + '\n')
    if cfg.max_records is not None:
        record_ids = record_ids[:cfg.max_records]
    missing = []
    for rid in record_ids:
        for ext in ('hea', 'dat', 'atr'):
            fp = data_path / f'{rid}.{ext}'
            if not fp.exists() or fp.stat().st_size == 0:
                missing.append(str(fp))
    if missing:
        raise FileNotFoundError('Missing WFDB files; examples:\n' + '\n'.join(missing[:20]))
    print(f'Dataset root: {data_path}')
    print(f'Records discovered: {len(record_ids)}')
    return data_path, record_ids

DATA_DIR, RECORD_IDS = ensure_dataset(cfg.data_dir)

@lru_cache(maxsize=256)
def rdheader_cached(rid: str):
    return wfdb.rdheader(str(DATA_DIR / rid))

@lru_cache(maxsize=256)
def rdann_cached(rid: str):
    return wfdb.rdann(str(DATA_DIR / rid), 'atr')

def cohort_from_record(rid: str) -> str:
    m = re.search(r'(\d+)', str(rid))
    if not m:
        return 'unknown'
    return 'adult_chd' if int(m.group(1)) >= 100 else 'pediatric'

def clean_signal_name(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9]', '', str(name)).upper()

def signal_map(sig_names: Sequence[str], desired_names: Sequence[str]) -> Tuple[List[Optional[int]], np.ndarray]:
    cleaned = [clean_signal_name(s) for s in sig_names]
    idxs: List[Optional[int]] = []
    present: List[bool] = []
    for desired in desired_names:
        d = clean_signal_name(desired)
        found = None
        for i, c in enumerate(cleaned):
            if c == d:
                found = i
                break
        if found is None and d.startswith('CS'):
            for i, c in enumerate(cleaned):
                if d in c:
                    found = i
                    break
        if found is None and d == 'RVA12':
            for i, c in enumerate(cleaned):
                if 'RVA' in c or c.startswith('RV'):
                    found = i
                    break
        if found is None and d == 'ABL12':
            for i, c in enumerate(cleaned):
                if 'ABL' in c or 'MAP' in c:
                    found = i
                    break
        idxs.append(found)
        present.append(found is not None)
    return idxs, np.asarray(present, dtype=bool)

def normalize_aux(aux: Any) -> str:
    if aux is None:
        return ''
    return str(aux).strip().replace('\x00', '')

def rhythm_label_from_aux(aux: Any) -> Optional[int]:
    a = normalize_aux(aux).upper()
    if not a:
        return None
    if 'AVNRT' in a:
        return 0
    if 'AVRT' in a:
        return 1
    return None

def rhythm_family_from_aux(aux: Any) -> str:
    a = normalize_aux(aux).upper().replace('(', '').replace(')', '')
    if not a:
        return 'NONE'
    if 'AVNRT' in a:
        return 'AVNRT'
    if 'AVRT' in a:
        return 'AVRT'
    for tok in OTHER_RHYTHMS:
        if tok and tok in a:
            return tok
    return a

def is_rhythm_marker(symbol: str, aux: Any) -> bool:
    aux_s = normalize_aux(aux)
    return str(symbol).strip() == '+' or aux_s.startswith('(')

def is_beat_annotation(symbol: str) -> bool:
    return str(symbol).strip() not in {'+', '~', '|'}

def is_paced_annotation(symbol: str, aux: Any) -> bool:
    a = normalize_aux(aux).upper()
    s = str(symbol).strip()
    return (s == '/') or ('/A' in a) or ('/V' in a) or ('PACE' in a) or ('PACED' in a)

def rhythm_intervals_all(rid: str) -> List[Tuple[int, int, str, Optional[int], str]]:
    h = rdheader_cached(rid)
    ann = rdann_cached(rid)
    sig_len = int(h.sig_len)
    points = []
    for sample, symbol, aux in zip(ann.sample, ann.symbol, ann.aux_note):
        if is_rhythm_marker(symbol, aux):
            fam = rhythm_family_from_aux(aux)
            lab = rhythm_label_from_aux(aux)
            points.append((int(sample), fam, lab, normalize_aux(aux)))
    points.sort(key=lambda x: x[0])
    intervals = []
    for i, (start, fam, lab, aux_s) in enumerate(points):
        end = points[i + 1][0] if i + 1 < len(points) else sig_len
        if end > start:
            intervals.append((start, end, fam, lab, aux_s))
    return intervals

def pacing_fraction_for_window(rid: str, start: int, end: int) -> float:
    ann = rdann_cached(rid)
    total = 0
    paced = 0
    for sample, symbol, aux in zip(ann.sample, ann.symbol, ann.aux_note):
        s = int(sample)
        if start <= s < end and is_beat_annotation(symbol):
            total += 1
            paced += int(is_paced_annotation(symbol, aux))
    return float(paced / max(total, 1))


Dataset root: C:\Users\hasee\Desktop\leipzig-heart-centerwith-congenital-heart-disease-1.0.0
Records discovered: 39


In [5]:
# ============================================================
# 5) Cohort audit and all-window/binary-window indices
# ============================================================
def record_inventory(record_ids: Sequence[str]) -> pd.DataFrame:
    rows = []
    for rid in tqdm(record_ids, desc='Auditing records'):
        try:
            h = rdheader_cached(rid)
            ann = rdann_cached(rid)
            ecg_idxs, ecg_present = signal_map(h.sig_name, ECG_LEADS)
            iegm_idxs, iegm_present = signal_map(h.sig_name, IEGM_CHANNELS)
            seconds_by_family: Dict[str, float] = {}
            for a, b, fam, lab, aux_s in rhythm_intervals_all(rid):
                seconds_by_family[fam] = seconds_by_family.get(fam, 0.0) + (b - a) / float(h.fs)
            n_beats = sum(is_beat_annotation(s) for s in ann.symbol)
            n_paced = sum(is_beat_annotation(s) and is_paced_annotation(s, aux) for s, aux in zip(ann.symbol, ann.aux_note))
            rows.append({
                'record': rid,
                'cohort_group': cohort_from_record(rid),
                'fs': float(h.fs),
                'duration_sec': float(h.sig_len) / float(h.fs),
                'n_channels_raw': int(h.n_sig),
                'n_beats': int(n_beats),
                'paced_fraction_beats': float(n_paced / max(n_beats, 1)),
                'n_ecg_present': int(ecg_present.sum()),
                'n_iegm_present': int(iegm_present.sum()),
                'all_ecg_present': bool(ecg_present.all()),
                'all_iegm_present': bool(iegm_present.all()),
                'sec_AVNRT': seconds_by_family.get('AVNRT', 0.0),
                'sec_AVRT': seconds_by_family.get('AVRT', 0.0),
                'sec_other_rhythm': sum(v for k, v in seconds_by_family.items() if k not in {'AVNRT', 'AVRT'}),
                'rhythm_seconds_json': json.dumps(seconds_by_family, sort_keys=True),
            })
        except Exception as e:
            rows.append({'record': rid, 'error': repr(e)})
    return pd.DataFrame(rows)

def overlap_summary_for_window(rid: str, start: int, end: int) -> Dict[str, Any]:
    win_len = max(1, end - start)
    fam_overlap: Dict[str, int] = {}
    lab_overlap = {0: 0, 1: 0}
    other_overlap = 0
    for a, b, fam, lab, aux_s in rhythm_intervals_all(rid):
        ov = max(0, min(end, b) - max(start, a))
        if ov <= 0:
            continue
        fam_overlap[fam] = fam_overlap.get(fam, 0) + ov
        if lab in (0, 1):
            lab_overlap[int(lab)] += ov
        else:
            other_overlap += ov
    if fam_overlap:
        dominant_family = max(fam_overlap, key=fam_overlap.get)
        dominant_family_fraction = fam_overlap[dominant_family] / win_len
    else:
        dominant_family = 'UNLABELED'
        dominant_family_fraction = 0.0
    best_lab = max(lab_overlap, key=lab_overlap.get)
    best_frac = lab_overlap[best_lab] / win_len
    label = int(best_lab) if best_frac >= cfg.min_label_fraction else -1
    label_name = BINARY_LABELS[label] if label in (0, 1) else 'NON_BINARY_OR_UNLABELED'
    return {
        'label': label,
        'label_name': label_name,
        'frac_AVNRT': lab_overlap[0] / win_len,
        'frac_AVRT': lab_overlap[1] / win_len,
        'frac_other_or_unlabeled': other_overlap / win_len,
        'dominant_fraction': float(best_frac),
        'ssl_family': dominant_family,
        'ssl_family_fraction': float(dominant_family_fraction),
    }

def build_all_window_index(record_ids: Sequence[str]) -> pd.DataFrame:
    rows = []
    for rid in tqdm(record_ids, desc='Indexing all windows for SSL and binary supervision'):
        try:
            h = rdheader_cached(rid)
            fs = float(h.fs)
            sig_len = int(h.sig_len)
            win = int(round(cfg.window_sec * fs))
            stride = int(round(cfg.stride_sec * fs))
            if sig_len < win:
                continue
            ecg_idxs, ecg_present = signal_map(h.sig_name, ECG_LEADS)
            iegm_idxs, iegm_present = signal_map(h.sig_name, IEGM_CHANNELS)
            for start in range(0, sig_len - win + 1, stride):
                end = start + win
                lab_meta = overlap_summary_for_window(rid, start, end)
                rows.append({
                    'record': rid,
                    'start': int(start),
                    'end': int(end),
                    'orig_fs': fs,
                    'pace_frac': float(pacing_fraction_for_window(rid, start, end)),
                    'cohort_group': cohort_from_record(rid),
                    'n_ecg_present': int(ecg_present.sum()),
                    'n_iegm_present': int(iegm_present.sum()),
                    'all_iegm_present': bool(iegm_present.all()),
                    **lab_meta,
                })
        except Exception as e:
            print(f'WARNING: skipping {rid}: {repr(e)}')
    df = pd.DataFrame(rows).reset_index(drop=True)
    if cfg.max_all_windows is not None and len(df) > cfg.max_all_windows:
        df = df.sample(cfg.max_all_windows, random_state=cfg.seed).sort_values(['record', 'start']).reset_index(drop=True)
    df['row_id'] = np.arange(len(df), dtype=int)
    return df

inventory_path = Path(cfg.table_dir) / 'dataset_inventory.csv'
if inventory_path.exists():
    df_inventory = pd.read_csv(inventory_path)
else:
    df_inventory = record_inventory(RECORD_IDS)
    df_inventory.to_csv(inventory_path, index=False)

all_index_path = Path(cfg.table_dir) / 'all_window_index_ssl_and_binary.csv'
if all_index_path.exists():
    df_all = pd.read_csv(all_index_path)
else:
    df_all = build_all_window_index(RECORD_IDS)
    df_all.to_csv(all_index_path, index=False)

df_binary = df_all[df_all['label'].isin([0, 1])].copy().reset_index(drop=True)
if cfg.max_binary_windows is not None and len(df_binary) > cfg.max_binary_windows:
    per_class = max(1, cfg.max_binary_windows // 2)
    df_binary = (df_binary.groupby('label', group_keys=False)
                 .apply(lambda x: x.sample(min(len(x), per_class), random_state=cfg.seed))
                 .sample(frac=1, random_state=cfg.seed).reset_index(drop=True))

print('All SSL candidate windows:', len(df_all), 'records:', df_all['record'].nunique())
print('Binary supervised windows:', len(df_binary), 'records:', df_binary['record'].nunique())
print(pd.crosstab(df_binary['label_name'], df_binary['cohort_group'], margins=True))
display(df_inventory.head())
display(df_binary.head())


Auditing records:   0%|          | 0/39 [00:00<?, ?it/s]

Indexing all windows for SSL and binary supervision:   0%|          | 0/39 [00:00<?, ?it/s]

All SSL candidate windows: 13228 records: 39
Binary supervised windows: 1410 records: 28
cohort_group  pediatric   All
label_name                   
AVNRT               537   537
AVRT                873   873
All                1410  1410


,record,cohort_group,fs,duration_sec,n_channels_raw,n_beats,paced_fraction_beats,n_ecg_present,n_iegm_present,all_ecg_present,all_iegm_present,sec_AVNRT,sec_AVRT,sec_other_rhythm,rhythm_seconds_json
0,x001,pediatric,977.0,8729.200614,19,15990,0.077423,12,7,True,True,0.000000,712.747185,7920.259980,"{""/V"": 81.62436028659161, ""A"": 1405.2998976458..."
1,x002,pediatric,977.0,6317.809621,18,10604,0.071200,12,6,True,False,878.821904,0.000000,5438.562948,"{""/V"": 74.12180143295805, ""A"": 362.99692937563..."
2,x003,pediatric,977.0,9031.749232,19,15248,0.096865,12,7,True,True,0.000000,108.241556,8923.190379,"{""/V"": 414.77993858751285, ""A"": 338.9631525076..."
3,x004,pediatric,977.0,8272.425793,19,14516,0.225200,12,7,True,True,0.000000,844.848516,7427.518936,"{""/V"": 608.439099283521, ""A"": 982.450358239508..."
4,x005,pediatric,977.0,1856.168884,18,2254,0.000000,12,6,True,False,0.000000,0.000000,1855.519959,"{""N"": 1855.5199590583418}"


,record,start,end,orig_fs,pace_frac,cohort_group,n_ecg_present,n_iegm_present,all_iegm_present,label,label_name,frac_AVNRT,frac_AVRT,frac_other_or_unlabeled,dominant_fraction,ssl_family,ssl_family_fraction,row_id
0,x001,3360880,3370650,977.0,0.0,pediatric,12,7,True,1,AVRT,0.0,0.700102,0.299898,0.700102,AVRT,0.700102,688
1,x001,3365765,3375535,977.0,0.0,pediatric,12,7,True,1,AVRT,0.0,1.000000,0.000000,1.000000,AVRT,1.000000,689
2,x001,3370650,3380420,977.0,0.0,pediatric,12,7,True,1,AVRT,0.0,1.000000,0.000000,1.000000,AVRT,1.000000,690
3,x001,3375535,3385305,977.0,0.0,pediatric,12,7,True,1,AVRT,0.0,1.000000,0.000000,1.000000,AVRT,1.000000,691
4,x001,3380420,3390190,977.0,0.0,pediatric,12,7,True,1,AVRT,0.0,1.000000,0.000000,1.000000,AVRT,1.000000,692


In [6]:
# ============================================================
# 6) Patient-wise folds for binary subset
# ============================================================
def assign_patientwise_folds_checked(df: pd.DataFrame, n_splits: int, seed: int, max_tries: int = 500) -> pd.DataFrame:
    if df.empty:
        raise ValueError('Cannot split empty dataframe.')
    classes = sorted(df['label'].unique().tolist())
    n_groups = df['record'].nunique()
    n_splits_eff = max(2, min(n_splits, n_groups))
    best_df = None
    best_score = -1e18
    for s in range(seed, seed + max_tries):
        try:
            cand = df.copy().reset_index(drop=True)
            sgkf = StratifiedGroupKFold(n_splits=n_splits_eff, shuffle=True, random_state=s)
            cand['fold'] = -1
            y = cand['label'].values
            groups = cand['record'].values
            for fold, (_, va_idx) in enumerate(sgkf.split(cand, y, groups)):
                cand.loc[va_idx, 'fold'] = fold
            ct = pd.crosstab(cand['fold'], cand['label']).reindex(index=range(n_splits_eff), columns=classes, fill_value=0)
            min_count = int(ct.min().min())
            nonzero = int((ct > 0).sum().sum())
            record_balance = cand.groupby('fold')['record'].nunique().std(ddof=0)
            score = min_count * 10000 + nonzero * 100 - float(record_balance)
            if score > best_score:
                best_df = cand.copy()
                best_score = score
            if (ct > 0).all().all():
                print(f'Using fold seed={s}')
                print(ct)
                return cand
        except Exception:
            continue
    print('WARNING: perfect class coverage per fold not found; using best available split.')
    if best_df is None:
        raise RuntimeError('Failed to create patient-wise folds.')
    print(pd.crosstab(best_df['fold'], best_df['label']).reindex(columns=classes, fill_value=0))
    return best_df

def make_inner_split(outer_train_df: pd.DataFrame, seed: int, n_inner_splits: int = 4) -> Tuple[pd.DataFrame, pd.DataFrame]:
    outer_train_df = outer_train_df.reset_index(drop=True)
    y = outer_train_df['label'].values
    groups = outer_train_df['record'].values
    n_groups = outer_train_df['record'].nunique()
    classes = sorted(np.unique(y).tolist())
    if n_groups < 3:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
        tr_idx, va_idx = next(splitter.split(outer_train_df, y, groups))
        return outer_train_df.iloc[tr_idx].reset_index(drop=True), outer_train_df.iloc[va_idx].reset_index(drop=True)
    n_splits = max(2, min(n_inner_splits, n_groups))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best = None
    best_score = -1e18
    for tr_idx, va_idx in sgkf.split(outer_train_df, y, groups):
        val_y = y[va_idx]
        nonzero = sum(int((val_y == c).sum() > 0) for c in classes)
        n_val_records = len(np.unique(groups[va_idx]))
        score = nonzero * 100 + n_val_records
        if score > best_score:
            best = (tr_idx, va_idx)
            best_score = score
    tr_idx, va_idx = best
    return outer_train_df.iloc[tr_idx].reset_index(drop=True), outer_train_df.iloc[va_idx].reset_index(drop=True)

df_binary = assign_patientwise_folds_checked(df_binary, cfg.n_outer_splits, cfg.seed)
df_binary.to_csv(Path(cfg.table_dir) / 'locked_patientwise_folds_binary.csv', index=False)
print(pd.crosstab(df_binary['fold'], df_binary['label_name'], margins=True))

for f in sorted(df_binary.fold.unique()):
    test_records = set(df_binary.loc[df_binary.fold == f, 'record'])
    train_records = set(df_binary.loc[df_binary.fold != f, 'record'])
    assert test_records.isdisjoint(train_records), 'Leakage: patient appears in train and test.'


Using fold seed=42
label    0    1
fold           
0       84   71
1      132  154
2       15   70
3      202  265
4      104  313
label_name  AVNRT  AVRT   All
fold                         
0              84    71   155
1             132   154   286
2              15    70    85
3             202   265   467
4             104   313   417
All           537   873  1410


In [7]:
# ============================================================
# 7) Signal preprocessing and tokenization
# ============================================================
def robust_zscore(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    med = np.nanmedian(x, axis=1, keepdims=True)
    q25 = np.nanpercentile(x, 25, axis=1, keepdims=True)
    q75 = np.nanpercentile(x, 75, axis=1, keepdims=True)
    scale = (q75 - q25) / 1.349
    std = np.nanstd(x, axis=1, keepdims=True)
    scale = np.where(scale > eps, scale, std)
    scale = np.where(scale > eps, scale, 1.0)
    out = (x - med) / scale
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def maybe_filter(x: np.ndarray, fs: int) -> np.ndarray:
    y = np.asarray(x, dtype=np.float32)
    if cfg.use_bandpass:
        high = min(float(cfg.bandpass_high_hz), fs / 2.0 - 1.0)
        low = max(float(cfg.bandpass_low_hz), 0.01)
        if high > low:
            try:
                sos = sps.butter(3, [low, high], btype='bandpass', fs=fs, output='sos')
                y = sps.sosfiltfilt(sos, y, axis=1).astype(np.float32)
            except Exception as e:
                print('Filter warning:', repr(e))
    if cfg.notch_hz is not None and 0 < cfg.notch_hz < fs / 2.0:
        try:
            b, a = sps.iirnotch(w0=cfg.notch_hz, Q=30, fs=fs)
            y = sps.filtfilt(b, a, y, axis=1).astype(np.float32)
        except Exception as e:
            print('Notch warning:', repr(e))
    return y.astype(np.float32)

def resample_to_target(x: np.ndarray, orig_fs: float, target_fs: int, target_len: int) -> np.ndarray:
    if int(round(orig_fs)) == int(target_fs):
        y = x.astype(np.float32)
    else:
        frac = Fraction(float(target_fs) / float(orig_fs)).limit_denominator(5000)
        y = sps.resample_poly(x, up=frac.numerator, down=frac.denominator, axis=1).astype(np.float32)
    if y.shape[1] < target_len:
        y = np.pad(y, ((0, 0), (0, target_len - y.shape[1])), mode='constant')
    elif y.shape[1] > target_len:
        y = y[:, :target_len]
    return y.astype(np.float32)

def load_window_streams(rid: str, start: int, end: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float, np.ndarray, np.ndarray]:
    h = rdheader_cached(rid)
    orig_fs = float(h.fs)
    target_len = int(round(cfg.window_sec * cfg.target_fs))
    rec = wfdb.rdrecord(str(DATA_DIR / rid), sampfrom=int(start), sampto=int(end), physical=True, return_res=32)
    sig = np.asarray(rec.p_signal, dtype=np.float32).T
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    sig = resample_to_target(sig, orig_fs, cfg.target_fs, target_len)
    sig_filt = maybe_filter(sig, cfg.target_fs)

    ecg_idxs, ecg_present = signal_map(rec.sig_name, ECG_LEADS)
    iegm_idxs, iegm_present = signal_map(rec.sig_name, IEGM_CHANNELS)
    ecg = np.zeros((len(ECG_LEADS), target_len), dtype=np.float32)
    iegm = np.zeros((len(IEGM_CHANNELS), target_len), dtype=np.float32)
    ecg_raw = np.zeros_like(ecg)
    iegm_raw = np.zeros_like(iegm)
    for j, src in enumerate(ecg_idxs):
        if src is not None and src < sig.shape[0]:
            ecg_raw[j] = sig[src]
            ecg[j] = sig_filt[src]
    for j, src in enumerate(iegm_idxs):
        if src is not None and src < sig.shape[0]:
            iegm_raw[j] = sig[src]
            iegm[j] = sig_filt[src]
    return ecg, iegm, ecg_present.astype(bool), iegm_present.astype(bool), orig_fs, ecg_raw, iegm_raw

def annotation_rpeaks_for_window(rid: str, start: int, end: int, orig_fs: float, target_fs: int) -> np.ndarray:
    ann = rdann_cached(rid)
    peaks = []
    for sample, symbol in zip(ann.sample, ann.symbol):
        s = int(sample)
        if start <= s < end and is_beat_annotation(symbol):
            peaks.append(int(round((s - start) * target_fs / float(orig_fs))))
    return np.asarray(sorted(set(peaks)), dtype=np.int64)

def detect_xqrs_rpeaks(ecg_scaled: np.ndarray, fs: int, ecg_present: Optional[np.ndarray] = None) -> np.ndarray:
    preferred = [ECG_LEADS.index('II'), ECG_LEADS.index('V5'), ECG_LEADS.index('V2'), ECG_LEADS.index('I')]
    if ecg_present is not None:
        candidates = [i for i in preferred if i < len(ecg_present) and bool(ecg_present[i])]
        candidates += [i for i, p in enumerate(ecg_present) if p and i not in candidates]
    else:
        candidates = preferred + list(range(ecg_scaled.shape[0]))
    lead_idx = candidates[0] if candidates else 0
    sig = np.nan_to_num(np.asarray(ecg_scaled[lead_idx], dtype=np.float32), nan=0.0)
    peaks = np.asarray([], dtype=np.int64)
    if processing is not None:
        try:
            peaks = processing.xqrs_detect(sig=sig, fs=fs, verbose=False).astype(np.int64)
        except TypeError:
            try:
                peaks = processing.xqrs_detect(sig=sig, fs=fs).astype(np.int64)
            except Exception:
                pass
        except Exception:
            pass
    if len(peaks) < 2:
        try:
            distance = int(max(0.20 * fs, 1))
            prominence = max(0.5, float(np.nanstd(sig)) * 0.35)
            peaks, _ = sps.find_peaks(sig, distance=distance, prominence=prominence)
            peaks = peaks.astype(np.int64)
        except Exception:
            peaks = np.asarray([], dtype=np.int64)
    return peaks

def choose_rpeaks(rid: str, start: int, end: int, orig_fs: float, ecg_scaled: np.ndarray, ecg_present: np.ndarray) -> np.ndarray:
    ann_peaks = annotation_rpeaks_for_window(rid, start, end, orig_fs, cfg.target_fs)
    xqrs_peaks = detect_xqrs_rpeaks(ecg_scaled, cfg.target_fs, ecg_present)
    if cfg.rpeak_source == 'annotation_first':
        peaks = ann_peaks if len(ann_peaks) > 0 else xqrs_peaks
    elif cfg.rpeak_source == 'xqrs_first':
        peaks = xqrs_peaks if len(xqrs_peaks) > 0 else ann_peaks
    elif cfg.rpeak_source == 'xqrs_only':
        peaks = xqrs_peaks
    else:
        raise ValueError(f'Unknown rpeak_source: {cfg.rpeak_source}')
    pre = int(round(cfg.pre_r_sec * cfg.target_fs))
    post = int(round(cfg.post_r_sec * cfg.target_fs))
    target_len = int(round(cfg.window_sec * cfg.target_fs))
    peaks = [int(p) for p in peaks if pre <= int(p) < target_len - post]
    return np.asarray(sorted(set(peaks)), dtype=np.int64)

def resize_snippet(snippet: np.ndarray, target_len: int) -> np.ndarray:
    if snippet.shape[1] == target_len:
        return snippet.astype(np.float32)
    if snippet.shape[1] < 2:
        return np.zeros((snippet.shape[0], target_len), dtype=np.float32)
    old_x = np.linspace(0.0, 1.0, snippet.shape[1], dtype=np.float32)
    new_x = np.linspace(0.0, 1.0, target_len, dtype=np.float32)
    out = np.empty((snippet.shape[0], target_len), dtype=np.float32)
    for c in range(snippet.shape[0]):
        out[c] = np.interp(new_x, old_x, snippet[c]).astype(np.float32)
    return out

def extract_beat_tokens(stream: np.ndarray, peaks: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    pre = int(round(cfg.pre_r_sec * cfg.target_fs))
    post = int(round(cfg.post_r_sec * cfg.target_fs))
    c, t = stream.shape
    valid = [int(p) for p in peaks if p - pre >= 0 and p + post <= t]
    if len(valid) > cfg.max_beats:
        idx = np.linspace(0, len(valid) - 1, cfg.max_beats).round().astype(int)
        valid = [valid[i] for i in idx]
    tokens = np.zeros((cfg.max_beats, c, cfg.beat_len), dtype=np.float32)
    mask = np.zeros((cfg.max_beats,), dtype=bool)
    for i, p in enumerate(valid):
        tokens[i] = resize_snippet(stream[:, p - pre:p + post], cfg.beat_len)
        mask[i] = True
    return tokens, mask, np.asarray(valid, dtype=np.int64)


In [8]:
# ============================================================
# 8) Mechanism tokens and handcrafted features
# ============================================================
MECH_TOKEN_NAMES = [
    'rr_prev_sec', 'rr_next_sec', 'heart_rate_bpm_scaled',
    'qrs_width_proxy_sec', 'retrograde_p_score', 'va_interval_proxy_sec',
    'cs_activation_slope_sec_per_channel', 'cs_spread_sec', 'cs_direction_tanh',
    'rva_to_cs_delay_sec', 'abl_to_cs_delay_sec',
    'pace_frac_window', 'ecg_missing_frac', 'iegm_missing_frac',
]

def safe_stats(values: np.ndarray) -> List[float]:
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [0.0, 0.0, 0.0, 0.0]
    return [float(np.mean(values)), float(np.std(values)), float(np.min(values)), float(np.max(values))]

def _lead_for_proxy(ecg_present: np.ndarray, lead_names: Sequence[str] = ('II', 'V1', 'V5', 'I')) -> int:
    for name in lead_names:
        idx = ECG_LEADS.index(name)
        if idx < len(ecg_present) and bool(ecg_present[idx]):
            return idx
    idxs = np.where(ecg_present.astype(bool))[0]
    return int(idxs[0]) if len(idxs) else 0

def _peak_time_abs(token_1d: np.ndarray) -> int:
    if token_1d.size == 0:
        return 0
    return int(np.argmax(np.abs(np.nan_to_num(token_1d))))

def compute_mechanism_tokens(
    ecg_tokens: np.ndarray,
    iegm_tokens: np.ndarray,
    beat_mask: np.ndarray,
    peaks: np.ndarray,
    ecg_present: np.ndarray,
    iegm_present: np.ndarray,
    pace_frac: float,
) -> np.ndarray:
    mech = np.zeros((cfg.max_beats, len(MECH_TOKEN_NAMES)), dtype=np.float32)
    valid_idx = np.where(beat_mask.astype(bool))[0]
    if len(valid_idx) == 0:
        return mech
    center = int(round(cfg.pre_r_sec / (cfg.pre_r_sec + cfg.post_r_sec) * cfg.beat_len))
    beat_dt = (cfg.pre_r_sec + cfg.post_r_sec) / float(cfg.beat_len)
    lead_idx = _lead_for_proxy(ecg_present)

    # ECG post-QRS region for retrograde-P proxy: roughly 60-260 ms after R.
    p0 = min(cfg.beat_len - 1, center + int(round(0.06 / beat_dt)))
    p1 = min(cfg.beat_len, center + int(round(0.26 / beat_dt)))
    q0 = max(0, center - int(round(0.05 / beat_dt)))
    q1 = min(cfg.beat_len, center + int(round(0.08 / beat_dt)))

    for local_i, bi in enumerate(valid_idx):
        # RR context.
        if local_i > 0:
            rr_prev = (float(peaks[local_i]) - float(peaks[local_i - 1])) / cfg.target_fs
        else:
            rr_prev = 0.0
        if local_i + 1 < len(peaks):
            rr_next = (float(peaks[local_i + 1]) - float(peaks[local_i])) / cfg.target_fs
        else:
            rr_next = 0.0
        rr_mean = np.mean([x for x in [rr_prev, rr_next] if x > 0]) if (rr_prev > 0 or rr_next > 0) else 0.0
        hr_scaled = (60.0 / max(rr_mean, 1e-3)) / 240.0 if rr_mean > 0 else 0.0

        # QRS width proxy using thresholded absolute ECG around R.
        sig = np.nan_to_num(ecg_tokens[bi, lead_idx], nan=0.0)
        qrs = np.abs(sig[q0:q1])
        if qrs.size and qrs.max() > 1e-6:
            thr = 0.40 * qrs.max()
            above = np.where(qrs >= thr)[0]
            qrs_width = float((above[-1] - above[0] + 1) * beat_dt) if len(above) else 0.0
        else:
            qrs_width = 0.0

        # Retrograde P-wave proxy: post-QRS energy normalized by QRS energy.
        post = np.abs(sig[p0:p1])
        qrs_amp = float(np.max(qrs)) if qrs.size else 0.0
        retro_p_score = float(np.max(post) / max(qrs_amp, 1e-4)) if post.size else 0.0
        retro_p_score = float(np.clip(retro_p_score, 0.0, 5.0) / 5.0)

        # IEGM activation timing proxies.
        cs_present_idx = [j for j in range(5) if j < len(iegm_present) and bool(iegm_present[j])]
        cs_times = []
        for j in cs_present_idx:
            cs_times.append((_peak_time_abs(iegm_tokens[bi, j]) - center) * beat_dt)
        if len(cs_times) >= 2:
            cs_times_arr = np.asarray(cs_times, dtype=np.float32)
            cs_x = np.asarray(cs_present_idx, dtype=np.float32)
            try:
                cs_slope = float(np.polyfit(cs_x, cs_times_arr, deg=1)[0])
            except Exception:
                cs_slope = 0.0
            cs_spread = float(cs_times_arr.max() - cs_times_arr.min())
            cs_center_t = float(cs_times_arr.mean())
        elif len(cs_times) == 1:
            cs_slope = 0.0
            cs_spread = 0.0
            cs_center_t = float(cs_times[0])
        else:
            cs_slope = 0.0
            cs_spread = 0.0
            cs_center_t = 0.0
        cs_direction = float(np.tanh(cs_slope / 0.03))

        rva_delay = 0.0
        if len(IEGM_CHANNELS) > 5 and len(iegm_present) > 5 and bool(iegm_present[5]):
            rva_t = (_peak_time_abs(iegm_tokens[bi, 5]) - center) * beat_dt
            rva_delay = float(rva_t - cs_center_t)
        abl_delay = 0.0
        if len(IEGM_CHANNELS) > 6 and len(iegm_present) > 6 and bool(iegm_present[6]):
            abl_t = (_peak_time_abs(iegm_tokens[bi, 6]) - center) * beat_dt
            abl_delay = float(abl_t - cs_center_t)

        va_proxy = cs_center_t
        mech[bi] = np.asarray([
            rr_prev, rr_next, hr_scaled,
            qrs_width, retro_p_score, va_proxy,
            cs_slope, cs_spread, cs_direction,
            rva_delay, abl_delay,
            pace_frac,
            1.0 - float(ecg_present.mean()), 1.0 - float(iegm_present.mean()),
        ], dtype=np.float32)

    # Clip extreme timing values for stability but keep sign.
    for k in [0, 1, 3, 5, 6, 7, 9, 10]:
        mech[:, k] = np.clip(mech[:, k], -2.0, 2.0)
    return np.nan_to_num(mech, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def channel_summary_features(tokens: np.ndarray, beat_mask: np.ndarray, present: np.ndarray) -> List[float]:
    valid = tokens[beat_mask]
    c = tokens.shape[1]
    if valid.size == 0:
        vals = np.zeros(c * 5, dtype=np.float32)
    else:
        rms = np.sqrt(np.mean(valid ** 2, axis=(0, 2)))
        ptp = np.ptp(valid, axis=(0, 2))
        maxabs = np.max(np.abs(valid), axis=(0, 2))
        slope = np.mean(np.abs(np.diff(valid, axis=2)), axis=(0, 2))
        energy = np.mean(valid ** 2, axis=(0, 2))
        vals = np.concatenate([rms, ptp, maxabs, slope, energy])
        vals = vals * np.tile(present.astype(np.float32), 5)
    return vals.astype(np.float32).tolist()

def handcrafted_features(
    ecg_tokens: np.ndarray,
    iegm_tokens: np.ndarray,
    beat_mask: np.ndarray,
    peaks: np.ndarray,
    ecg_present: np.ndarray,
    iegm_present: np.ndarray,
    pace_frac: float,
    mech_tokens: np.ndarray,
) -> np.ndarray:
    feats: List[float] = []
    n_beats = int(beat_mask.sum())
    feats += [float(n_beats), float(pace_frac), float(ecg_present.sum()), float(iegm_present.sum())]
    if len(peaks) >= 2:
        rr = np.diff(peaks) / float(cfg.target_fs)
        feats += safe_stats(rr)
        feats += [float(60.0 / max(np.mean(rr), 1e-3)), float(np.median(rr))]
    else:
        feats += [0.0] * 6
    feats += channel_summary_features(ecg_tokens, beat_mask, ecg_present)
    feats += channel_summary_features(iegm_tokens, beat_mask, iegm_present)
    valid_mech = mech_tokens[beat_mask.astype(bool)] if beat_mask.sum() else np.zeros((0, len(MECH_TOKEN_NAMES)), dtype=np.float32)
    if valid_mech.size:
        for j in range(valid_mech.shape[1]):
            feats += safe_stats(valid_mech[:, j])
    else:
        feats += [0.0] * (4 * len(MECH_TOKEN_NAMES))
    return np.asarray(feats, dtype=np.float32)


In [9]:
# ============================================================
# 9) Token cache
# ============================================================
def cache_file_for_row(row_id: int) -> Path:
    return Path(cfg.cache_dir) / f'win_{int(row_id):07d}.npz'

def process_window(row: pd.Series) -> Dict[str, np.ndarray]:
    ecg_filt, iegm_filt, ecg_present, iegm_present, orig_fs, ecg_unfilt, iegm_unfilt = load_window_streams(row.record, int(row.start), int(row.end))
    ecg_scaled = robust_zscore(ecg_filt)
    iegm_scaled = robust_zscore(iegm_filt)
    peaks = choose_rpeaks(row.record, int(row.start), int(row.end), orig_fs, ecg_scaled, ecg_present)
    ecg_tokens, beat_mask, selected_peaks = extract_beat_tokens(ecg_scaled, peaks)
    iegm_tokens, _, _ = extract_beat_tokens(iegm_scaled, selected_peaks)
    mech_tokens = compute_mechanism_tokens(ecg_tokens, iegm_tokens, beat_mask, selected_peaks, ecg_present, iegm_present, float(row.pace_frac))
    features = handcrafted_features(ecg_tokens, iegm_tokens, beat_mask, selected_peaks, ecg_present, iegm_present, float(row.pace_frac), mech_tokens)
    return {
        'ecg': ecg_tokens.astype(np.float32),
        'iegm': iegm_tokens.astype(np.float32),
        'beat_mask': beat_mask.astype(bool),
        'ecg_ch_mask': ecg_present.astype(bool),
        'iegm_ch_mask': iegm_present.astype(bool),
        'mech': mech_tokens.astype(np.float32),
        'features': features.astype(np.float32),
        'rpeaks': selected_peaks.astype(np.int64),
        'label': np.asarray(int(row.label), dtype=np.int64),
        'pace_frac': np.asarray(float(row.pace_frac), dtype=np.float32),
        'record_hash': np.asarray(stable_hash({'record': row.record, 'start': int(row.start), 'end': int(row.end)})),
    }

def build_token_cache(df: pd.DataFrame, overwrite: bool = False) -> pd.DataFrame:
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    failures = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building token cache'):
        fp = cache_file_for_row(int(row.row_id))
        if fp.exists() and not overwrite:
            continue
        try:
            z = process_window(row)
            np.savez_compressed(fp, **z)
        except Exception as e:
            failures.append({'row_id': int(row.row_id), 'record': row.record, 'start': int(row.start), 'error': repr(e)})
    fail_df = pd.DataFrame(failures)
    if len(fail_df):
        fail_df.to_csv(Path(cfg.log_dir) / 'cache_failures.csv', index=False)
        print(f'Cache failures: {len(fail_df)}')
    return fail_df

def infer_dims(df: pd.DataFrame) -> Tuple[int, int]:
    for _, row in df.iterrows():
        fp = cache_file_for_row(int(row.row_id))
        if not fp.exists():
            np.savez_compressed(fp, **process_window(row))
        z = np.load(fp, allow_pickle=False)
        return int(z['features'].shape[0]), int(z['mech'].shape[1])
    raise ValueError('No rows available to infer dimensions.') 

# Build binary cache now; fold-specific SSL pretraining will cache more windows as needed.
build_token_cache(df_binary, overwrite=False)
FEATURE_DIM, MECH_DIM = infer_dims(df_binary)
print('Feature dim:', FEATURE_DIM, 'Mechanism token dim:', MECH_DIM)
save_json({'feature_dim': FEATURE_DIM, 'mech_dim': MECH_DIM, 'mech_token_names': MECH_TOKEN_NAMES}, Path(cfg.log_dir) / 'feature_and_mechanism_dims.json')


Building token cache:   0%|          | 0/1410 [00:00<?, ?it/s]

Feature dim: 161 Mechanism token dim: 14


In [10]:
# ============================================================
# 10) Datasets and dataloaders
# ============================================================
def load_npz_for_row(row: pd.Series) -> Dict[str, Any]:
    fp = cache_file_for_row(int(row.row_id))
    if not fp.exists():
        np.savez_compressed(fp, **process_window(row))
    z = np.load(fp, allow_pickle=False)
    item = {k: z[k] for k in z.files}
    item['record'] = row.record
    item['row_id'] = int(row.row_id)
    item['start'] = int(row.start)
    item['end'] = int(row.end)
    return item

def augment_tokens_np(x: np.ndarray, ch_mask: np.ndarray, beat_mask: np.ndarray, train: bool = True) -> np.ndarray:
    x = x.copy()
    if not (train and cfg.use_augmentation):
        return x.astype(np.float32)
    scale = np.random.normal(1.0, cfg.aug_scale_std, size=(1, x.shape[1], 1)).astype(np.float32)
    x *= scale
    if cfg.aug_noise_std > 0:
        x += np.random.normal(0.0, cfg.aug_noise_std, size=x.shape).astype(np.float32)
    max_shift = int(round(cfg.aug_time_shift_ms * cfg.beat_len / ((cfg.pre_r_sec + cfg.post_r_sec) * 1000)))
    if max_shift > 0:
        sh = np.random.randint(-max_shift, max_shift + 1)
        if sh != 0:
            x = np.roll(x, sh, axis=2)
    if cfg.aug_lead_dropout_p > 0:
        drop = (np.random.rand(x.shape[1]) < cfg.aug_lead_dropout_p) & ch_mask.astype(bool)
        x[:, drop, :] = 0.0
    if cfg.aug_beat_dropout_p > 0 and beat_mask.sum() > 1:
        drop_b = (np.random.rand(x.shape[0]) < cfg.aug_beat_dropout_p) & beat_mask.astype(bool)
        if drop_b.sum() < beat_mask.sum():
            x[drop_b] = 0.0
    return x.astype(np.float32)

class SVTWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = False, augment: bool = False):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.augment = bool(augment and train)
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        item = load_npz_for_row(row)
        ecg = item['ecg'].astype(np.float32)
        iegm = item['iegm'].astype(np.float32)
        beat_mask = item['beat_mask'].astype(bool)
        ecg_ch_mask = item['ecg_ch_mask'].astype(bool)
        iegm_ch_mask = item['iegm_ch_mask'].astype(bool)
        ecg = augment_tokens_np(ecg, ecg_ch_mask, beat_mask, train=self.augment)
        iegm = augment_tokens_np(iegm, iegm_ch_mask, beat_mask, train=self.augment)
        return {
            'ecg': torch.tensor(ecg, dtype=torch.float32),
            'iegm': torch.tensor(iegm, dtype=torch.float32),
            'beat_mask': torch.tensor(beat_mask, dtype=torch.bool),
            'ecg_ch_mask': torch.tensor(ecg_ch_mask, dtype=torch.bool),
            'iegm_ch_mask': torch.tensor(iegm_ch_mask, dtype=torch.bool),
            'features': torch.tensor(item['features'].astype(np.float32), dtype=torch.float32),
            'mech': torch.tensor(item['mech'].astype(np.float32), dtype=torch.float32),
            'label': torch.tensor(int(item['label']), dtype=torch.long),
            'pace_frac': torch.tensor(float(item['pace_frac']), dtype=torch.float32),
            'row_id': torch.tensor(int(item['row_id']), dtype=torch.long),
            'record': item['record'],
            'start': item['start'],
            'end': item['end'],
        }

class RecordBagDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = False):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.records = sorted(self.df['record'].unique().tolist())
    def __len__(self) -> int:
        return len(self.records)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        record = self.records[idx]
        g = self.df[self.df['record'] == record].sort_values('start').reset_index(drop=True)
        label = int(g['label'].mode().iloc[0])
        max_w = cfg.max_windows_per_record_train if self.train else cfg.max_windows_per_record_eval
        if len(g) > max_w:
            if self.train:
                pick = np.sort(np.random.choice(np.arange(len(g)), size=max_w, replace=False))
            else:
                pick = np.linspace(0, len(g) - 1, max_w).round().astype(int)
            g = g.iloc[pick].reset_index(drop=True)
        items = []
        for _, row in g.iterrows():
            item = load_npz_for_row(row)
            ecg = item['ecg'].astype(np.float32)
            iegm = item['iegm'].astype(np.float32)
            beat_mask = item['beat_mask'].astype(bool)
            ecg_ch_mask = item['ecg_ch_mask'].astype(bool)
            iegm_ch_mask = item['iegm_ch_mask'].astype(bool)
            ecg = augment_tokens_np(ecg, ecg_ch_mask, beat_mask, train=self.train)
            iegm = augment_tokens_np(iegm, iegm_ch_mask, beat_mask, train=self.train)
            items.append({
                'ecg': ecg,
                'iegm': iegm,
                'beat_mask': beat_mask,
                'ecg_ch_mask': ecg_ch_mask,
                'iegm_ch_mask': iegm_ch_mask,
                'features': item['features'].astype(np.float32),
                'mech': item['mech'].astype(np.float32),
                'row_id': int(item['row_id']),
                'start': int(row.start),
                'end': int(row.end),
            })
        return {'record': record, 'label': label, 'items': items}

def record_bag_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    b = len(batch)
    max_w = max(len(x['items']) for x in batch)
    ecg = torch.zeros((b, max_w, cfg.max_beats, len(ECG_LEADS), cfg.beat_len), dtype=torch.float32)
    iegm = torch.zeros((b, max_w, cfg.max_beats, len(IEGM_CHANNELS), cfg.beat_len), dtype=torch.float32)
    beat_mask = torch.zeros((b, max_w, cfg.max_beats), dtype=torch.bool)
    ecg_ch_mask = torch.zeros((b, max_w, len(ECG_LEADS)), dtype=torch.bool)
    iegm_ch_mask = torch.zeros((b, max_w, len(IEGM_CHANNELS)), dtype=torch.bool)
    features = torch.zeros((b, max_w, FEATURE_DIM), dtype=torch.float32)
    mech = torch.zeros((b, max_w, cfg.max_beats, MECH_DIM), dtype=torch.float32)
    window_mask = torch.zeros((b, max_w), dtype=torch.bool)
    row_ids = torch.full((b, max_w), -1, dtype=torch.long)
    starts = torch.zeros((b, max_w), dtype=torch.long)
    ends = torch.zeros((b, max_w), dtype=torch.long)
    labels = torch.tensor([x['label'] for x in batch], dtype=torch.long)
    records = [x['record'] for x in batch]
    for i, rec in enumerate(batch):
        for j, item in enumerate(rec['items']):
            ecg[i, j] = torch.tensor(item['ecg'], dtype=torch.float32)
            iegm[i, j] = torch.tensor(item['iegm'], dtype=torch.float32)
            beat_mask[i, j] = torch.tensor(item['beat_mask'], dtype=torch.bool)
            ecg_ch_mask[i, j] = torch.tensor(item['ecg_ch_mask'], dtype=torch.bool)
            iegm_ch_mask[i, j] = torch.tensor(item['iegm_ch_mask'], dtype=torch.bool)
            features[i, j] = torch.tensor(item['features'], dtype=torch.float32)
            mech[i, j] = torch.tensor(item['mech'], dtype=torch.float32)
            window_mask[i, j] = True
            row_ids[i, j] = int(item['row_id'])
            starts[i, j] = int(item['start'])
            ends[i, j] = int(item['end'])
    return {
        'ecg': ecg,
        'iegm': iegm,
        'beat_mask': beat_mask,
        'ecg_ch_mask': ecg_ch_mask,
        'iegm_ch_mask': iegm_ch_mask,
        'features': features,
        'mech': mech,
        'window_mask': window_mask,
        'row_ids': row_ids,
        'starts': starts,
        'ends': ends,
        'label': labels,
        'record': records,
    }

def move_batch(batch: Dict[str, Any], device: str) -> Dict[str, Any]:
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

def make_window_loader(df_part: pd.DataFrame, batch_size: int, train: bool = False, balanced: bool = False) -> DataLoader:
    df_part = df_part.reset_index(drop=True)
    if train and balanced and 'label' in df_part and df_part['label'].isin([0, 1]).all():
        y = df_part['label'].values.astype(int)
        counts = np.bincount(y, minlength=2).astype(np.float32)
        sample_weights = (1.0 / np.maximum(counts, 1.0))[y]
        sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)
        return DataLoader(SVTWindowDataset(df_part, train=True, augment=True), batch_size=batch_size, sampler=sampler, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    return DataLoader(SVTWindowDataset(df_part, train=train, augment=train), batch_size=batch_size, shuffle=train, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())

def make_record_loader(df_part: pd.DataFrame, train: bool = False) -> DataLoader:
    return DataLoader(RecordBagDataset(df_part, train=train), batch_size=cfg.record_batch_size, shuffle=train, collate_fn=record_bag_collate, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())


In [11]:
# ============================================================
# 11) Metrics and calibration
# ============================================================
def expected_calibration_error_binary(probs_pos: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:
    probs_pos = np.asarray(probs_pos, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    conf = np.maximum(probs_pos, 1.0 - probs_pos)
    pred = (probs_pos >= 0.5).astype(int)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def specificity_by_class(y_true: np.ndarray, y_pred: np.ndarray, class_id: int) -> float:
    y_true_bin = (np.asarray(y_true) == class_id).astype(int)
    y_pred_bin = (np.asarray(y_pred) == class_id).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    return float(tn / max(tn + fp, 1))

def compute_binary_metrics(y_true: np.ndarray, probs: np.ndarray, prefix: str = '', threshold: float = 0.5, pred: Optional[np.ndarray] = None) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    if probs.ndim == 1:
        probs_pos = probs
        probs2 = np.column_stack([1.0 - probs_pos, probs_pos])
    else:
        probs2 = probs
        probs_pos = probs[:, 1]
    y_pred = np.asarray(pred, dtype=int) if pred is not None else (probs_pos >= threshold).astype(int)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    out = {
        prefix + 'threshold': float(threshold),
        prefix + 'accuracy': float(accuracy_score(y_true, y_pred)),
        prefix + 'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        prefix + 'macro_precision': float(np.mean(precision)),
        prefix + 'macro_recall': float(np.mean(recall)),
        prefix + 'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        prefix + 'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        prefix + 'mcc': float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_true)) > 1 else 0.0,
        prefix + 'cohen_kappa': float(cohen_kappa_score(y_true, y_pred)),
        prefix + 'ece': expected_calibration_error_binary(probs_pos, y_true),
        prefix + 'brier': float(np.mean((probs_pos - y_true) ** 2)),
        prefix + 'specificity_AVNRT': specificity_by_class(y_true, y_pred, 0),
        prefix + 'specificity_AVRT': specificity_by_class(y_true, y_pred, 1),
        prefix + 'sensitivity_AVNRT': float(recall[0]),
        prefix + 'sensitivity_AVRT': float(recall[1]),
    }
    try:
        out[prefix + 'roc_auc'] = float(roc_auc_score(y_true, probs_pos)) if len(np.unique(y_true)) == 2 else float('nan')
    except Exception:
        out[prefix + 'roc_auc'] = float('nan')
    try:
        out[prefix + 'pr_auc'] = float(average_precision_score(y_true, probs_pos)) if len(np.unique(y_true)) == 2 else float('nan')
    except Exception:
        out[prefix + 'pr_auc'] = float('nan')
    try:
        out[prefix + 'log_loss'] = float(log_loss(y_true, np.clip(probs2, 1e-7, 1 - 1e-7), labels=[0, 1]))
    except Exception:
        out[prefix + 'log_loss'] = float('nan')
    for i, name in enumerate(BINARY_LABELS):
        out[prefix + f'precision_{name}'] = float(precision[i])
        out[prefix + f'recall_{name}'] = float(recall[i])
        out[prefix + f'f1_{name}'] = float(f1[i])
        out[prefix + f'support_{name}'] = float(support[i])
    return out

def softmax_np(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=float)
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.clip(e.sum(axis=1, keepdims=True), 1e-12, None)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))
    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / torch.exp(self.log_t).clamp_min(1e-3)

def fit_temperature(logits: np.ndarray, labels: np.ndarray, device: str = 'cpu') -> float:
    logits_t = torch.tensor(logits, dtype=torch.float32, device=device)
    labels_t = torch.tensor(labels, dtype=torch.long, device=device)
    scaler = TemperatureScaler().to(device)
    optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.05, max_iter=100)
    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(scaler(logits_t), labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(torch.exp(scaler.log_t.detach()).cpu().item())

def find_best_threshold(y_true: np.ndarray, probs_pos: np.ndarray, metric: str = 'macro_f1') -> Tuple[float, float]:
    y_true = np.asarray(y_true).astype(int)
    probs_pos = np.asarray(probs_pos).astype(float)
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t = 0.5
    best_v = -1.0
    for t in thresholds:
        pred = (probs_pos >= t).astype(int)
        if metric == 'macro_f1':
            v = f1_score(y_true, pred, average='macro', zero_division=0)
        else:
            v = balanced_accuracy_score(y_true, pred)
        if v > best_v:
            best_v = float(v)
            best_t = float(t)
    return best_t, best_v


In [12]:
# ============================================================
# 12) Classical ML baselines
# ============================================================
def load_feature_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    x, y = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading engineered features'):
        item = load_npz_for_row(row)
        x.append(item['features'].astype(np.float32))
        y.append(int(row.label))
    return np.vstack(x), np.asarray(y, dtype=int)

def classical_model_registry() -> Dict[str, Any]:
    models: Dict[str, Any] = {
        'logistic_regression_l2': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=5000, class_weight='balanced'))]),
        'random_forest': RandomForestClassifier(n_estimators=500, random_state=cfg.seed, class_weight='balanced', min_samples_leaf=2, n_jobs=-1),
        'extra_trees': ExtraTreesClassifier(n_estimators=500, random_state=cfg.seed, class_weight='balanced', min_samples_leaf=2, n_jobs=-1),
        'svm_rbf': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', probability=True, class_weight='balanced', C=1.0, gamma='scale'))]),
        'hist_gradient_boosting': HistGradientBoostingClassifier(random_state=cfg.seed),
    }
    if XGBClassifier is not None:
        models['xgboost'] = XGBClassifier(
            n_estimators=400, max_depth=3, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9,
            objective='binary:logistic', eval_metric='logloss', random_state=cfg.seed, n_jobs=-1,
        )
    if LGBMClassifier is not None:
        models['lightgbm'] = LGBMClassifier(n_estimators=400, learning_rate=0.03, random_state=cfg.seed, class_weight='balanced')
    return models

def align_binary_proba(clf: Any, probs: np.ndarray) -> np.ndarray:
    if probs.shape[1] == 2:
        return probs
    classes = None
    if hasattr(clf, 'classes_'):
        classes = clf.classes_
    elif isinstance(clf, Pipeline) and hasattr(clf[-1], 'classes_'):
        classes = clf[-1].classes_
    aligned = np.zeros((probs.shape[0], 2), dtype=float)
    if classes is not None:
        for j, c in enumerate(classes):
            aligned[:, int(c)] = probs[:, j]
    else:
        aligned[:, -probs.shape[1]:] = probs
    return aligned / np.clip(aligned.sum(axis=1, keepdims=True), 1e-12, None)

def aggregate_window_predictions_to_records(window_pred: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, ablation, fold, record), g in window_pred.groupby(['model', 'ablation', 'fold', 'record']):
        p = g[['prob_AVNRT', 'prob_AVRT']].values.astype(float).mean(axis=0)
        y = int(g['label'].mode().iloc[0])
        threshold = float(g['threshold'].iloc[0]) if 'threshold' in g else 0.5
        rows.append({
            'model': model, 'ablation': ablation, 'fold': int(fold), 'record': record,
            'label': y, 'label_name': BINARY_LABELS[y], 'pred': int(p[1] >= threshold),
            'threshold': threshold, 'confidence': float(np.max(p)), 'n_windows': int(len(g)),
            'pace_frac_mean': float(g['pace_frac'].mean()) if 'pace_frac' in g else np.nan,
            'cohort_group': g['cohort_group'].mode().iloc[0] if 'cohort_group' in g else 'unknown',
            'prob_AVNRT': float(p[0]), 'prob_AVRT': float(p[1]),
        })
    return pd.DataFrame(rows)

def run_classical_fold(df_task: pd.DataFrame, fold: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    outer_train = df_task[df_task.fold != fold].reset_index(drop=True)
    outer_test = df_task[df_task.fold == fold].reset_index(drop=True)
    inner_train, inner_val = make_inner_split(outer_train, seed=cfg.seed + 101 * fold, n_inner_splits=cfg.n_inner_splits)
    x_train, y_train = load_feature_matrix(inner_train)
    x_val, y_val = load_feature_matrix(inner_val)
    x_test, y_test = load_feature_matrix(outer_test)
    result_rows, pred_frames = [], []
    for name, model in classical_model_registry().items():
        t0 = time.time()
        clf = clone(model)
        clf.fit(x_train, y_train)
        train_sec = time.time() - t0
        val_probs = align_binary_proba(clf, clf.predict_proba(x_val)) if hasattr(clf, 'predict_proba') else softmax_np(clf.decision_function(x_val))
        threshold, val_f1 = find_best_threshold(y_val, val_probs[:, 1]) if cfg.tune_threshold else (0.5, np.nan)
        t1 = time.time()
        probs = align_binary_proba(clf, clf.predict_proba(x_test)) if hasattr(clf, 'predict_proba') else softmax_np(clf.decision_function(x_test))
        infer_sec = time.time() - t1
        pred = (probs[:, 1] >= threshold).astype(int)
        metrics = compute_binary_metrics(y_test, probs, prefix='window_', threshold=threshold, pred=pred)
        result_rows.append({
            'model': name, 'ablation': 'classical', 'fold': int(fold), 'n_params': np.nan,
            'training_time_sec': train_sec, 'inference_time_ms_per_window': 1000 * infer_sec / max(len(outer_test), 1),
            'val_threshold': threshold, 'val_macro_f1_at_threshold': val_f1, **metrics,
        })
        pred_df = outer_test[['record', 'row_id', 'start', 'end', 'label', 'label_name', 'pace_frac', 'cohort_group']].copy().reset_index(drop=True)
        pred_df['model'] = name
        pred_df['ablation'] = 'classical'
        pred_df['fold'] = int(fold)
        pred_df['threshold'] = threshold
        pred_df['pred'] = pred
        pred_df['confidence'] = probs.max(axis=1)
        pred_df['prob_AVNRT'] = probs[:, 0]
        pred_df['prob_AVRT'] = probs[:, 1]
        pred_frames.append(pred_df)
    window_pred = pd.concat(pred_frames, ignore_index=True)
    record_pred = aggregate_window_predictions_to_records(window_pred)
    result_df = pd.DataFrame(result_rows)
    result_df.to_csv(Path(cfg.table_dir) / f'classical_results_fold{fold}.csv', index=False)
    window_pred.to_csv(Path(cfg.table_dir) / f'classical_window_predictions_fold{fold}.csv', index=False)
    record_pred.to_csv(Path(cfg.table_dir) / f'classical_record_predictions_fold{fold}.csv', index=False)
    return result_df, window_pred, record_pred


In [13]:
# ============================================================
# 13) Neural network components
# ============================================================
def masked_softmax(logits: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    mask = mask.bool()
    masked_logits = logits.masked_fill(~mask, torch.finfo(logits.dtype).min)
    probs = torch.softmax(masked_logits, dim=dim) * mask.float()
    return probs / probs.sum(dim=dim, keepdim=True).clamp_min(1e-8)

class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class AttentionPool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        logits = self.score(x).squeeze(-1)
        weights = masked_softmax(logits, mask.bool(), dim=-1)
        return torch.sum(weights.unsqueeze(-1) * x, dim=1), weights

class MultiScaleBeatEncoder(nn.Module):
    def __init__(self, n_channels: int, d_model: int, dropout: float, multiscale: bool = True):
        super().__init__()
        kernels = [3, 7, 15] if multiscale else [7]
        branch_dim = max(16, d_model // len(kernels))
        self.branches = nn.ModuleList()
        for k in kernels:
            self.branches.append(nn.Sequential(
                nn.Conv1d(n_channels, branch_dim, kernel_size=k, padding=k // 2),
                nn.BatchNorm1d(branch_dim), nn.GELU(),
                nn.Conv1d(branch_dim, branch_dim, kernel_size=k, padding=k // 2),
                nn.BatchNorm1d(branch_dim), nn.GELU(),
            ))
        self.mix = nn.Sequential(nn.Conv1d(branch_dim * len(kernels), d_model, kernel_size=1), nn.GELU(), nn.Dropout(dropout))
        self.res_proj = nn.Conv1d(n_channels, d_model, kernel_size=1)
        self.se = nn.Sequential(nn.Linear(d_model, max(8, d_model // 4)), nn.GELU(), nn.Linear(max(8, d_model // 4), d_model), nn.Sigmoid())
        self.ch_embed = nn.Embedding(n_channels, d_model)
        self.sample_proj = nn.Sequential(nn.LayerNorm(cfg.beat_len), nn.Linear(cfg.beat_len, d_model), nn.GELU(), nn.Dropout(dropout))
        self.ch_gate = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
    def forward(self, x: torch.Tensor, ch_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # x: [B, N, C, L]
        b, n, c, l = x.shape
        x = x * ch_mask[:, None, :, None].float()
        flat = x.reshape(b * n, c, l)
        z = torch.cat([branch(flat) for branch in self.branches], dim=1)
        z = self.mix(z) + self.res_proj(flat)
        pooled = F.adaptive_avg_pool1d(z, 1).squeeze(-1)
        pooled = pooled * self.se(pooled)
        beat_emb = pooled.reshape(b, n, -1)
        ch_z = self.sample_proj(x.reshape(b * n * c, l)).reshape(b, n, c, -1)
        idx = torch.arange(c, device=x.device)
        ch_z = ch_z + self.ch_embed(idx).view(1, 1, c, -1)
        ch_logits = self.ch_gate(ch_z).squeeze(-1)
        ch_weights = masked_softmax(ch_logits, ch_mask[:, None, :].expand(b, n, c), dim=-1)
        ch_context = torch.sum(ch_weights.unsqueeze(-1) * ch_z, dim=2)
        beat_emb = beat_emb + ch_context
        return beat_emb, ch_z, ch_weights

class StreamEncoder(nn.Module):
    def __init__(self, n_channels: int, cfg: CFG):
        super().__init__()
        self.beat_encoder = MultiScaleBeatEncoder(n_channels, cfg.d_model, cfg.dropout, multiscale=True)
        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
        layer = nn.TransformerEncoderLayer(d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=cfg.d_model * 4,
                                           dropout=cfg.dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)
        self.drop_path = DropPath(cfg.stochastic_depth)
        self.norm = nn.LayerNorm(cfg.d_model)
        nn.init.trunc_normal_(self.cls, std=0.02)
    def forward(self, x: torch.Tensor, beat_mask: torch.Tensor, ch_mask: torch.Tensor) -> Dict[str, torch.Tensor]:
        b, n, c, _ = x.shape
        beat_emb, ch_z, ch_weights = self.beat_encoder(x, ch_mask)
        cls = self.cls.expand(b, -1, -1)
        seq = torch.cat([cls, beat_emb], dim=1)
        full_mask = torch.cat([torch.ones((b, 1), dtype=torch.bool, device=x.device), beat_mask.bool()], dim=1)
        pad_mask = ~full_mask
        out = self.encoder(seq, src_key_padding_mask=pad_mask)
        out = self.norm(seq + self.drop_path(out - seq))
        return {'seq': out, 'cls': out[:, 0], 'beats': out[:, 1:], 'pad_mask': pad_mask, 'beat_mask': beat_mask.bool(), 'ch_z': ch_z, 'ch_weights': ch_weights}

class FeatureEncoder(nn.Module):
    def __init__(self, feature_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(feature_dim), nn.Linear(feature_dim, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.net(features)

class MechanismTokenEncoder(nn.Module):
    def __init__(self, mech_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(mech_dim), nn.Linear(mech_dim, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.pool = AttentionPool(d_model)
    def forward(self, mech: torch.Tensor, beat_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z = self.proj(mech)
        ctx, w = self.pool(z, beat_mask.bool())
        return ctx, z, w

def einthoven_repr_loss(ecg_ch_z: torch.Tensor, beat_mask: torch.Tensor, ecg_ch_mask: torch.Tensor) -> torch.Tensor:
    if ecg_ch_z.shape[2] < 3:
        return ecg_ch_z.new_tensor(0.0)
    present = (ecg_ch_mask[:, 0] & ecg_ch_mask[:, 1] & ecg_ch_mask[:, 2]).float()[:, None]
    valid = beat_mask.float() * present
    if valid.sum() < 1:
        return ecg_ch_z.new_tensor(0.0)
    z_i = F.normalize(ecg_ch_z[:, :, 0], dim=-1)
    z_ii = F.normalize(ecg_ch_z[:, :, 1], dim=-1)
    z_iii = F.normalize(ecg_ch_z[:, :, 2], dim=-1)
    target = F.normalize(z_i + z_iii, dim=-1)
    loss = 1.0 - (z_ii * target).sum(dim=-1)
    return (loss * valid).sum() / valid.sum().clamp_min(1.0)

class CSOrderRegularizer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.time_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Linear(d_model // 2, 1), nn.Sigmoid())
    def forward(self, iegm_ch_z: torch.Tensor, beat_mask: torch.Tensor, iegm_ch_mask: torch.Tensor) -> torch.Tensor:
        if iegm_ch_z.shape[2] < 5:
            return iegm_ch_z.new_tensor(0.0)
        pred_t = self.time_head(iegm_ch_z[:, :, :5]).squeeze(-1)
        losses = []
        for i in range(4):
            pair_present = (iegm_ch_mask[:, i] & iegm_ch_mask[:, i + 1]).float()[:, None]
            valid = beat_mask.float() * pair_present
            if valid.sum() < 1:
                continue
            inv = F.relu(pred_t[:, :, i] - pred_t[:, :, i + 1]) ** 2
            losses.append((inv * valid).sum() / valid.sum().clamp_min(1.0))
        return torch.stack(losses).mean() if losses else iegm_ch_z.new_tensor(0.0)


In [14]:
# ============================================================
# 14) Window encoder, SSL heads, and MIL model
# ============================================================
class MechXWindowEncoder(nn.Module):
    def __init__(self, feature_dim: int, mech_dim: int, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.ecg = StreamEncoder(len(ECG_LEADS), cfg)
        self.iegm = StreamEncoder(len(IEGM_CHANNELS), cfg)
        self.ecg_pool = AttentionPool(cfg.d_model)
        self.iegm_pool = AttentionPool(cfg.d_model)
        self.mech_encoder = MechanismTokenEncoder(mech_dim, cfg.d_model, cfg.dropout)
        self.feature_encoder = FeatureEncoder(feature_dim, cfg.d_model, cfg.dropout)
        self.ecg_to_iegm = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, batch_first=True)
        self.iegm_to_ecg = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, batch_first=True)
        self.cs_reg = CSOrderRegularizer(cfg.d_model)
        fused_dim = cfg.d_model * 8
        self.emb_proj = nn.Sequential(nn.LayerNorm(fused_dim), nn.Linear(fused_dim, cfg.d_model * 2), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(cfg.d_model * 2, cfg.d_model))
        self.classifier = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Dropout(cfg.dropout), nn.Linear(cfg.d_model, 2))
        # Reconstruction heads for SSL.
        self.ecg_recon = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, len(ECG_LEADS) * cfg.beat_len))
        self.iegm_recon = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, len(IEGM_CHANNELS) * cfg.beat_len))
        self.ecg_to_iegm_recon = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, len(IEGM_CHANNELS) * cfg.beat_len))
        self.iegm_to_ecg_recon = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, len(ECG_LEADS) * cfg.beat_len))
    def forward(self, batch: Dict[str, torch.Tensor], return_recon: bool = False, return_attention: bool = False) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        e = self.ecg(batch['ecg'], batch['beat_mask'], batch['ecg_ch_mask'])
        g = self.iegm(batch['iegm'], batch['beat_mask'], batch['iegm_ch_mask'])
        ecg_ctx, ew = self.ecg_pool(e['beats'], e['beat_mask'])
        iegm_ctx, gw = self.iegm_pool(g['beats'], g['beat_mask'])
        mech_ctx, mech_z, mw = self.mech_encoder(batch['mech'], batch['beat_mask'])
        feat_ctx = self.feature_encoder(batch['features'])
        e2g_ctx, e2g_w = self.ecg_to_iegm(e['cls'].unsqueeze(1), g['seq'], g['seq'], key_padding_mask=g['pad_mask'], need_weights=return_attention)
        g2e_ctx, g2e_w = self.iegm_to_ecg(g['cls'].unsqueeze(1), e['seq'], e['seq'], key_padding_mask=e['pad_mask'], need_weights=return_attention)
        fused = torch.cat([e['cls'], ecg_ctx, g['cls'], iegm_ctx, e2g_ctx.squeeze(1), g2e_ctx.squeeze(1), mech_ctx, feat_ctx], dim=-1)
        emb = self.emb_proj(fused)
        logits = self.classifier(emb)
        aux: Dict[str, torch.Tensor] = {
            'embedding': emb,
            'ecg_cls': e['cls'],
            'iegm_cls': g['cls'],
            'einthoven': einthoven_repr_loss(e['ch_z'], e['beat_mask'], batch['ecg_ch_mask']),
            'cs_order': self.cs_reg(g['ch_z'], g['beat_mask'], batch['iegm_ch_mask']),
        }
        if return_recon:
            b, n = batch['beat_mask'].shape
            aux['ecg_recon'] = self.ecg_recon(e['beats']).reshape(b, n, len(ECG_LEADS), cfg.beat_len)
            aux['iegm_recon'] = self.iegm_recon(g['beats']).reshape(b, n, len(IEGM_CHANNELS), cfg.beat_len)
            aux['ecg_to_iegm_recon'] = self.ecg_to_iegm_recon(e['beats']).reshape(b, n, len(IEGM_CHANNELS), cfg.beat_len)
            aux['iegm_to_ecg_recon'] = self.iegm_to_ecg_recon(g['beats']).reshape(b, n, len(ECG_LEADS), cfg.beat_len)
        if return_attention:
            aux.update({
                'ecg_beat_attention': ew.detach(),
                'iegm_beat_attention': gw.detach(),
                'mech_attention': mw.detach(),
                'ecg_channel_weights': e['ch_weights'].detach(),
                'iegm_channel_weights': g['ch_weights'].detach(),
                'ecg_to_iegm_attention': e2g_w.detach() if e2g_w is not None else None,
                'iegm_to_ecg_attention': g2e_w.detach() if g2e_w is not None else None,
            })
        return logits, aux

class RecordMILMechXFormer(nn.Module):
    def __init__(self, window_encoder: MechXWindowEncoder, cfg: CFG):
        super().__init__()
        self.window_encoder = window_encoder
        self.cfg = cfg
        layer = nn.TransformerEncoderLayer(d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=cfg.d_model * 4,
                                           dropout=cfg.dropout, activation='gelu', batch_first=True, norm_first=True)
        self.record_encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)
        self.window_pool = AttentionPool(cfg.d_model)
        self.classifier = nn.Sequential(nn.LayerNorm(cfg.d_model * 2), nn.Dropout(cfg.dropout), nn.Linear(cfg.d_model * 2, cfg.d_model), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(cfg.d_model, 2))
    def forward(self, batch: Dict[str, torch.Tensor], return_attention: bool = False) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        b, w = batch['window_mask'].shape
        flat = {
            'ecg': batch['ecg'].reshape(b * w, cfg.max_beats, len(ECG_LEADS), cfg.beat_len),
            'iegm': batch['iegm'].reshape(b * w, cfg.max_beats, len(IEGM_CHANNELS), cfg.beat_len),
            'beat_mask': batch['beat_mask'].reshape(b * w, cfg.max_beats),
            'ecg_ch_mask': batch['ecg_ch_mask'].reshape(b * w, len(ECG_LEADS)),
            'iegm_ch_mask': batch['iegm_ch_mask'].reshape(b * w, len(IEGM_CHANNELS)),
            'features': batch['features'].reshape(b * w, FEATURE_DIM),
            'mech': batch['mech'].reshape(b * w, cfg.max_beats, MECH_DIM),
        }
        window_logits, aux_w = self.window_encoder(flat, return_recon=False, return_attention=False)
        win_emb = aux_w['embedding'].reshape(b, w, cfg.d_model)
        window_logits = window_logits.reshape(b, w, 2)
        enc = self.record_encoder(win_emb, src_key_padding_mask=~batch['window_mask'].bool())
        att_ctx, att_w = self.window_pool(enc, batch['window_mask'].bool())
        mean_ctx = torch.sum(enc * batch['window_mask'].float().unsqueeze(-1), dim=1) / batch['window_mask'].float().sum(dim=1, keepdim=True).clamp_min(1.0)
        rec_emb = torch.cat([att_ctx, mean_ctx], dim=-1)
        logits = self.classifier(rec_emb)
        aux: Dict[str, torch.Tensor] = {
            'record_embedding': att_ctx,
            'window_logits': window_logits,
            'window_attention': att_w,
            'einthoven': aux_w.get('einthoven', logits.new_tensor(0.0)),
            'cs_order': aux_w.get('cs_order', logits.new_tensor(0.0)),
        }
        return logits, aux

def parameter_count(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


In [15]:
# ============================================================
# 15) Self-supervised pretraining losses
# ============================================================
def make_ssl_mask(x: torch.Tensor, ch_mask: torch.Tensor, beat_mask: torch.Tensor, patch_p: float, channel_p: float, patch_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
    # x: [B,N,C,L]
    b, n, c, l = x.shape
    device = x.device
    valid = beat_mask[:, :, None, None].bool() & ch_mask[:, None, :, None].bool()
    n_patch = max(1, l // patch_len)
    patch_mask = torch.rand((b, n, c, n_patch), device=device) < patch_p
    patch_mask = patch_mask.repeat_interleave(patch_len, dim=-1)[..., :l]
    channel_mask = (torch.rand((b, 1, c, 1), device=device) < channel_p).expand(b, n, c, l)
    mask = (patch_mask | channel_mask) & valid.expand(b, n, c, l)
    x_masked = x.clone()
    x_masked[mask] = 0.0
    return x_masked, mask

def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    mask = mask.bool()
    if mask.sum() < 1:
        return pred.new_tensor(0.0)
    return ((pred - target) ** 2)[mask].mean()

def modal_contrastive_loss(ecg_z: torch.Tensor, iegm_z: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    if ecg_z.shape[0] < 2:
        return ecg_z.new_tensor(0.0)
    a = F.normalize(ecg_z, dim=-1)
    b = F.normalize(iegm_z, dim=-1)
    logits = a @ b.T / temperature
    labels = torch.arange(a.shape[0], device=a.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def ssl_step_loss(model: MechXWindowEncoder, batch: Dict[str, torch.Tensor], cfg_used: CFG) -> Tuple[torch.Tensor, Dict[str, float]]:
    ecg_orig = batch['ecg']
    iegm_orig = batch['iegm']
    ecg_masked, ecg_mask = make_ssl_mask(ecg_orig, batch['ecg_ch_mask'], batch['beat_mask'], cfg_used.ssl_mask_patch_p, cfg_used.ssl_mask_channel_p, cfg_used.ssl_patch_len)
    iegm_masked, iegm_mask = make_ssl_mask(iegm_orig, batch['iegm_ch_mask'], batch['beat_mask'], cfg_used.ssl_mask_patch_p, cfg_used.ssl_mask_channel_p, cfg_used.ssl_patch_len)
    masked_batch = dict(batch)
    masked_batch['ecg'] = ecg_masked
    masked_batch['iegm'] = iegm_masked
    _, aux = model(masked_batch, return_recon=True, return_attention=False)
    loss_ecg = masked_mse(aux['ecg_recon'], ecg_orig, ecg_mask)
    loss_iegm = masked_mse(aux['iegm_recon'], iegm_orig, iegm_mask)
    # Cross-modal prediction is evaluated over valid observed values, not only masked values.
    ecg_valid = batch['beat_mask'][:, :, None, None].bool() & batch['ecg_ch_mask'][:, None, :, None].bool()
    iegm_valid = batch['beat_mask'][:, :, None, None].bool() & batch['iegm_ch_mask'][:, None, :, None].bool()
    loss_cross_ecg = masked_mse(aux['iegm_to_ecg_recon'], ecg_orig, ecg_valid.expand_as(ecg_orig))
    loss_cross_iegm = masked_mse(aux['ecg_to_iegm_recon'], iegm_orig, iegm_valid.expand_as(iegm_orig))
    loss_modal = modal_contrastive_loss(aux['ecg_cls'], aux['iegm_cls'], temperature=cfg_used.contrastive_temperature)
    loss = cfg_used.lambda_ssl_recon * (loss_ecg + loss_iegm) + cfg_used.lambda_ssl_cross_recon * (loss_cross_ecg + loss_cross_iegm) + cfg_used.lambda_ssl_modal_contrast * loss_modal
    return loss, {'ssl_ecg': float(loss_ecg.detach().cpu()), 'ssl_iegm': float(loss_iegm.detach().cpu()), 'ssl_cross_ecg': float(loss_cross_ecg.detach().cpu()), 'ssl_cross_iegm': float(loss_cross_iegm.detach().cpu()), 'ssl_modal': float(loss_modal.detach().cpu())}

def ssl_checkpoint_path(fold: int) -> Path:
    return Path(cfg.model_dir) / f'mechx_window_ssl_fold{fold}.pt'

def pretrain_ssl_for_fold(fold: int, ssl_train_df: pd.DataFrame, feature_dim: int, mech_dim: int, cfg_used: CFG) -> Path:
    ckpt_path = ssl_checkpoint_path(fold)
    if ckpt_path.exists() and not cfg_used.ssl_overwrite:
        print(f'SSL checkpoint exists for fold {fold}: {ckpt_path}')
        return ckpt_path
    print(f'Caching {len(ssl_train_df)} SSL train windows for fold {fold}...')
    build_token_cache(ssl_train_df, overwrite=False)
    model = MechXWindowEncoder(feature_dim, mech_dim, cfg_used).to(cfg_used.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg_used.ssl_lr, weight_decay=cfg_used.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda')))
    loader = make_window_loader(ssl_train_df, cfg_used.ssl_batch_size, train=True, balanced=False)
    history = []
    for epoch in range(1, cfg_used.ssl_epochs + 1):
        model.train()
        losses = []
        parts: Dict[str, List[float]] = {'ssl_ecg': [], 'ssl_iegm': [], 'ssl_cross_ecg': [], 'ssl_cross_iegm': [], 'ssl_modal': []}
        t0 = time.time()
        for batch in loader:
            batch = move_batch(batch, cfg_used.device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda'))):
                loss, loss_parts = ssl_step_loss(model, batch, cfg_used)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg_used.grad_clip)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
            for k in parts:
                parts[k].append(loss_parts[k])
        row = {'epoch': epoch, 'ssl_loss': float(np.mean(losses)), 'seconds': time.time() - t0}
        row.update({k: float(np.mean(v)) for k, v in parts.items()})
        history.append(row)
        print(f'[SSL fold {fold}] epoch {epoch:03d} loss={row["ssl_loss"]:.4f}')
    pd.DataFrame(history).to_csv(Path(cfg.log_dir) / f'ssl_history_fold{fold}.csv', index=False)
    torch.save({'state_dict': model.state_dict(), 'cfg': asdict(cfg_used), 'feature_dim': feature_dim, 'mech_dim': mech_dim, 'fold': fold}, ckpt_path)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ckpt_path


In [16]:
# ============================================================
# 16) Supervised losses, prototype contrastive learning, prediction
# ============================================================
def class_weights_from_df(train_df: pd.DataFrame, device: str) -> torch.Tensor:
    counts = np.bincount(train_df['label'].values.astype(int), minlength=2).astype(np.float32)
    weights = counts.sum() / (2 * np.maximum(counts, 1.0))
    weights = weights / np.mean(weights)
    return torch.tensor(weights, dtype=torch.float32, device=device)

def supervised_loss_per_sample(logits: torch.Tensor, y: torch.Tensor, class_weights: torch.Tensor, cfg_used: CFG) -> torch.Tensor:
    if not cfg_used.use_focal_loss:
        return F.cross_entropy(logits, y, weight=class_weights, reduction='none', label_smoothing=cfg_used.label_smoothing)
    ce = F.cross_entropy(logits, y, weight=class_weights, reduction='none', label_smoothing=cfg_used.label_smoothing)
    pt = torch.exp(-ce).clamp(1e-6, 1.0)
    return ((1.0 - pt) ** cfg_used.focal_gamma) * ce

def supervised_contrastive_loss(z: torch.Tensor, y: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    if z.shape[0] < 2:
        return z.new_tensor(0.0)
    z = F.normalize(z, dim=-1)
    sim = z @ z.T / temperature
    y = y.view(-1, 1)
    pos = torch.eq(y, y.T).float().to(z.device)
    eye = torch.eye(z.shape[0], device=z.device)
    pos = pos * (1.0 - eye)
    if pos.sum() < 1:
        return z.new_tensor(0.0)
    sim = sim - torch.max(sim, dim=1, keepdim=True)[0].detach()
    exp_sim = torch.exp(sim) * (1.0 - eye)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True).clamp_min(1e-8))
    mean_log_prob_pos = (pos * log_prob).sum(dim=1) / pos.sum(dim=1).clamp_min(1.0)
    valid = (pos.sum(dim=1) > 0).float()
    return -(valid * mean_log_prob_pos).sum() / valid.sum().clamp_min(1.0)

def prototype_margin_loss(z: torch.Tensor, y: torch.Tensor, prototypes: Optional[torch.Tensor], margin: float = 0.25) -> torch.Tensor:
    if prototypes is None:
        return z.new_tensor(0.0)
    z = F.normalize(z, dim=-1)
    p = F.normalize(prototypes.to(z.device), dim=-1)
    d = torch.cdist(z, p, p=2)  # [B,2]
    own = d[torch.arange(z.shape[0], device=z.device), y]
    other = d[torch.arange(z.shape[0], device=z.device), 1 - y]
    return F.relu(own - other + margin).mean()

@torch.no_grad()
def compute_record_prototypes(model: RecordMILMechXFormer, loader: DataLoader, device: str) -> Optional[torch.Tensor]:
    model.eval()
    embs, ys = [], []
    for batch in loader:
        batch = move_batch(batch, device)
        logits, aux = model(batch, return_attention=False)
        embs.append(aux['record_embedding'].detach().cpu())
        ys.append(batch['label'].detach().cpu())
    if not embs:
        return None
    z = torch.cat(embs, dim=0)
    y = torch.cat(ys, dim=0)
    protos = []
    for c in [0, 1]:
        if (y == c).sum() < 1:
            protos.append(torch.zeros(z.shape[1]))
        else:
            protos.append(z[y == c].mean(dim=0))
    return torch.stack(protos, dim=0)

def tta_perturb_batch(batch: Dict[str, Any], rep: int) -> Dict[str, Any]:
    if rep == 0:
        return batch
    out = dict(batch)
    for key, ch_key in [('ecg', 'ecg_ch_mask'), ('iegm', 'iegm_ch_mask')]:
        x = out[key].clone()
        scale = torch.randn((*x.shape[:-2], x.shape[-2], 1), device=x.device, dtype=x.dtype) * cfg.aug_scale_std + 1.0
        x = x * scale
        x = x + torch.randn_like(x) * cfg.aug_noise_std
        max_shift = int(round(cfg.aug_time_shift_ms * cfg.beat_len / ((cfg.pre_r_sec + cfg.post_r_sec) * 1000)))
        if max_shift > 0:
            sh = int(torch.randint(-max_shift, max_shift + 1, (1,), device=x.device).item())
            if sh != 0:
                x = torch.roll(x, shifts=sh, dims=-1)
        out[key] = x
    return out

@torch.no_grad()
def predict_window_loader(model: MechXWindowEncoder, loader: DataLoader, device: str, temperature: float = 1.0, threshold: float = 0.5, use_tta: bool = False) -> pd.DataFrame:
    model.eval()
    rows = []
    reps = cfg.n_tta if use_tta else 1
    for batch in loader:
        records = batch['record']
        starts = batch['start'].numpy()
        ends = batch['end'].numpy()
        row_ids = batch['row_id'].numpy()
        labels = batch['label'].numpy()
        batch_dev = move_batch(batch, device)
        logits_acc = None
        for r in range(reps):
            b2 = tta_perturb_batch(batch_dev, r) if use_tta else batch_dev
            logits, aux = model(b2, return_recon=False, return_attention=False)
            logits_acc = logits if logits_acc is None else logits_acc + logits
        logits = (logits_acc / reps).detach().cpu().numpy()
        probs = softmax_np(logits / float(temperature))
        pred = (probs[:, 1] >= threshold).astype(int)
        for i in range(len(labels)):
            rows.append({
                'record': records[i], 'row_id': int(row_ids[i]), 'start': int(starts[i]), 'end': int(ends[i]),
                'label': int(labels[i]), 'label_name': BINARY_LABELS[int(labels[i])],
                'pred': int(pred[i]), 'threshold': float(threshold), 'confidence': float(probs[i].max()),
                'prob_AVNRT': float(probs[i, 0]), 'prob_AVRT': float(probs[i, 1]),
                'logit_AVNRT': float(logits[i, 0]), 'logit_AVRT': float(logits[i, 1]),
            })
    return pd.DataFrame(rows)

@torch.no_grad()
def predict_record_loader(model: RecordMILMechXFormer, loader: DataLoader, device: str, temperature: float = 1.0, threshold: float = 0.5, use_tta: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
    model.eval()
    record_rows, window_rows = [], []
    reps = cfg.n_tta if use_tta else 1
    for batch in loader:
        records = batch['record']
        labels_np = batch['label'].numpy()
        batch_dev = move_batch(batch, device)
        logits_acc = None
        aux_last = None
        for r in range(reps):
            b2 = tta_perturb_batch(batch_dev, r) if use_tta else batch_dev
            logits, aux = model(b2, return_attention=True)
            logits_acc = logits if logits_acc is None else logits_acc + logits
            aux_last = aux
        logits = (logits_acc / reps).detach().cpu().numpy()
        probs = softmax_np(logits / float(temperature))
        pred = (probs[:, 1] >= threshold).astype(int)
        att = aux_last['window_attention'].detach().cpu().numpy()
        win_logits = aux_last['window_logits'].detach().cpu().numpy()
        bsz, max_w = batch['window_mask'].shape
        for i in range(bsz):
            record_rows.append({
                'record': records[i], 'label': int(labels_np[i]), 'label_name': BINARY_LABELS[int(labels_np[i])],
                'pred': int(pred[i]), 'threshold': float(threshold), 'confidence': float(probs[i].max()),
                'prob_AVNRT': float(probs[i, 0]), 'prob_AVRT': float(probs[i, 1]),
                'logit_AVNRT': float(logits[i, 0]), 'logit_AVRT': float(logits[i, 1]),
                'n_windows': int(batch['window_mask'][i].sum().item()),
            })
            for j in range(max_w):
                if not bool(batch['window_mask'][i, j]):
                    continue
                wp = softmax_np(win_logits[i, j:j+1])[0]
                window_rows.append({
                    'record': records[i], 'row_id': int(batch['row_ids'][i, j].item()), 'start': int(batch['starts'][i, j].item()), 'end': int(batch['ends'][i, j].item()),
                    'label': int(labels_np[i]), 'label_name': BINARY_LABELS[int(labels_np[i])],
                    'window_attention': float(att[i, j]), 'prob_AVNRT': float(wp[0]), 'prob_AVRT': float(wp[1]),
                })
    return pd.DataFrame(record_rows), pd.DataFrame(window_rows)


In [17]:
# ============================================================
# 17) Train scratch/SSL window classifier for ablation
# ============================================================
def train_window_transformer_fold(
    model_name: str,
    df_task: pd.DataFrame,
    fold: int,
    feature_dim: int,
    mech_dim: int,
    cfg_used: CFG,
    ssl_ckpt: Optional[Path] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    seed_everything(cfg_used.seed + fold * 1000 + sum(ord(c) for c in model_name))
    outer_train = df_task[df_task.fold != fold].reset_index(drop=True)
    outer_test = df_task[df_task.fold == fold].reset_index(drop=True)
    inner_train, inner_val = make_inner_split(outer_train, seed=cfg_used.seed + 777 + fold, n_inner_splits=cfg_used.n_inner_splits)
    train_loader = make_window_loader(inner_train, cfg_used.batch_size, train=True, balanced=True)
    val_loader = make_window_loader(inner_val, cfg_used.batch_size, train=False)
    test_loader = make_window_loader(outer_test, cfg_used.batch_size, train=False)

    model = MechXWindowEncoder(feature_dim, mech_dim, cfg_used).to(cfg_used.device)
    if ssl_ckpt is not None and Path(ssl_ckpt).exists():
        ckpt = torch.load(ssl_ckpt, map_location=cfg_used.device)
        missing, unexpected = model.load_state_dict(ckpt['state_dict'], strict=False)
        print(f'Loaded SSL weights into {model_name}: missing={len(missing)} unexpected={len(unexpected)}')
    class_weights = class_weights_from_df(inner_train, cfg_used.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg_used.lr_scratch if ssl_ckpt is None else cfg_used.lr_head, weight_decay=cfg_used.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda')))
    best_metric, best_epoch, patience = -1.0, -1, 0
    ckpt_path = Path(cfg_used.model_dir) / f'ckpt_{model_name}_fold{fold}.pt'
    history = []
    t_train = time.time()
    for epoch in range(1, cfg_used.epochs_window + 1):
        model.train()
        losses = []
        for batch in train_loader:
            batch = move_batch(batch, cfg_used.device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda'))):
                logits, aux = model(batch, return_recon=False, return_attention=False)
                sup = supervised_loss_per_sample(logits, batch['label'], class_weights, cfg_used).mean()
                loss = sup + 0.005 * aux.get('einthoven', logits.new_tensor(0.0)) + 0.005 * aux.get('cs_order', logits.new_tensor(0.0))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg_used.grad_clip)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        val_pred = predict_window_loader(model, val_loader, cfg_used.device, temperature=1.0, threshold=0.5, use_tta=False)
        val_metrics = compute_binary_metrics(val_pred['label'].values, val_pred[['prob_AVNRT','prob_AVRT']].values, prefix='val_', threshold=0.5)
        row = {'epoch': epoch, 'train_loss': float(np.mean(losses)), **val_metrics}
        history.append(row)
        print(f'[{model_name}/fold{fold}] epoch {epoch:03d} loss={row["train_loss"]:.4f} valF1={row["val_macro_f1"]:.4f}')
        if val_metrics['val_macro_f1'] > best_metric + 1e-5:
            best_metric = val_metrics['val_macro_f1']
            best_epoch = epoch
            patience = 0
            torch.save({'state_dict': model.state_dict(), 'model_name': model_name, 'cfg': asdict(cfg_used), 'feature_dim': feature_dim, 'mech_dim': mech_dim, 'fold': fold, 'epoch': epoch}, ckpt_path)
        else:
            patience += 1
            if patience >= cfg_used.patience:
                print(f'Early stopping at epoch {epoch}; best epoch {best_epoch}.')
                break
    training_time = time.time() - t_train
    pd.DataFrame(history).to_csv(Path(cfg_used.log_dir) / f'history_{model_name}_fold{fold}.csv', index=False)
    ckpt = torch.load(ckpt_path, map_location=cfg_used.device)
    model.load_state_dict(ckpt['state_dict'])
    val_logits_df = predict_window_loader(model, val_loader, cfg_used.device, temperature=1.0, threshold=0.5, use_tta=False)
    val_logits = val_logits_df[['logit_AVNRT', 'logit_AVRT']].values
    y_val = val_logits_df['label'].values.astype(int)
    temperature = fit_temperature(val_logits, y_val, cfg_used.device) if cfg_used.use_temperature_scaling else 1.0
    val_probs_cal = softmax_np(val_logits / temperature)
    threshold, _ = find_best_threshold(y_val, val_probs_cal[:, 1]) if cfg_used.tune_threshold else (0.5, np.nan)
    test_pred = predict_window_loader(model, test_loader, cfg_used.device, temperature=temperature, threshold=threshold, use_tta=cfg_used.use_tta)
    test_pred['model'] = model_name
    test_pred['ablation'] = 'window_level'
    test_pred['fold'] = int(fold)
    test_pred['temperature'] = float(temperature)
    test_pred = test_pred.merge(outer_test[['row_id','pace_frac','cohort_group']], on='row_id', how='left')
    record_pred = aggregate_window_predictions_to_records(test_pred)
    record_pred['model'] = model_name
    record_pred['ablation'] = 'window_level'
    record_pred['fold'] = int(fold)
    win_metrics = compute_binary_metrics(test_pred['label'].values, test_pred[['prob_AVNRT','prob_AVRT']].values, prefix='window_', threshold=threshold, pred=test_pred['pred'].values)
    rec_metrics = compute_binary_metrics(record_pred['label'].values, record_pred[['prob_AVNRT','prob_AVRT']].values, prefix='record_', threshold=threshold, pred=record_pred['pred'].values)
    result_df = pd.DataFrame([{ 'model': model_name, 'ablation': 'window_level', 'fold': int(fold), 'n_params': parameter_count(model), 'best_epoch': best_epoch, 'training_time_sec': training_time, 'temperature': temperature, 'threshold': threshold, **win_metrics, **rec_metrics }])
    result_df.to_csv(Path(cfg_used.table_dir) / f'result_{model_name}_fold{fold}.csv', index=False)
    test_pred.to_csv(Path(cfg_used.table_dir) / f'window_predictions_{model_name}_fold{fold}.csv', index=False)
    record_pred.to_csv(Path(cfg_used.table_dir) / f'record_predictions_{model_name}_fold{fold}.csv', index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df, test_pred, record_pred


In [18]:
# ============================================================
# 18) Full record-level MIL training
# ============================================================
def train_mechxformer_mil_fold(
    df_task: pd.DataFrame,
    fold: int,
    feature_dim: int,
    mech_dim: int,
    cfg_used: CFG,
    ssl_ckpt: Optional[Path] = None,
    ablation: str = 'full',
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    model_name = 'mechxformer_av_mil'
    seed_everything(cfg_used.seed + fold * 1000 + 9301 + sum(ord(c) for c in ablation))
    outer_train = df_task[df_task.fold != fold].reset_index(drop=True)
    outer_test = df_task[df_task.fold == fold].reset_index(drop=True)
    inner_train, inner_val = make_inner_split(outer_train, seed=cfg_used.seed + 1234 + fold, n_inner_splits=cfg_used.n_inner_splits)

    train_loader = make_record_loader(inner_train, train=True)
    train_eval_loader = make_record_loader(inner_train, train=False)
    val_loader = make_record_loader(inner_val, train=False)
    test_loader = make_record_loader(outer_test, train=False)

    window_encoder = MechXWindowEncoder(feature_dim, mech_dim, cfg_used)
    if ssl_ckpt is not None and Path(ssl_ckpt).exists() and ablation != 'no_ssl_pretraining':
        ckpt = torch.load(ssl_ckpt, map_location=cfg_used.device)
        missing, unexpected = window_encoder.load_state_dict(ckpt['state_dict'], strict=False)
        print(f'Loaded SSL window encoder: missing={len(missing)} unexpected={len(unexpected)}')
    model = RecordMILMechXFormer(window_encoder, cfg_used).to(cfg_used.device)

    # Differential LR: pretrained encoder receives lower LR; MIL/classifier receives higher LR.
    encoder_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('window_encoder'):
            encoder_params.append(p)
        else:
            head_params.append(p)
    opt = torch.optim.AdamW([
        {'params': encoder_params, 'lr': cfg_used.lr_encoder if ssl_ckpt is not None else cfg_used.lr_scratch},
        {'params': head_params, 'lr': cfg_used.lr_head},
    ], weight_decay=cfg_used.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda')))
    class_weights = class_weights_from_df(inner_train, cfg_used.device)
    best_metric, best_epoch, patience = -1.0, -1, 0
    ckpt_path = Path(cfg_used.model_dir) / f'ckpt_{model_name}_{ablation}_fold{fold}.pt'
    history = []
    training_start = time.time()
    prototypes = None
    for epoch in range(1, cfg_used.epochs_mil + 1):
        if epoch > cfg_used.prototype_warmup_epochs and ablation != 'no_prototype_contrast':
            prototypes = compute_record_prototypes(model, train_eval_loader, cfg_used.device)
        model.train()
        losses = []
        parts = {'ce': [], 'window_aux': [], 'supcon': [], 'proto': [], 'ein': [], 'cs': []}
        for batch in train_loader:
            batch = move_batch(batch, cfg_used.device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(cfg_used.use_amp and cfg_used.device.startswith('cuda'))):
                logits, aux = model(batch, return_attention=False)
                ce = supervised_loss_per_sample(logits, batch['label'], class_weights, cfg_used).mean()
                # Window auxiliary CE: every binary window in the patient's bag inherits record label.
                w_logits = aux['window_logits']
                b, w, _ = w_logits.shape
                y_win = batch['label'][:, None].expand(b, w).reshape(-1)
                wm = batch['window_mask'].reshape(-1).bool()
                win_loss = F.cross_entropy(w_logits.reshape(b * w, 2)[wm], y_win[wm], weight=class_weights, label_smoothing=cfg_used.label_smoothing) if wm.sum() else logits.new_tensor(0.0)
                supcon = supervised_contrastive_loss(aux['record_embedding'], batch['label'], cfg_used.contrastive_temperature) if ablation != 'no_supcon' else logits.new_tensor(0.0)
                proto = prototype_margin_loss(aux['record_embedding'], batch['label'], prototypes) if ablation != 'no_prototype_contrast' else logits.new_tensor(0.0)
                ein = aux.get('einthoven', logits.new_tensor(0.0))
                cs = aux.get('cs_order', logits.new_tensor(0.0))
                loss = ce + cfg_used.window_aux_lambda * win_loss + cfg_used.supcon_lambda * supcon + cfg_used.prototype_lambda * proto + 0.005 * ein + 0.005 * cs
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg_used.grad_clip)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
            parts['ce'].append(float(ce.detach().cpu()))
            parts['window_aux'].append(float(win_loss.detach().cpu()))
            parts['supcon'].append(float(supcon.detach().cpu()))
            parts['proto'].append(float(proto.detach().cpu()))
            parts['ein'].append(float(ein.detach().cpu()))
            parts['cs'].append(float(cs.detach().cpu()))
        val_pred, _ = predict_record_loader(model, val_loader, cfg_used.device, temperature=1.0, threshold=0.5, use_tta=False)
        val_metrics = compute_binary_metrics(val_pred['label'].values, val_pred[['prob_AVNRT','prob_AVRT']].values, prefix='val_', threshold=0.5, pred=val_pred['pred'].values)
        row = {'epoch': epoch, 'train_loss': float(np.mean(losses)), **{f'loss_{k}': float(np.mean(v)) for k, v in parts.items()}, **val_metrics}
        history.append(row)
        print(f'[{model_name}/{ablation}/fold{fold}] epoch {epoch:03d} loss={row["train_loss"]:.4f} valF1={row["val_macro_f1"]:.4f}')
        if val_metrics['val_macro_f1'] > best_metric + 1e-5:
            best_metric = val_metrics['val_macro_f1']
            best_epoch = epoch
            patience = 0
            torch.save({'state_dict': model.state_dict(), 'model_name': model_name, 'cfg': asdict(cfg_used), 'feature_dim': feature_dim, 'mech_dim': mech_dim, 'fold': fold, 'epoch': epoch, 'ablation': ablation}, ckpt_path)
        else:
            patience += 1
            if patience >= cfg_used.patience:
                print(f'Early stopping at epoch {epoch}; best epoch {best_epoch}.')
                break
    training_time = time.time() - training_start
    pd.DataFrame(history).to_csv(Path(cfg_used.log_dir) / f'history_{model_name}_{ablation}_fold{fold}.csv', index=False)
    ckpt = torch.load(ckpt_path, map_location=cfg_used.device)
    model.load_state_dict(ckpt['state_dict'])

    val_pred, _ = predict_record_loader(model, val_loader, cfg_used.device, temperature=1.0, threshold=0.5, use_tta=False)
    val_logits = val_pred[['logit_AVNRT', 'logit_AVRT']].values
    y_val = val_pred['label'].values.astype(int)
    temperature = fit_temperature(val_logits, y_val, cfg_used.device) if cfg_used.use_temperature_scaling else 1.0
    val_probs_cal = softmax_np(val_logits / temperature)
    threshold, _ = find_best_threshold(y_val, val_probs_cal[:, 1]) if cfg_used.tune_threshold else (0.5, np.nan)

    record_pred, window_att = predict_record_loader(model, test_loader, cfg_used.device, temperature=temperature, threshold=threshold, use_tta=cfg_used.use_tta)
    record_pred['model'] = model_name
    record_pred['ablation'] = ablation
    record_pred['fold'] = int(fold)
    record_pred['temperature'] = float(temperature)
    record_pred = record_pred.merge(outer_test.groupby('record')[['cohort_group','pace_frac']].mean(numeric_only=True).reset_index(), on='record', how='left')
    window_att['model'] = model_name
    window_att['ablation'] = ablation
    window_att['fold'] = int(fold)
    rec_metrics = compute_binary_metrics(record_pred['label'].values, record_pred[['prob_AVNRT','prob_AVRT']].values, prefix='record_', threshold=threshold, pred=record_pred['pred'].values)
    result_df = pd.DataFrame([{ 'model': model_name, 'ablation': ablation, 'fold': int(fold), 'n_params': parameter_count(model), 'best_epoch': best_epoch, 'training_time_sec': training_time, 'temperature': temperature, 'threshold': threshold, **rec_metrics }])
    result_df.to_csv(Path(cfg_used.table_dir) / f'result_{model_name}_{ablation}_fold{fold}.csv', index=False)
    record_pred.to_csv(Path(cfg_used.table_dir) / f'record_predictions_{model_name}_{ablation}_fold{fold}.csv', index=False)
    window_att.to_csv(Path(cfg_used.table_dir) / f'window_attention_{model_name}_{ablation}_fold{fold}.csv', index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df, record_pred, window_att, pd.DataFrame(history)


In [19]:
# ============================================================
# 19) Summary, bootstrap CIs, statistical comparisons, plotting
# ============================================================
def summarize_predictions(pred_df: pd.DataFrame, level: str = 'record') -> pd.DataFrame:
    rows = []
    for (model, ablation), g in pred_df.groupby(['model', 'ablation']):
        probs = g[['prob_AVNRT', 'prob_AVRT']].values.astype(float)
        pred = g['pred'].values.astype(int) if 'pred' in g else None
        threshold = float(g['threshold'].iloc[0]) if 'threshold' in g else 0.5
        metrics = compute_binary_metrics(g['label'].values.astype(int), probs, prefix=f'{level}_', threshold=threshold, pred=pred)
        rows.append({'model': model, 'ablation': ablation, 'n': int(len(g)), **metrics})
    return pd.DataFrame(rows).sort_values(f'{level}_macro_f1', ascending=False)

def metric_value(y: np.ndarray, p: np.ndarray, metric: str, pred: Optional[np.ndarray] = None, threshold: float = 0.5) -> float:
    return compute_binary_metrics(y, p, prefix='', pred=pred, threshold=threshold).get(metric, float('nan'))

def clustered_bootstrap_ci(pred_df: pd.DataFrame, level: str = 'record', n_boot: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    id_col = 'record' if 'record' in pred_df.columns else 'row_id'
    metrics = ['macro_f1', 'roc_auc', 'pr_auc', 'balanced_accuracy', 'sensitivity_AVRT', 'specificity_AVRT', 'mcc', 'ece']
    for (model, ablation), g in pred_df.groupby(['model', 'ablation']):
        ids = np.asarray(sorted(g[id_col].unique()))
        if len(ids) < 2:
            continue
        threshold = float(g['threshold'].iloc[0]) if 'threshold' in g else 0.5
        base = compute_binary_metrics(g['label'].values.astype(int), g[['prob_AVNRT','prob_AVRT']].values.astype(float), pred=g['pred'].values.astype(int) if 'pred' in g else None, threshold=threshold, prefix='')
        for metric in metrics:
            vals = []
            for _ in range(n_boot):
                sampled_ids = rng.choice(ids, size=len(ids), replace=True)
                boot = pd.concat([g[g[id_col] == sid] for sid in sampled_ids], ignore_index=True)
                y = boot['label'].values.astype(int)
                p = boot[['prob_AVNRT','prob_AVRT']].values.astype(float)
                pr = boot['pred'].values.astype(int) if 'pred' in boot else None
                vals.append(metric_value(y, p, metric, pred=pr, threshold=threshold))
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            lo, hi = np.percentile(vals, [2.5, 97.5]) if vals.size else (np.nan, np.nan)
            rows.append({'model': model, 'ablation': ablation, 'level': level, 'metric': metric, 'estimate': base.get(metric, np.nan), 'ci_low': float(lo), 'ci_high': float(hi), 'n_units': int(len(ids))})
    return pd.DataFrame(rows)

def mcnemar_test_paired(record_pred: pd.DataFrame, baseline_model: str, proposed_model: str = 'mechxformer_av_mil', proposed_ablation: str = 'full') -> Dict[str, Any]:
    base = record_pred[record_pred.model == baseline_model].copy()
    prop = record_pred[(record_pred.model == proposed_model) & (record_pred.ablation == proposed_ablation)].copy()
    merged = base.merge(prop, on=['record', 'fold', 'label'], suffixes=('_base', '_prop'))
    if merged.empty:
        return {'baseline': baseline_model, 'proposed': proposed_model, 'n': 0, 'b': np.nan, 'c': np.nan, 'p_value': np.nan}
    base_correct = merged['pred_base'].values == merged['label'].values
    prop_correct = merged['pred_prop'].values == merged['label'].values
    b = int(np.sum(base_correct & ~prop_correct))
    c = int(np.sum(~base_correct & prop_correct))
    p = float(stats.binomtest(min(b, c), n=b+c, p=0.5).pvalue) if (b+c) > 0 else 1.0
    return {'baseline': baseline_model, 'proposed': proposed_model, 'n': int(len(merged)), 'baseline_correct_proposed_wrong': b, 'baseline_wrong_proposed_correct': c, 'p_value': p}

def paired_permutation_macro_f1(record_pred: pd.DataFrame, baseline_model: str, proposed_model: str = 'mechxformer_av_mil', proposed_ablation: str = 'full', n_perm: int = 5000, seed: int = 42) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    base = record_pred[record_pred.model == baseline_model].copy()
    prop = record_pred[(record_pred.model == proposed_model) & (record_pred.ablation == proposed_ablation)].copy()
    merged = base.merge(prop, on=['record', 'fold', 'label'], suffixes=('_base', '_prop'))
    if merged.empty:
        return {'baseline': baseline_model, 'proposed': proposed_model, 'n': 0, 'delta_macro_f1': np.nan, 'p_value': np.nan}
    y = merged['label'].values.astype(int)
    pb = merged[['prob_AVNRT_base','prob_AVRT_base']].values.astype(float)
    pp = merged[['prob_AVNRT_prop','prob_AVRT_prop']].values.astype(float)
    pred_b = merged['pred_base'].values.astype(int)
    pred_p = merged['pred_prop'].values.astype(int)
    obs = f1_score(y, pred_p, average='macro', zero_division=0) - f1_score(y, pred_b, average='macro', zero_division=0)
    count = 0
    for _ in range(n_perm):
        swap = rng.random(len(y)) < 0.5
        pr1 = pred_p.copy(); pr0 = pred_b.copy()
        pr1[swap], pr0[swap] = pr0[swap], pr1[swap]
        delta = f1_score(y, pr1, average='macro', zero_division=0) - f1_score(y, pr0, average='macro', zero_division=0)
        count += int(abs(delta) >= abs(obs))
    return {'baseline': baseline_model, 'proposed': proposed_model, 'n': int(len(merged)), 'delta_macro_f1': float(obs), 'p_value': float((count + 1) / (n_perm + 1))}

def plot_confusion(y_true: np.ndarray, pred: np.ndarray, title: str, save_path: Optional[Path] = None) -> None:
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1], BINARY_LABELS)
    ax.set_yticks([0, 1], BINARY_LABELS)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(int(cm[i, j])), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [20]:

# ============================================================
# Input configuration, feature blocks, ablations, and loader extensions
# ============================================================
PROPOSED_MODEL = 'ssl_window_transformer'
PROPOSED_ABLATION = 'window_level'
PROPOSED_ABLATION_ALIASES = ['window_level', 'full']
PROPOSED_DISPLAY_NAME = 'MechSSL-Window Transformer'
PRIMARY_LEVEL = 'record'
PRIMARY_METRIC = 'record_macro_f1'

PRIMARY_BASELINES = [
    'random_forest',
    'hist_gradient_boosting',
    'extra_trees',
    'svm_rbf',
    'xgboost',
    'logistic_regression_l2',
]

TRANSFORMER_INPUT_ABLATIONS = [
    'window_level',
    'ecg_only',
    'iegm_only',
    'no_mechanism_tokens',
    'signal_only',
    'no_handcrafted_features',
]

CLASSICAL_FEATURE_ABLATIONS = [
    'full',
    'ecg_only',
    'iegm_only',
    'no_mechanism_features',
    'morphology_only',
]

REPEAT_SEEDS = [42] if SMOKE_TEST else [42, 2024, 3407]
RPEAK_EXPERIMENTS = ['annotation_first', 'xqrs_only']
CLASSICAL_FULL_ABLATION_ALIASES = ['classical', 'classical_full']

MECH_IDX = {name: i for i, name in enumerate(MECH_TOKEN_NAMES)}


def transformer_ablation_label(mode: str) -> str:
    """Return the normalized transformer ablation label."""
    return PROPOSED_ABLATION if mode in ('full', 'window_level', None) else str(mode)


def classical_ablation_label(mode: str) -> str:
    """Return the normalized classical feature-ablation label."""
    return 'classical' if mode in ('full', 'classical', 'classical_full', None) else f'classical_{mode}'


def is_proposed_model_row(df: pd.DataFrame) -> pd.Series:
    return (df['model'].eq(PROPOSED_MODEL) & df['ablation'].isin(PROPOSED_ABLATION_ALIASES))


def configure_feature_blocks(feature_dim: int, mech_dim: int) -> Dict[str, slice]:
    """Define feature block slices defensively from configured channel/token counts."""
    global BASE_SL, RR_SL, ECG_FEAT_SL, IEGM_FEAT_SL, MECH_FEAT_SL, FEATURE_BLOCKS
    if mech_dim != len(MECH_TOKEN_NAMES):
        raise ValueError(f'MECH_DIM={mech_dim} but len(MECH_TOKEN_NAMES)={len(MECH_TOKEN_NAMES)}; mechanism-token ablations would be unsafe.')
    base_end = 4
    rr_end = base_end + 6
    ecg_end = rr_end + len(ECG_LEADS) * 5
    iegm_end = ecg_end + len(IEGM_CHANNELS) * 5
    mech_end = iegm_end + len(MECH_TOKEN_NAMES) * 4
    if feature_dim != 161:
        warnings.warn(
            f'FEATURE_DIM={feature_dim}; expected 161. Computing feature slices from ECG_LEADS, IEGM_CHANNELS, and MECH_TOKEN_NAMES.',
            RuntimeWarning,
        )
    if feature_dim < mech_end:
        raise ValueError(f'FEATURE_DIM={feature_dim} is smaller than the expected feature layout length {mech_end}.')
    BASE_SL = slice(0, base_end)
    RR_SL = slice(base_end, rr_end)
    ECG_FEAT_SL = slice(rr_end, ecg_end)
    IEGM_FEAT_SL = slice(ecg_end, iegm_end)
    MECH_FEAT_SL = slice(iegm_end, mech_end)
    FEATURE_BLOCKS = {
        'base': BASE_SL,
        'rr': RR_SL,
        'ecg_morphology': ECG_FEAT_SL,
        'iegm_morphology': IEGM_FEAT_SL,
        'mechanism_summary': MECH_FEAT_SL,
    }
    return FEATURE_BLOCKS

FEATURE_BLOCKS = configure_feature_blocks(FEATURE_DIM, MECH_DIM)
save_json({k: [v.start, v.stop] for k, v in FEATURE_BLOCKS.items()}, Path(cfg.log_dir) / 'feature_block_slices.json')

IEGM_DERIVED_MECH_TOKENS = [
    'va_interval_proxy_sec',
    'cs_activation_slope_sec_per_channel',
    'cs_spread_sec',
    'cs_direction_tanh',
    'rva_to_cs_delay_sec',
    'abl_to_cs_delay_sec',
    'iegm_missing_frac',
]
ECG_DERIVED_MECH_TOKENS = [
    'qrs_width_proxy_sec',
    'retrograde_p_score',
    'ecg_missing_frac',
]


def _zero_feature_slice(features: np.ndarray, sl: slice) -> None:
    if features.shape[-1] >= (sl.stop or 0):
        features[..., sl] = 0.0


def _zero_feature_index(features: np.ndarray, idx: int) -> None:
    if features.shape[-1] > idx:
        features[..., idx] = 0.0


def _zero_mech_tokens(mech: np.ndarray, token_names: Sequence[str]) -> None:
    for name in token_names:
        j = MECH_IDX.get(name)
        if j is not None and mech.shape[-1] > j:
            mech[..., j] = 0.0


def apply_input_ablation_np(
    ecg: np.ndarray,
    iegm: np.ndarray,
    beat_mask: np.ndarray,
    ecg_ch_mask: np.ndarray,
    iegm_ch_mask: np.ndarray,
    features: np.ndarray,
    mech: np.ndarray,
    mode: str = 'window_level',
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Apply transformer input ablations without modifying cached npz arrays in place.

    Returns: ecg, iegm, ecg_ch_mask, iegm_ch_mask, features, mech.
    """
    mode = transformer_ablation_label(mode)
    ecg = np.copy(ecg).astype(np.float32)
    iegm = np.copy(iegm).astype(np.float32)
    ecg_ch_mask = np.copy(ecg_ch_mask).astype(bool)
    iegm_ch_mask = np.copy(iegm_ch_mask).astype(bool)
    features = np.copy(features).astype(np.float32)
    mech = np.copy(mech).astype(np.float32)

    if mode == PROPOSED_ABLATION:
        return ecg, iegm, ecg_ch_mask, iegm_ch_mask, features, mech

    if mode == 'ecg_only':
        # Remove the IEGM stream and all IEGM-derived summary/mechanism information.
        iegm[...] = 0.0
        iegm_ch_mask[...] = False
        _zero_feature_slice(features, IEGM_FEAT_SL)
        _zero_feature_index(features, 3)  # n_iegm_present
        _zero_mech_tokens(mech, IEGM_DERIVED_MECH_TOKENS)
        _zero_feature_slice(features, MECH_FEAT_SL)
    elif mode == 'iegm_only':
        # Remove the ECG stream and all ECG-derived summary/mechanism information.
        ecg[...] = 0.0
        ecg_ch_mask[...] = False
        _zero_feature_slice(features, ECG_FEAT_SL)
        _zero_feature_index(features, 2)  # n_ecg_present
        _zero_mech_tokens(mech, ECG_DERIVED_MECH_TOKENS)
        _zero_feature_slice(features, MECH_FEAT_SL)
    elif mode == 'no_mechanism_tokens':
        # Test the contribution of RR/QRS/VA/CS/RVA/ABL/pacing mechanism tokens.
        mech[...] = 0.0
        _zero_feature_index(features, 1)  # pace_frac
        _zero_feature_slice(features, RR_SL)
        _zero_feature_slice(features, MECH_FEAT_SL)
    elif mode == 'signal_only':
        # Waveforms only: no handcrafted feature vector or mechanism tokens.
        features[...] = 0.0
        mech[...] = 0.0
    elif mode == 'no_handcrafted_features':
        # Waveforms + mechanism tokens, no appended handcrafted feature vector.
        features[...] = 0.0
    else:
        raise ValueError(f'Unknown transformer input ablation: {mode}')
    return ecg, iegm, ecg_ch_mask, iegm_ch_mask, features, mech


def ablate_feature_vector(x: np.ndarray, mode: str = 'full') -> np.ndarray:
    """Classical baseline feature ablations. Handles 1-D or 2-D arrays."""
    mode = 'full' if mode in ('classical', 'classical_full', None) else str(mode)
    arr = np.copy(x).astype(np.float32)
    if mode == 'full':
        return arr
    if mode == 'ecg_only':
        _zero_feature_index(arr, 3)  # n_iegm_present
        _zero_feature_slice(arr, IEGM_FEAT_SL)
        _zero_feature_slice(arr, MECH_FEAT_SL)
    elif mode == 'iegm_only':
        _zero_feature_index(arr, 2)  # n_ecg_present
        _zero_feature_slice(arr, ECG_FEAT_SL)
        _zero_feature_slice(arr, MECH_FEAT_SL)
    elif mode == 'no_mechanism_features':
        _zero_feature_index(arr, 1)  # pace fraction
        _zero_feature_slice(arr, RR_SL)
        _zero_feature_slice(arr, MECH_FEAT_SL)
    elif mode == 'morphology_only':
        out = np.zeros_like(arr)
        out[..., ECG_FEAT_SL] = arr[..., ECG_FEAT_SL]
        out[..., IEGM_FEAT_SL] = arr[..., IEGM_FEAT_SL]
        arr = out
    else:
        raise ValueError(f'Unknown classical feature ablation: {mode}')
    return arr


def run_ablation_unit_tests() -> None:
    """Minimal executable assertions proving that key leakage channels are zeroed."""
    ecg = np.ones((cfg.max_beats, len(ECG_LEADS), cfg.beat_len), dtype=np.float32)
    iegm = np.ones((cfg.max_beats, len(IEGM_CHANNELS), cfg.beat_len), dtype=np.float32)
    beat_mask = np.ones(cfg.max_beats, dtype=bool)
    ecg_mask = np.ones(len(ECG_LEADS), dtype=bool)
    iegm_mask = np.ones(len(IEGM_CHANNELS), dtype=bool)
    feats = np.ones(FEATURE_DIM, dtype=np.float32)
    mech = np.ones((cfg.max_beats, MECH_DIM), dtype=np.float32)
    _, iegm2, _, iegm_mask2, feats2, mech2 = apply_input_ablation_np(ecg, iegm, beat_mask, ecg_mask, iegm_mask, feats, mech, 'ecg_only')
    assert np.allclose(iegm2, 0.0) and not iegm_mask2.any()
    assert np.allclose(feats2[IEGM_FEAT_SL], 0.0) and np.allclose(feats2[MECH_FEAT_SL], 0.0)
    assert all(np.allclose(mech2[..., MECH_IDX[n]], 0.0) for n in IEGM_DERIVED_MECH_TOKENS if n in MECH_IDX)
    ecg2, _, ecg_mask2, _, feats3, mech3 = apply_input_ablation_np(ecg, iegm, beat_mask, ecg_mask, iegm_mask, feats, mech, 'iegm_only')
    assert np.allclose(ecg2, 0.0) and not ecg_mask2.any()
    assert np.allclose(feats3[ECG_FEAT_SL], 0.0) and np.allclose(feats3[MECH_FEAT_SL], 0.0)
    assert all(np.allclose(mech3[..., MECH_IDX[n]], 0.0) for n in ECG_DERIVED_MECH_TOKENS if n in MECH_IDX)
    _, _, _, _, feats4, mech4 = apply_input_ablation_np(ecg, iegm, beat_mask, ecg_mask, iegm_mask, feats, mech, 'no_mechanism_tokens')
    assert np.allclose(mech4, 0.0) and np.allclose(feats4[RR_SL], 0.0) and np.allclose(feats4[MECH_FEAT_SL], 0.0)
    assert np.allclose(ablate_feature_vector(feats, 'morphology_only')[BASE_SL], 0.0)
    print('Ablation unit tests passed.')

run_ablation_unit_tests()

# ------------------------------
# Dataset and dataloader overrides with input_ablation support.
# ------------------------------
class SVTWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = False, augment: bool = False, input_ablation: str = PROPOSED_ABLATION):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.augment = bool(augment and train)
        self.input_ablation = transformer_ablation_label(input_ablation)
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        item = load_npz_for_row(row)
        ecg = item['ecg'].astype(np.float32)
        iegm = item['iegm'].astype(np.float32)
        beat_mask = item['beat_mask'].astype(bool)
        ecg_ch_mask = item['ecg_ch_mask'].astype(bool)
        iegm_ch_mask = item['iegm_ch_mask'].astype(bool)
        features = item['features'].astype(np.float32)
        mech = item['mech'].astype(np.float32)
        ecg = augment_tokens_np(ecg, ecg_ch_mask, beat_mask, train=self.augment)
        iegm = augment_tokens_np(iegm, iegm_ch_mask, beat_mask, train=self.augment)
        ecg, iegm, ecg_ch_mask, iegm_ch_mask, features, mech = apply_input_ablation_np(
            ecg, iegm, beat_mask, ecg_ch_mask, iegm_ch_mask, features, mech, mode=self.input_ablation
        )
        return {
            'ecg': torch.tensor(ecg, dtype=torch.float32),
            'iegm': torch.tensor(iegm, dtype=torch.float32),
            'beat_mask': torch.tensor(beat_mask, dtype=torch.bool),
            'ecg_ch_mask': torch.tensor(ecg_ch_mask, dtype=torch.bool),
            'iegm_ch_mask': torch.tensor(iegm_ch_mask, dtype=torch.bool),
            'features': torch.tensor(features, dtype=torch.float32),
            'mech': torch.tensor(mech, dtype=torch.float32),
            'label': torch.tensor(int(item['label']), dtype=torch.long),
            'pace_frac': torch.tensor(float(item['pace_frac']), dtype=torch.float32),
            'row_id': torch.tensor(int(item['row_id']), dtype=torch.long),
            'record': item['record'],
            'start': item['start'],
            'end': item['end'],
        }

class RecordBagDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = False, input_ablation: str = PROPOSED_ABLATION):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.input_ablation = transformer_ablation_label(input_ablation)
        self.records = sorted(self.df['record'].unique().tolist())
    def __len__(self) -> int:
        return len(self.records)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        record = self.records[idx]
        g = self.df[self.df['record'] == record].sort_values('start').reset_index(drop=True)
        label = int(g['label'].mode().iloc[0])
        max_w = cfg.max_windows_per_record_train if self.train else cfg.max_windows_per_record_eval
        if len(g) > max_w:
            if self.train:
                pick = np.sort(np.random.choice(np.arange(len(g)), size=max_w, replace=False))
            else:
                pick = np.linspace(0, len(g) - 1, max_w).round().astype(int)
            g = g.iloc[pick].reset_index(drop=True)
        items = []
        for _, row in g.iterrows():
            item = load_npz_for_row(row)
            ecg = item['ecg'].astype(np.float32)
            iegm = item['iegm'].astype(np.float32)
            beat_mask = item['beat_mask'].astype(bool)
            ecg_ch_mask = item['ecg_ch_mask'].astype(bool)
            iegm_ch_mask = item['iegm_ch_mask'].astype(bool)
            features = item['features'].astype(np.float32)
            mech = item['mech'].astype(np.float32)
            ecg = augment_tokens_np(ecg, ecg_ch_mask, beat_mask, train=self.train)
            iegm = augment_tokens_np(iegm, iegm_ch_mask, beat_mask, train=self.train)
            ecg, iegm, ecg_ch_mask, iegm_ch_mask, features, mech = apply_input_ablation_np(
                ecg, iegm, beat_mask, ecg_ch_mask, iegm_ch_mask, features, mech, mode=self.input_ablation
            )
            items.append({
                'ecg': ecg,
                'iegm': iegm,
                'beat_mask': beat_mask,
                'ecg_ch_mask': ecg_ch_mask,
                'iegm_ch_mask': iegm_ch_mask,
                'features': features,
                'mech': mech,
                'row_id': int(item['row_id']),
                'start': int(row.start),
                'end': int(row.end),
            })
        return {'record': record, 'label': label, 'items': items}


def make_window_loader(
    df_part: pd.DataFrame,
    batch_size: int,
    train: bool = False,
    balanced: bool = False,
    input_ablation: str = PROPOSED_ABLATION,
) -> DataLoader:
    df_part = df_part.reset_index(drop=True)
    ds = SVTWindowDataset(df_part, train=train, augment=train, input_ablation=input_ablation)
    if train and balanced and 'label' in df_part and df_part['label'].isin([0, 1]).all():
        y = df_part['label'].values.astype(int)
        counts = np.bincount(y, minlength=2).astype(np.float32)
        sample_weights = (1.0 / np.maximum(counts, 1.0))[y]
        sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
    return DataLoader(ds, batch_size=batch_size, shuffle=train, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())


def make_record_loader(df_part: pd.DataFrame, train: bool = False, input_ablation: str = PROPOSED_ABLATION) -> DataLoader:
    return DataLoader(
        RecordBagDataset(df_part, train=train, input_ablation=input_ablation),
        batch_size=cfg.record_batch_size,
        shuffle=train,
        collate_fn=record_bag_collate,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )


Ablation unit tests passed.


## V3 configuration and scientific safeguards

The blocks below own all V3 execution. The multimodal reference model and the controlled initialization experiment use a **nested SSL corpus** that excludes both the inner-validation patients and the outer-test patients of each partition. Final operating-point parameters are fitted on **inner-validation patients**, not validation windows.


In [21]:

from scipy import optimize
from contextlib import contextmanager

V3_ROOT = Path('./outputs_mechxformer_av_v3_final_validation')
V3_MODEL_DIR = V3_ROOT / 'models'
V3_TABLE_DIR = V3_ROOT / 'tables'
V3_FIG_DIR = V3_ROOT / 'figures'
V3_LOG_DIR = V3_ROOT / 'logs'
V3_CACHE_DIR = V3_ROOT / 'cache'
for _p in [V3_ROOT, V3_MODEL_DIR, V3_TABLE_DIR, V3_FIG_DIR, V3_LOG_DIR, V3_CACHE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# Keep the reusable base cache inside V3 as the annotation-first representation.
BASE_ANNOTATION_CACHE = Path(cfg.cache_dir)

# Execution controls.
V3_FORCE_RERUN = False
RUN_V3_REFERENCE_MODEL = True
RUN_V3_CLINICAL_REFERENCE_AUDIT = True
RUN_V3_STRICT_MODALITIES = True
RUN_V3_AUTOMATIC_LOCALIZATION = True
RUN_V3_PACING_REMOVAL = True
RUN_V3_REPEATED_MATCHED_SSL = True
RUN_V3_EXTERNAL_VALIDATION_CHECK = True

# Matched training design.
V3_PATIENT_THRESHOLD_GRID = np.linspace(0.05, 0.95, 181)
V3_MIN_PATIENTS_FOR_TEMPERATURE = 4
V3_EARLYSTOP_USE_TTA = True
V3_EARLYSTOP_PATIENCE = cfg.patience
V3_MATCHED_SUPERVISED_LR = 1e-4
V3_REFERENCE_SEED_BASE = 88000
V3_MATCHED_SEED_BASES = [88000, 99000, 110000] if not SMOKE_TEST else [88000]

# Strict single-modality controls.
V3_STRICT_MODALITIES = ['ecg', 'iegm']
V3_STRICT_EPOCHS = cfg.epochs_window
V3_STRICT_PATIENCE = cfg.patience
V3_STRICT_LR = 1e-4
V3_STRICT_BATCH_SIZE = 8
V3_STRICT_N_TOKENS = 64
V3_STRICT_PATIENT_BALANCED_SAMPLING = True

# Localization audit.
V3_PEAK_MATCH_TOLERANCES_MS = [50.0, 100.0, 150.0]
V3_PRIMARY_PEAK_TOLERANCE_MS = 150.0
V3_AUTO_CACHE_DIR = V3_CACHE_DIR / 'automatic_ecg_rpeaks'
V3_AUTO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Optional independent cohort. Leave empty when unavailable.
V3_EXTERNAL_DATA_DIR = os.environ.get('MECHSSL_EXTERNAL_DATA_DIR', '').strip()

# Fixed fold safety.
_record_fold = df_binary[['record','fold']].drop_duplicates()
if _record_fold.groupby('record')['fold'].nunique().max() != 1:
    raise RuntimeError('A patient appears in more than one outer partition.')
if int(df_binary.record.nunique()) < 2:
    raise RuntimeError('Supervised cohort is empty or incomplete.')

V3_MANIFEST = {
    'analysis_release': 'V3 final validation',
    'data_dir': str(DATA_DIR),
    'output_dir': str(V3_ROOT),
    'n_binary_windows': int(len(df_binary)),
    'n_binary_patients': int(df_binary.record.nunique()),
    'outer_partitions': sorted(map(int, df_binary.fold.unique())),
    'patient_threshold_grid': [float(V3_PATIENT_THRESHOLD_GRID.min()), float(V3_PATIENT_THRESHOLD_GRID.max()), int(len(V3_PATIENT_THRESHOLD_GRID))],
    'matched_supervised_lr': float(V3_MATCHED_SUPERVISED_LR),
    'matched_seed_bases': list(map(int, V3_MATCHED_SEED_BASES)),
    'peak_match_tolerances_ms': list(map(float, V3_PEAK_MATCH_TOLERANCES_MS)),
    'notes': [
        'Outer-test labels are never used for model selection, calibration, or threshold fitting.',
        'Nested SSL excludes inner-validation and outer-test patients of the partition.',
        'Patient-level temperature and threshold are fitted on inner-validation patients.',
        'External validation requires a genuinely independent synchronized cohort.',
    ],
}
save_json(V3_MANIFEST, V3_LOG_DIR / 'v3_manifest.json')
print(json.dumps(V3_MANIFEST, indent=2))


{
  "analysis_release": "V3 final validation",
  "data_dir": "C:\\Users\\hasee\\Desktop\\leipzig-heart-centerwith-congenital-heart-disease-1.0.0",
  "output_dir": "outputs_mechxformer_av_v3_final_validation",
  "n_binary_windows": 1410,
  "n_binary_patients": 28,
  "outer_partitions": [
    0,
    1,
    2,
    3,
    4
  ],
  "patient_threshold_grid": [
    0.05,
    0.95,
    181
  ],
  "matched_supervised_lr": 0.0001,
  "matched_seed_bases": [
    88000,
    99000,
    110000
  ],
  "peak_match_tolerances_ms": [
    50.0,
    100.0,
    150.0
  ],
  "notes": [
    "Outer-test labels are never used for model selection, calibration, or threshold fitting.",
    "Nested SSL excludes inner-validation and outer-test patients of the partition.",
    "Patient-level temperature and threshold are fitted on inner-validation patients.",
    "External validation requires a genuinely independent synchronized cohort."
  ]
}


## Patient-level calibration, uncertainty, paired inference, and resumable training


In [22]:

def v3_safe_name(x: Any) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', str(x)).strip('_')


def v3_patient_probs_from_window_logits(window_df: pd.DataFrame, temperature: float = 1.0) -> pd.DataFrame:
    if window_df is None or window_df.empty:
        return pd.DataFrame()
    req = {'record','label','logit_AVNRT','logit_AVRT'}
    missing = req - set(window_df.columns)
    if missing:
        raise ValueError(f'Missing columns for patient aggregation: {sorted(missing)}')
    z = window_df[['logit_AVNRT','logit_AVRT']].to_numpy(float)
    p = softmax_np(z / max(float(temperature), 1e-6))
    tmp = window_df[['record','label']].copy()
    tmp['prob_AVNRT'] = p[:,0]
    tmp['prob_AVRT'] = p[:,1]
    rows=[]
    for record,g in tmp.groupby('record',sort=True):
        probs=g[['prob_AVNRT','prob_AVRT']].to_numpy(float).mean(axis=0)
        probs=probs/np.clip(probs.sum(),1e-12,None)
        rows.append({'record':record,'label':int(g.label.mode().iloc[0]),'prob_AVNRT':float(probs[0]),'prob_AVRT':float(probs[1]),'n_windows':int(len(g))})
    return pd.DataFrame(rows)


def v3_patient_nll(window_df: pd.DataFrame, temperature: float) -> float:
    rec=v3_patient_probs_from_window_logits(window_df,temperature)
    if rec.empty:
        return np.inf
    y=rec.label.to_numpy(int)
    p=rec[['prob_AVNRT','prob_AVRT']].to_numpy(float)
    chosen=np.clip(p[np.arange(len(y)),y],1e-8,1.0)
    return float(-np.log(chosen).mean())


def v3_fit_patient_temperature(window_df: pd.DataFrame) -> Tuple[float,Dict[str,Any]]:
    rec0=v3_patient_probs_from_window_logits(window_df,1.0)
    status={'n_patients':int(len(rec0)),'n_classes':int(rec0.label.nunique()) if len(rec0) else 0}
    if len(rec0)<V3_MIN_PATIENTS_FOR_TEMPERATURE or rec0.label.nunique()<2:
        status['temperature_status']='fallback_T1_small_or_single_class'
        return 1.0,status
    fit=optimize.minimize_scalar(
        lambda log_t:v3_patient_nll(window_df,float(np.exp(log_t))),
        bounds=(math.log(0.05),math.log(20.0)),method='bounded',options={'xatol':1e-4,'maxiter':200})
    T=float(np.exp(fit.x)) if fit.success and np.isfinite(fit.fun) else 1.0
    status.update({'temperature_status':'optimized' if fit.success else 'fallback_T1_optimizer_failure',
                   'patient_nll_T1':v3_patient_nll(window_df,1.0),'patient_nll_Tfit':v3_patient_nll(window_df,T),
                   'optimizer_success':bool(fit.success)})
    return T,status


def v3_select_patient_threshold(record_val: pd.DataFrame) -> Tuple[float,float,Dict[str,Any]]:
    if record_val is None or record_val.empty:
        return 0.5,np.nan,{'threshold_status':'fallback_empty'}
    y=record_val.label.to_numpy(int); p=record_val.prob_AVRT.to_numpy(float)
    if len(np.unique(y))<2:
        return 0.5,np.nan,{'threshold_status':'fallback_single_class','n_patients':int(len(y))}
    best_t,best_v=0.5,-1.0; ties=[]
    for t in V3_PATIENT_THRESHOLD_GRID:
        v=float(f1_score(y,(p>=t).astype(int),average='macro',zero_division=0))
        if v>best_v+1e-12:
            best_t,best_v,ties=float(t),v,[float(t)]
        elif abs(v-best_v)<=1e-12:
            ties.append(float(t))
    return best_t,best_v,{'threshold_status':'optimized_patient_macro_f1','n_patients':int(len(y)),
                          'best_macro_f1':float(best_v),'n_tied_maximizers':int(len(ties)),
                          'lowest_tied_maximizer':float(ties[0] if ties else best_t),
                          'highest_tied_maximizer':float(ties[-1] if ties else best_t)}


def v3_fit_patient_operating_point(val_window_logits: pd.DataFrame):
    T,tstat=v3_fit_patient_temperature(val_window_logits)
    rec=v3_patient_probs_from_window_logits(val_window_logits,T)
    tau,val_f1,hstat=v3_select_patient_threshold(rec)
    if len(rec):
        rec['pred']=(rec.prob_AVRT>=tau).astype(int); rec['temperature']=T; rec['threshold']=tau
    info={**tstat,**hstat,'temperature':float(T),'threshold':float(tau),
          'val_patient_macro_f1_at_threshold':float(val_f1) if pd.notna(val_f1) else np.nan}
    return T,tau,rec,info


def v3_apply_patient_operating_point(window_logits:pd.DataFrame,temperature:float,threshold:float,*,model:str,ablation:str,fold:int):
    w=window_logits.copy()
    z=w[['logit_AVNRT','logit_AVRT']].to_numpy(float)
    p=softmax_np(z/max(float(temperature),1e-6))
    w['prob_AVNRT']=p[:,0]; w['prob_AVRT']=p[:,1]; w['temperature']=float(temperature); w['threshold']=float(threshold)
    w['model']=model; w['ablation']=ablation; w['fold']=int(fold); w['pred']=(w.prob_AVRT>=threshold).astype(int)
    r=v3_patient_probs_from_window_logits(w,temperature)
    r['pred']=(r.prob_AVRT>=threshold).astype(int); r['temperature']=float(temperature); r['threshold']=float(threshold)
    r['model']=model; r['ablation']=ablation; r['fold']=int(fold)
    return w,r


def v3_pooled_record_metrics(record_pred:pd.DataFrame,prefix:str='')->Dict[str,Any]:
    if record_pred is None or record_pred.empty:
        return {prefix+'n_patients':0}
    y=record_pred.label.to_numpy(int); p=record_pred[['prob_AVNRT','prob_AVRT']].to_numpy(float); pred=record_pred.pred.to_numpy(int)
    out=compute_binary_metrics(y,p,prefix=prefix,threshold=0.5,pred=pred)
    out[prefix+'n_patients']=int(len(record_pred))
    return out


def v3_record_bootstrap_ci(record_pred:pd.DataFrame,metric:str='macro_f1',n_boot:int=1000,seed:int=42):
    if record_pred is None or record_pred.empty:return (np.nan,np.nan)
    rng=np.random.default_rng(seed); vals=[]; n=len(record_pred)
    for _ in range(int(n_boot)):
        g=record_pred.iloc[rng.integers(0,n,size=n)]
        y=g.label.to_numpy(int); pr=g.pred.to_numpy(int)
        if metric=='macro_f1':vals.append(f1_score(y,pr,average='macro',zero_division=0))
        elif metric=='balanced_accuracy':vals.append(balanced_accuracy_score(y,pr))
    return tuple(map(float,np.percentile(vals,[2.5,97.5]))) if vals else (np.nan,np.nan)


def v3_paired_permutation_delta(a:pd.DataFrame,b:pd.DataFrame,n_perm:int=5000,seed:int=42)->Dict[str,Any]:
    m=a[['record','label','pred']].merge(b[['record','label','pred']],on=['record','label'],suffixes=('_a','_b'))
    if m.empty:return {'n':0,'delta_macro_f1':np.nan,'p_value':np.nan}
    y=m.label.to_numpy(int); pa=m.pred_a.to_numpy(int); pb=m.pred_b.to_numpy(int)
    obs=float(f1_score(y,pa,average='macro',zero_division=0)-f1_score(y,pb,average='macro',zero_division=0))
    rng=np.random.default_rng(seed); ge=0
    for _ in range(int(n_perm)):
        sw=rng.integers(0,2,size=len(m)).astype(bool); xa=pa.copy(); xb=pb.copy(); xa[sw],xb[sw]=pb[sw],pa[sw]
        d=f1_score(y,xa,average='macro',zero_division=0)-f1_score(y,xb,average='macro',zero_division=0)
        ge+=int(abs(d)>=abs(obs)-1e-12)
    return {'n':int(len(m)),'delta_macro_f1':obs,'p_value':float((ge+1)/(int(n_perm)+1))}


def v3_mcnemar(a:pd.DataFrame,b:pd.DataFrame)->Dict[str,Any]:
    m=a[['record','label','pred']].merge(b[['record','label','pred']],on=['record','label'],suffixes=('_a','_b'))
    if m.empty:return {'n':0,'a_correct_b_wrong':np.nan,'a_wrong_b_correct':np.nan,'p_value':np.nan}
    y=m.label.to_numpy(int); ca=m.pred_a.to_numpy(int)==y; cb=m.pred_b.to_numpy(int)==y
    x=int(np.sum(ca & ~cb)); z=int(np.sum(~ca & cb)); p=float(stats.binomtest(min(x,z),n=x+z,p=0.5).pvalue) if x+z else 1.0
    return {'n':int(len(m)),'a_correct_b_wrong':x,'a_wrong_b_correct':z,'p_value':p}


def v3_holm(pvals:Sequence[float])->List[float]:
    p=np.asarray([np.nan if v is None else float(v) for v in pvals],float); out=np.full_like(p,np.nan); finite=np.where(np.isfinite(p))[0]
    if not len(finite):return out.tolist()
    order=finite[np.argsort(p[finite])]; m=len(order); running=0.0
    for rank,idx in enumerate(order):
        adj=min(1.0,(m-rank)*p[idx]); running=max(running,adj); out[idx]=running
    return out.tolist()


@contextmanager
def v3_cache_context(cache_dir:Path,rpeak_source:Optional[str]=None):
    old_cache,old_source=cfg.cache_dir,cfg.rpeak_source
    cfg.cache_dir=str(cache_dir)
    if rpeak_source is not None:cfg.rpeak_source=str(rpeak_source)
    try:yield
    finally:
        cfg.cache_dir=old_cache; cfg.rpeak_source=old_source


def v3_train_patient_endpoint_fold(df_task:pd.DataFrame,fold:int,*,model_name:str,ablation:str,init_ckpt:Optional[Path],supervised_lr:float,experiment_tag:str,seed_value:int,force:bool=V3_FORCE_RERUN):
    out_dir=V3_ROOT/experiment_tag; model_dir=out_dir/'models'; table_dir=out_dir/'tables'; log_dir=out_dir/'logs'
    for p in [model_dir,table_dir,log_dir]:p.mkdir(parents=True,exist_ok=True)
    sm,sa=v3_safe_name(model_name),v3_safe_name(ablation)
    rp=table_dir/f'record_predictions_{sm}_{sa}_fold{fold}.csv'; wp=table_dir/f'window_predictions_{sm}_{sa}_fold{fold}.csv'; pp=table_dir/f'fold_parameters_{sm}_{sa}_fold{fold}.csv'; rr=table_dir/f'result_{sm}_{sa}_fold{fold}.csv'
    if all(p.exists() for p in [rp,wp,pp,rr]) and not force:
        return {'record_pred':pd.read_csv(rp),'window_pred':pd.read_csv(wp),'params':pd.read_csv(pp),'result':pd.read_csv(rr)}
    seed_everything(int(seed_value))
    outer_train=df_task[df_task.fold!=fold].reset_index(drop=True); outer_test=df_task[df_task.fold==fold].reset_index(drop=True)
    inner_train,inner_val=make_inner_split(outer_train,seed=cfg.seed+777+fold,n_inner_splits=cfg.n_inner_splits)
    tr=make_window_loader(inner_train,cfg.batch_size,train=True,balanced=True,input_ablation=ablation)
    va=make_window_loader(inner_val,cfg.batch_size,train=False,input_ablation=ablation)
    te=make_window_loader(outer_test,cfg.batch_size,train=False,input_ablation=ablation)
    seed_everything(int(seed_value)); model=MechXWindowEncoder(FEATURE_DIM,MECH_DIM,cfg).to(cfg.device); init_status='random_initialization'
    if init_ckpt is not None:
        ck=torch.load(init_ckpt,map_location=cfg.device); state=ck.get('state_dict',ck) if isinstance(ck,dict) else ck; model.load_state_dict(state,strict=False); init_status=str(init_ckpt)
    cw=class_weights_from_df(inner_train,cfg.device); opt=torch.optim.AdamW(model.parameters(),lr=float(supervised_lr),weight_decay=cfg.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda'))); ckpt=model_dir/f'ckpt_{sm}_{sa}_fold{fold}.pt'
    hist=[]; best=-1.; best_epoch=-1; wait=0; t0=time.time()
    for epoch in range(1,cfg.epochs_window+1):
        model.train(); losses=[]
        for batch in tr:
            batch=move_batch(batch,cfg.device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda'))):
                logits,aux=model(batch,return_recon=False,return_attention=False); sup=supervised_loss_per_sample(logits,batch['label'],cw,cfg).mean(); loss=sup+0.005*aux.get('einthoven',logits.new_tensor(0.0))+0.005*aux.get('cs_order',logits.new_tensor(0.0))
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        vl=predict_window_loader(model,va,cfg.device,temperature=1.0,threshold=0.5,use_tta=V3_EARLYSTOP_USE_TTA)
        vr=v3_patient_probs_from_window_logits(vl,1.0); vf=float(f1_score(vr.label.to_numpy(int),(vr.prob_AVRT.to_numpy(float)>=0.5).astype(int),average='macro',zero_division=0)) if len(vr) else -1.
        hist.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'val_patient_macro_f1_at_0.5':vf}); print(f'[{experiment_tag}/{model_name}/{ablation}/fold{fold}] epoch={epoch:03d} loss={np.mean(losses):.4f} valPatientF1={vf:.4f}')
        if vf>best+1e-5:
            best,best_epoch,wait=vf,epoch,0; torch.save({'state_dict':model.state_dict(),'fold':fold,'epoch':epoch,'seed':int(seed_value),'lr':float(supervised_lr),'ablation':ablation},ckpt)
        else:
            wait+=1
            if wait>=V3_EARLYSTOP_PATIENCE:break
    pd.DataFrame(hist).to_csv(log_dir/f'history_{sm}_{sa}_fold{fold}.csv',index=False)
    model.load_state_dict(torch.load(ckpt,map_location=cfg.device)['state_dict']); training_time=time.time()-t0
    vl=predict_window_loader(model,va,cfg.device,temperature=1.0,threshold=0.5,use_tta=cfg.use_tta); T,tau,vrec,info=v3_fit_patient_operating_point(vl)
    tl=predict_window_loader(model,te,cfg.device,temperature=1.0,threshold=0.5,use_tta=cfg.use_tta); win,rec=v3_apply_patient_operating_point(tl,T,tau,model=model_name,ablation=ablation,fold=fold)
    metrics=v3_pooled_record_metrics(rec,prefix='record_')
    params=pd.DataFrame([{'model':model_name,'ablation':ablation,'fold':fold,'seed':int(seed_value),'supervised_lr':float(supervised_lr),'initialization':init_status,'best_epoch':int(best_epoch),'training_time_sec':float(training_time),'n_inner_train_patients':int(inner_train.record.nunique()),'n_inner_val_patients':int(inner_val.record.nunique()),'n_outer_test_patients':int(outer_test.record.nunique()),**info}])
    result=pd.DataFrame([{'model':model_name,'ablation':ablation,'fold':fold,'best_epoch':best_epoch,'training_time_sec':training_time,**metrics}])
    rec.to_csv(rp,index=False); win.to_csv(wp,index=False); params.to_csv(pp,index=False); result.to_csv(rr,index=False)
    del model; gc.collect();
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return {'record_pred':rec,'window_pred':win,'params':params,'result':result}


## Fold-contained self-supervised pretraining with safe empty-objective handling


In [23]:

def v3_nested_ssl_pretrain(fold:int,inner_train:pd.DataFrame,inner_val:pd.DataFrame,outer_test:pd.DataFrame,*,seed_value:int,tag:str,force:bool=V3_FORCE_RERUN)->Path:
    out_dir=V3_ROOT/'ssl_pretraining'/v3_safe_name(tag); out_dir.mkdir(parents=True,exist_ok=True)
    ck=out_dir/f'nested_ssl_fold{fold}.pt'; hp=out_dir/f'nested_ssl_history_fold{fold}.csv'
    if ck.exists() and hp.exists() and not force:return ck
    forbidden=set(inner_val.record)|set(outer_test.record); ssl_df=df_all[~df_all.record.isin(forbidden)].copy().reset_index(drop=True)
    binary_records=set(df_binary.record.unique()); target_ssl=set(ssl_df.record)&binary_records; allowed=set(inner_train.record.unique())
    if not target_ssl.issubset(allowed):raise RuntimeError(f'Nested SSL leakage in partition {fold}: {sorted(target_ssl-allowed)}')
    build_token_cache(ssl_df,overwrite=False)
    seed_everything(seed_value); model=MechXWindowEncoder(FEATURE_DIM,MECH_DIM,cfg).to(cfg.device)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.ssl_lr,weight_decay=cfg.weight_decay); scaler=torch.amp.GradScaler('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda')))
    loader=make_window_loader(ssl_df,cfg.ssl_batch_size,train=True,balanced=False,input_ablation=PROPOSED_ABLATION); hist=[]; skipped_total=0
    for epoch in range(1,cfg.ssl_epochs+1):
        model.train(); losses=[]; skipped=0; nbatch=0
        for batch in loader:
            nbatch+=1; batch=move_batch(batch,cfg.device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda'))):loss,parts=ssl_step_loss(model,batch,cfg)
            if not isinstance(loss,torch.Tensor) or not loss.requires_grad or loss.grad_fn is None:
                skipped+=1; skipped_total+=1; continue
            if not bool(torch.isfinite(loss).all().item()):raise RuntimeError(f'Non-finite SSL loss: partition={fold}, epoch={epoch}, batch={nbatch}')
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        mean=float(np.mean(losses)) if losses else np.nan
        hist.append({'epoch':epoch,'ssl_loss':mean,'n_ssl_windows':len(ssl_df),'n_ssl_records':ssl_df.record.nunique(),'n_batches':nbatch,'n_optimized_batches':len(losses),'n_skipped_no_gradient_batches':skipped})
        print(f'[nested SSL {tag} partition {fold}] epoch={epoch:03d} loss={mean:.4f} skipped={skipped}')
    if sum(x['n_optimized_batches'] for x in hist)==0:raise RuntimeError(f'No differentiable SSL updates in partition {fold}.')
    pd.DataFrame(hist).to_csv(hp,index=False); torch.save({'state_dict':model.state_dict(),'fold':fold,'seed':seed_value,'forbidden_validation_records':sorted(inner_val.record.unique()),'forbidden_test_records':sorted(outer_test.record.unique()),'n_skipped_no_gradient_batches':int(skipped_total)},ck)
    del model;gc.collect();
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return ck


def v3_run_matched_seed(seed_base:int,*,include_random_arm:bool=True,tag_prefix:str='matched_initialization',force:bool=V3_FORCE_RERUN):
    frames=[]; params=[]
    for fold in sorted(map(int,df_binary.fold.unique())):
        outer_train=df_binary[df_binary.fold!=fold].reset_index(drop=True); outer_test=df_binary[df_binary.fold==fold].reset_index(drop=True)
        inner_train,inner_val=make_inner_split(outer_train,seed=cfg.seed+777+fold,n_inner_splits=cfg.n_inner_splits); init_seed=int(seed_base)+fold*1000+77
        ssl_ck=v3_nested_ssl_pretrain(fold,inner_train,inner_val,outer_test,seed_value=init_seed,tag=f'{tag_prefix}_{seed_base}',force=force)
        arms=[('ssl_initialized',ssl_ck)]
        if include_random_arm:arms.append(('random_initialized',None))
        for arm,init in arms:
            o=v3_train_patient_endpoint_fold(df_binary,fold,model_name=arm,ablation=PROPOSED_ABLATION,init_ckpt=init,supervised_lr=V3_MATCHED_SUPERVISED_LR,experiment_tag=f'{tag_prefix}/seed_{seed_base}',seed_value=init_seed,force=force)
            x=o['record_pred'].copy();x['matched_arm']=arm;x['seed_base']=int(seed_base);frames.append(x);params.append(o['params'].assign(matched_arm=arm,seed_base=int(seed_base)))
    pred=pd.concat(frames,ignore_index=True); return pred,pd.concat(params,ignore_index=True)


def v3_summary_by_group(pred:pd.DataFrame,group_col:str)->pd.DataFrame:
    rows=[]
    for name,g in pred.groupby(group_col):
        m=v3_pooled_record_metrics(g); lo,hi=v3_record_bootstrap_ci(g,'macro_f1',cfg.n_bootstrap,seed=cfg.seed+sum(map(ord,str(name))))
        rows.append({group_col:name,**m,'macro_f1_ci_low':lo,'macro_f1_ci_high':hi})
    return pd.DataFrame(rows)


## Analysis 1 — Standalone multimodal reference model with patient-level operating-point selection


In [24]:

def run_v3_reference_model(force:bool=V3_FORCE_RERUN):
    pred,params=v3_run_matched_seed(V3_REFERENCE_SEED_BASE,include_random_arm=False,tag_prefix='matched_initialization',force=force)
    pred=pred[pred.matched_arm=='ssl_initialized'].copy(); pred['analysis']='reference_multimodal_patient_operating_point'
    pred.to_csv(V3_TABLE_DIR/'Table_V3A_reference_patient_predictions.csv',index=False)
    metrics=v3_pooled_record_metrics(pred); lo,hi=v3_record_bootstrap_ci(pred,'macro_f1',cfg.n_bootstrap,cfg.seed)
    summary=pd.DataFrame([{**metrics,'macro_f1_ci_low':lo,'macro_f1_ci_high':hi,'seed_base':V3_REFERENCE_SEED_BASE}])
    summary.to_csv(V3_TABLE_DIR/'Table_V3A_reference_patient_level_operating_point_summary.csv',index=False)
    params.to_csv(V3_TABLE_DIR/'Table_V3A_reference_fold_parameters.csv',index=False)
    return {'record_pred':pred,'summary':summary,'params':params}

V3_REFERENCE_OUTPUTS=run_v3_reference_model() if RUN_V3_REFERENCE_MODEL else {}
if V3_REFERENCE_OUTPUTS:display(V3_REFERENCE_OUTPUTS['summary'])


Building token cache:   0%|          | 0/9522 [00:00<?, ?it/s]

[nested SSL matched_initialization_88000 partition 0] epoch=001 loss=115.5145 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=002 loss=114.4037 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=003 loss=117.0007 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=004 loss=114.0898 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=005 loss=114.8389 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=006 loss=112.3504 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=007 loss=112.3392 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=008 loss=115.2157 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=009 loss=115.9689 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=010 loss=111.5847 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=011 loss=112.7765 skipped=0
[nested SSL matched_initializati

FileNotFoundError: [Errno 2] No such file or directory: 'outputs_mechxformer_av_v3_final_validation\\matched_initialization\\seed_88000\\tables\\record_predictions_ssl_initialized_window_level_fold0.csv'

In [25]:
# ============================================================
# V3 RECOVERY PATCH
# - fixes nested output-directory FileNotFoundError
# - keeps the current live kernel
# - reuses an already-trained fold checkpoint when available
# - resumes Analysis 1 with force=False
# - patch remains active for later V3 analyses
# ============================================================

from pathlib import Path
import gc
import time
import math
import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("V3 RECOVERY PATCH: fixing output paths and resuming safely")
print("=" * 80)

# ------------------------------------------------------------
# 1. Make the V3 output root ABSOLUTE.
#    This prevents relative-path changes from breaking nested saves.
# ------------------------------------------------------------
V3_ROOT = Path(V3_ROOT).resolve()

V3_MODEL_DIR = V3_ROOT / "models"
V3_TABLE_DIR = V3_ROOT / "tables"
V3_FIG_DIR   = V3_ROOT / "figures"
V3_LOG_DIR   = V3_ROOT / "logs"
V3_CACHE_DIR = V3_ROOT / "cache"

for _p in [
    V3_ROOT,
    V3_MODEL_DIR,
    V3_TABLE_DIR,
    V3_FIG_DIR,
    V3_LOG_DIR,
    V3_CACHE_DIR,
]:
    _p.mkdir(parents=True, exist_ok=True)

# Pre-create the nested matched-initialization folders used later.
for _seed_base in V3_MATCHED_SEED_BASES:
    for _sub in ["models", "tables", "logs", "ssl"]:
        (V3_ROOT / "matched_initialization" /
         f"seed_{int(_seed_base)}" / _sub).mkdir(
            parents=True,
            exist_ok=True
        )

V3_FORCE_RERUN = False

print("Stable V3 output root:")
print(V3_ROOT)


# ------------------------------------------------------------
# 2. Replace the fold trainer with a recovery-safe version.
#
#    Important improvement:
#    If a fold checkpoint already exists but result CSVs were not
#    written because of the crash, this function loads the checkpoint,
#    performs validation/test inference, writes the missing outputs,
#    and DOES NOT retrain that completed fold.
# ------------------------------------------------------------
def v3_train_patient_endpoint_fold(
    df_task: pd.DataFrame,
    fold: int,
    *,
    model_name: str,
    ablation: str,
    init_ckpt,
    supervised_lr: float,
    experiment_tag: str,
    seed_value: int,
    force: bool = False,
):
    # Stable absolute experiment paths
    out_dir   = (V3_ROOT / Path(experiment_tag)).resolve()
    model_dir = out_dir / "models"
    table_dir = out_dir / "tables"
    log_dir   = out_dir / "logs"

    for _p in [out_dir, model_dir, table_dir, log_dir]:
        _p.mkdir(parents=True, exist_ok=True)

    sm = v3_safe_name(model_name)
    sa = v3_safe_name(ablation)

    rp = table_dir / f"record_predictions_{sm}_{sa}_fold{fold}.csv"
    wp = table_dir / f"window_predictions_{sm}_{sa}_fold{fold}.csv"
    pp = table_dir / f"fold_parameters_{sm}_{sa}_fold{fold}.csv"
    rr = table_dir / f"result_{sm}_{sa}_fold{fold}.csv"

    ckpt = model_dir / f"ckpt_{sm}_{sa}_fold{fold}.pt"
    hist_path = log_dir / f"history_{sm}_{sa}_fold{fold}.csv"

    # --------------------------------------------------------
    # A. Fully completed fold -> load it immediately
    # --------------------------------------------------------
    if (
        not force
        and all(_p.exists() for _p in [rp, wp, pp, rr])
    ):
        print(
            f"[fold {fold}] Reusing completed outputs: "
            f"{model_name}/{ablation}"
        )
        return {
            "record_pred": pd.read_csv(rp),
            "window_pred": pd.read_csv(wp),
            "params": pd.read_csv(pp),
            "result": pd.read_csv(rr),
        }

    # --------------------------------------------------------
    # B. Reconstruct the same split/loaders
    # --------------------------------------------------------
    seed_everything(int(seed_value))

    outer_train = (
        df_task[df_task.fold != fold]
        .reset_index(drop=True)
    )
    outer_test = (
        df_task[df_task.fold == fold]
        .reset_index(drop=True)
    )

    inner_train, inner_val = make_inner_split(
        outer_train,
        seed=cfg.seed + 777 + fold,
        n_inner_splits=cfg.n_inner_splits,
    )

    tr = make_window_loader(
        inner_train,
        cfg.batch_size,
        train=True,
        balanced=True,
        input_ablation=ablation,
    )

    va = make_window_loader(
        inner_val,
        cfg.batch_size,
        train=False,
        input_ablation=ablation,
    )

    te = make_window_loader(
        outer_test,
        cfg.batch_size,
        train=False,
        input_ablation=ablation,
    )

    seed_everything(int(seed_value))

    model = MechXWindowEncoder(
        FEATURE_DIM,
        MECH_DIM,
        cfg,
    ).to(cfg.device)

    init_status = (
        str(init_ckpt)
        if init_ckpt is not None
        else "random_initialization"
    )

    best_epoch = -1
    training_time = np.nan
    recovered_from_checkpoint = False

    # --------------------------------------------------------
    # C. CRASH RECOVERY:
    #    checkpoint exists, outputs do not.
    #    Use the trained checkpoint instead of retraining.
    # --------------------------------------------------------
    if ckpt.exists() and not force:
        print(
            f"[fold {fold}] Found trained checkpoint but incomplete "
            f"CSV outputs -> recovering without retraining."
        )
        print(" checkpoint:", ckpt)

        saved = torch.load(
            ckpt,
            map_location=cfg.device,
        )

        state = (
            saved.get("state_dict", saved)
            if isinstance(saved, dict)
            else saved
        )

        model.load_state_dict(
            state,
            strict=False,
        )

        if isinstance(saved, dict):
            best_epoch = int(
                saved.get("epoch", -1)
            )

        recovered_from_checkpoint = True

    # --------------------------------------------------------
    # D. No checkpoint -> train normally
    # --------------------------------------------------------
    else:
        if init_ckpt is not None:
            init_ckpt = Path(init_ckpt)

            if not init_ckpt.exists():
                raise FileNotFoundError(
                    f"Initialization checkpoint not found: {init_ckpt}"
                )

            init_obj = torch.load(
                init_ckpt,
                map_location=cfg.device,
            )

            init_state = (
                init_obj.get("state_dict", init_obj)
                if isinstance(init_obj, dict)
                else init_obj
            )

            model.load_state_dict(
                init_state,
                strict=False,
            )

        cw = class_weights_from_df(
            inner_train,
            cfg.device,
        )

        opt = torch.optim.AdamW(
            model.parameters(),
            lr=float(supervised_lr),
            weight_decay=cfg.weight_decay,
        )

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=(
                cfg.use_amp
                and str(cfg.device).startswith("cuda")
            ),
        )

        history = []
        best = -1.0
        wait = 0
        start_time = time.time()

        for epoch in range(
            1,
            cfg.epochs_window + 1,
        ):
            model.train()
            losses = []

            for batch in tr:
                batch = move_batch(
                    batch,
                    cfg.device,
                )

                opt.zero_grad(
                    set_to_none=True
                )

                with torch.amp.autocast(
                    "cuda",
                    enabled=(
                        cfg.use_amp
                        and str(cfg.device).startswith("cuda")
                    ),
                ):
                    logits, aux = model(
                        batch,
                        return_recon=False,
                        return_attention=False,
                    )

                    sup = supervised_loss_per_sample(
                        logits,
                        batch["label"],
                        cw,
                        cfg,
                    ).mean()

                    loss = (
                        sup
                        + 0.005 * aux.get(
                            "einthoven",
                            logits.new_tensor(0.0),
                        )
                        + 0.005 * aux.get(
                            "cs_order",
                            logits.new_tensor(0.0),
                        )
                    )

                scaler.scale(loss).backward()

                scaler.unscale_(opt)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg.grad_clip,
                )

                scaler.step(opt)
                scaler.update()

                losses.append(
                    float(
                        loss.detach().cpu()
                    )
                )

            # Patient-level validation endpoint
            vl = predict_window_loader(
                model,
                va,
                cfg.device,
                temperature=1.0,
                threshold=0.5,
                use_tta=V3_EARLYSTOP_USE_TTA,
            )

            vr = v3_patient_probs_from_window_logits(
                vl,
                1.0,
            )

            if len(vr):
                vf = float(
                    f1_score(
                        vr.label.to_numpy(int),
                        (
                            vr.prob_AVRT
                            .to_numpy(float)
                            >= 0.5
                        ).astype(int),
                        average="macro",
                        zero_division=0,
                    )
                )
            else:
                vf = -1.0

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": float(
                        np.mean(losses)
                    ),
                    "val_patient_macro_f1_at_0.5": vf,
                }
            )

            print(
                f"[{experiment_tag}/"
                f"{model_name}/"
                f"{ablation}/fold{fold}] "
                f"epoch={epoch:03d} "
                f"loss={np.mean(losses):.4f} "
                f"valPatientF1={vf:.4f}"
            )

            if vf > best + 1e-5:
                best = vf
                best_epoch = epoch
                wait = 0

                ckpt.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                torch.save(
                    {
                        "state_dict":
                            model.state_dict(),
                        "fold": fold,
                        "epoch": epoch,
                        "seed": int(seed_value),
                        "lr": float(
                            supervised_lr
                        ),
                        "ablation": ablation,
                    },
                    ckpt,
                )

            else:
                wait += 1

                if (
                    wait
                    >= V3_EARLYSTOP_PATIENCE
                ):
                    break

        training_time = (
            time.time() - start_time
        )

        hist_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        pd.DataFrame(
            history
        ).to_csv(
            hist_path,
            index=False,
        )

        # Reload best model
        saved = torch.load(
            ckpt,
            map_location=cfg.device,
        )

        model.load_state_dict(
            saved["state_dict"]
        )

    # --------------------------------------------------------
    # E. Validation inference -> PATIENT-LEVEL operating point
    # --------------------------------------------------------
    model.eval()

    vl = predict_window_loader(
        model,
        va,
        cfg.device,
        temperature=1.0,
        threshold=0.5,
        use_tta=cfg.use_tta,
    )

    T, tau, val_record_pred, info = (
        v3_fit_patient_operating_point(vl)
    )

    # --------------------------------------------------------
    # F. Outer-test inference
    # --------------------------------------------------------
    tl = predict_window_loader(
        model,
        te,
        cfg.device,
        temperature=1.0,
        threshold=0.5,
        use_tta=cfg.use_tta,
    )

    win, rec = (
        v3_apply_patient_operating_point(
            tl,
            T,
            tau,
            model=model_name,
            ablation=ablation,
            fold=fold,
        )
    )

    metrics = v3_pooled_record_metrics(
        rec,
        prefix="record_",
    )

    params = pd.DataFrame(
        [
            {
                "model": model_name,
                "ablation": ablation,
                "fold": int(fold),
                "seed": int(seed_value),
                "supervised_lr":
                    float(supervised_lr),
                "initialization":
                    init_status,
                "best_epoch":
                    int(best_epoch),
                "training_time_sec":
                    float(training_time)
                    if np.isfinite(training_time)
                    else np.nan,
                "recovered_from_checkpoint":
                    bool(
                        recovered_from_checkpoint
                    ),
                "n_inner_train_patients":
                    int(
                        inner_train
                        .record
                        .nunique()
                    ),
                "n_inner_val_patients":
                    int(
                        inner_val
                        .record
                        .nunique()
                    ),
                "n_outer_test_patients":
                    int(
                        outer_test
                        .record
                        .nunique()
                    ),
                **info,
            }
        ]
    )

    result = pd.DataFrame(
        [
            {
                "model": model_name,
                "ablation": ablation,
                "fold": int(fold),
                "best_epoch":
                    int(best_epoch),
                "training_time_sec":
                    float(training_time)
                    if np.isfinite(training_time)
                    else np.nan,
                "recovered_from_checkpoint":
                    bool(
                        recovered_from_checkpoint
                    ),
                **metrics,
            }
        ]
    )

    # --------------------------------------------------------
    # G. Critical fix:
    #    create parent directories IMMEDIATELY before every save
    # --------------------------------------------------------
    for _path in [
        rp,
        wp,
        pp,
        rr,
    ]:
        _path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    rec.to_csv(
        rp,
        index=False,
    )

    win.to_csv(
        wp,
        index=False,
    )

    params.to_csv(
        pp,
        index=False,
    )

    result.to_csv(
        rr,
        index=False,
    )

    print(
        f"[fold {fold}] Saved recovered/completed outputs:"
    )
    print(" ", rp)

    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "record_pred": rec,
        "window_pred": win,
        "params": params,
        "result": result,
    }


# ------------------------------------------------------------
# 3. Memory cleanup after failed call
# ------------------------------------------------------------
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 4. Show recovery state before resuming
# ------------------------------------------------------------
print("\nReference seed recovery state:")

_ref_dir = (
    V3_ROOT
    / "matched_initialization"
    / f"seed_{V3_REFERENCE_SEED_BASE}"
)

for _fold in sorted(
    map(
        int,
        df_binary.fold.unique(),
    )
):
    _sm = v3_safe_name(
        "ssl_initialized"
    )

    _sa = v3_safe_name(
        PROPOSED_ABLATION
    )

    _ck = (
        _ref_dir
        / "models"
        / f"ckpt_{_sm}_{_sa}_fold{_fold}.pt"
    )

    _rp = (
        _ref_dir
        / "tables"
        / f"record_predictions_{_sm}_{_sa}_fold{_fold}.csv"
    )

    print(
        f" fold {_fold}: "
        f"checkpoint="
        f"{'YES' if _ck.exists() else 'NO'}, "
        f"result="
        f"{'DONE' if _rp.exists() else 'PENDING'}"
    )


# ------------------------------------------------------------
# 5. RESUME Analysis 1
#    force=False means:
#      - completed result folds are reused
#      - checkpoint-only fold(s) are recovered without retraining
#      - genuinely incomplete fold(s) train normally
# ------------------------------------------------------------
print(
    "\nResuming V3 Analysis 1...\n"
)

V3_REFERENCE_OUTPUTS = (
    run_v3_reference_model(
        force=False
    )
)

print(
    "\n" + "=" * 80
)
print(
    "V3 ANALYSIS 1 COMPLETE"
)
print(
    "=" * 80
)

display(
    V3_REFERENCE_OUTPUTS[
        "summary"
    ]
)

display(
    V3_REFERENCE_OUTPUTS[
        "params"
    ]
)

print(
    "\nYou can now continue with the next notebook cell "
    "(Analysis 2) without restarting the kernel."
)

V3 RECOVERY PATCH: fixing output paths and resuming safely
Stable V3 output root:
C:\Users\hasee\Desktop\Haseeb Phd Work\New Updated Phd Papers\Paper 2 Nadeem Sarwar Drafts\Paper 2 Revised with code n Results\outputs_mechxformer_av_v3_final_validation

Reference seed recovery state:
 fold 0: checkpoint=YES, result=PENDING
 fold 1: checkpoint=NO, result=PENDING
 fold 2: checkpoint=NO, result=PENDING
 fold 3: checkpoint=NO, result=PENDING
 fold 4: checkpoint=NO, result=PENDING

Resuming V3 Analysis 1...

[fold 0] Found trained checkpoint but incomplete CSV outputs -> recovering without retraining.
 checkpoint: C:\Users\hasee\Desktop\Haseeb Phd Work\New Updated Phd Papers\Paper 2 Nadeem Sarwar Drafts\Paper 2 Revised with code n Results\outputs_mechxformer_av_v3_final_validation\matched_initialization\seed_88000\models\ckpt_ssl_initialized_window_level_fold0.pt


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\hasee\\Desktop\\Haseeb Phd Work\\New Updated Phd Papers\\Paper 2 Nadeem Sarwar Drafts\\Paper 2 Revised with code n Results\\outputs_mechxformer_av_v3_final_validation\\matched_initialization\\seed_88000\\tables\\record_predictions_ssl_initialized_window_level_fold0.csv'

In [26]:
# ============================================================
# V3 RECOVERY 2
# FIX WINDOWS LONG-PATH ERROR AFTER PROJECT FOLDER WAS MOVED
#
# DO NOT restart kernel.
# DO NOT rerun earlier training cells.
#
# This cell:
#   1) preserves existing checkpoints/results
#   2) moves V3 OUTPUTS to a short Windows path
#   3) keeps current kernel objects/functions
#   4) recovers fold 0 from its checkpoint
#   5) continues folds 1-4 normally
# ============================================================

from pathlib import Path
import shutil
import gc
import os
import torch

print("=" * 90)
print("V3 LONG-PATH RECOVERY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Remember current long V3 path
# ------------------------------------------------------------
OLD_V3_ROOT = Path(V3_ROOT).resolve()

print("\nCurrent/old V3 root:")
print(OLD_V3_ROOT)
print("Path length:", len(str(OLD_V3_ROOT)))


# ------------------------------------------------------------
# 2. Use a SHORT output path
#
# Example:
# C:\Users\hasee\V3_AV_OUT
#
# Much safer than keeping outputs inside the deeply nested
# manuscript/project directory.
# ------------------------------------------------------------
NEW_V3_ROOT = (Path.home() / "V3_AV_OUT").resolve()

NEW_V3_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nNew SHORT V3 root:")
print(NEW_V3_ROOT)
print("Path length:", len(str(NEW_V3_ROOT)))


# ------------------------------------------------------------
# 3. Copy existing useful V3 outputs/checkpoints.
#
# We intentionally do NOT copy the potentially large V3 cache.
# The important trained checkpoints and tables are preserved.
# ------------------------------------------------------------
if OLD_V3_ROOT != NEW_V3_ROOT:

    if OLD_V3_ROOT.exists():

        preserve_dirs = [
            "matched_initialization",
            "models",
            "tables",
            "logs",
            "figures",
        ]

        for name in preserve_dirs:

            src = OLD_V3_ROOT / name
            dst = NEW_V3_ROOT / name

            if src.exists():

                print(f"\nCopying existing {name} ...")

                shutil.copytree(
                    src,
                    dst,
                    dirs_exist_ok=True,
                )

        # Preserve any top-level metadata files as well
        for src in OLD_V3_ROOT.iterdir():

            if src.is_file():

                dst = NEW_V3_ROOT / src.name

                try:
                    shutil.copy2(
                        src,
                        dst,
                    )
                except Exception as e:
                    print(
                        f"Warning: could not copy "
                        f"{src.name}: {e}"
                    )

    else:

        print(
            "\nWARNING: old V3 output directory "
            "is no longer present."
        )


# ------------------------------------------------------------
# 4. Re-point V3 GLOBAL PATHS.
#
# Existing functions access these globals at execution time,
# so the kernel does NOT need to restart.
# ------------------------------------------------------------
V3_ROOT = NEW_V3_ROOT

V3_MODEL_DIR = V3_ROOT / "models"
V3_TABLE_DIR = V3_ROOT / "tables"
V3_FIG_DIR   = V3_ROOT / "figures"
V3_LOG_DIR   = V3_ROOT / "logs"
V3_CACHE_DIR = V3_ROOT / "cache"

for p in [
    V3_ROOT,
    V3_MODEL_DIR,
    V3_TABLE_DIR,
    V3_FIG_DIR,
    V3_LOG_DIR,
    V3_CACHE_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 5. Pre-create all matched experiment directories
# ------------------------------------------------------------
for seed_base in V3_MATCHED_SEED_BASES:

    seed_root = (
        V3_ROOT
        / "matched_initialization"
        / f"seed_{int(seed_base)}"
    )

    for sub in [
        "models",
        "tables",
        "logs",
        "ssl",
    ]:

        (seed_root / sub).mkdir(
            parents=True,
            exist_ok=True,
        )


# Never force completed folds to rerun
V3_FORCE_RERUN = False


# ------------------------------------------------------------
# 6. Verify Windows can REALLY write to the new location
# ------------------------------------------------------------
test_dir = (
    V3_ROOT
    / "matched_initialization"
    / f"seed_{V3_REFERENCE_SEED_BASE}"
    / "tables"
)

test_dir.mkdir(
    parents=True,
    exist_ok=True,
)

test_file = (
    test_dir
    / "_write_test.csv"
)

pd.DataFrame(
    {
        "status": ["ok"]
    }
).to_csv(
    test_file,
    index=False,
)

assert test_file.exists(), (
    "Write test failed in new short directory."
)

test_file.unlink()

print(
    "\n[OK] Pandas write test succeeded "
    "in the new short directory."
)


# ------------------------------------------------------------
# 7. Check preserved fold states
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("REFERENCE-SEED RECOVERY STATUS")
print("=" * 90)

_sm = v3_safe_name(
    "ssl_initialized"
)

_sa = v3_safe_name(
    PROPOSED_ABLATION
)

for fold in sorted(
    map(
        int,
        df_binary.fold.unique(),
    )
):

    base = (
        V3_ROOT
        / "matched_initialization"
        / f"seed_{V3_REFERENCE_SEED_BASE}"
    )

    ck = (
        base
        / "models"
        / f"ckpt_{_sm}_{_sa}_fold{fold}.pt"
    )

    rp = (
        base
        / "tables"
        / f"record_predictions_{_sm}_{_sa}_fold{fold}.csv"
    )

    print(
        f"fold {fold}: "
        f"checkpoint={'YES' if ck.exists() else 'NO'} | "
        f"result={'DONE' if rp.exists() else 'PENDING'} | "
        f"path_len={len(str(rp))}"
    )


# ------------------------------------------------------------
# 8. IMPORTANT sanity check:
#    fold 0 checkpoint MUST have survived the copy.
# ------------------------------------------------------------
fold0_ckpt = (
    V3_ROOT
    / "matched_initialization"
    / f"seed_{V3_REFERENCE_SEED_BASE}"
    / "models"
    / f"ckpt_{_sm}_{_sa}_fold0.pt"
)

if not fold0_ckpt.exists():

    raise FileNotFoundError(
        "\nFold-0 checkpoint was not found in the "
        "new short output directory.\n"
        f"Expected:\n{fold0_ckpt}\n\n"
        "STOP HERE — do not retrain yet."
    )

print(
    "\n[OK] Fold-0 trained checkpoint preserved:"
)
print(fold0_ckpt)


# ------------------------------------------------------------
# 9. Clear only unused GPU memory.
#
# This does NOT reset the kernel.
# ------------------------------------------------------------
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 10. Resume Analysis 1.
#
# The patched v3_train_patient_endpoint_fold from the previous
# recovery cell is still active.
#
# fold 0:
#   checkpoint exists -> inference/save only
#
# folds 1-4:
#   no checkpoint -> train normally
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("RESUMING V3 ANALYSIS 1")
print("=" * 90)

V3_REFERENCE_OUTPUTS = run_v3_reference_model(
    force=False
)


# ------------------------------------------------------------
# 11. Display completed result
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("V3 ANALYSIS 1 COMPLETE")
print("=" * 90)

display(
    V3_REFERENCE_OUTPUTS[
        "summary"
    ]
)

display(
    V3_REFERENCE_OUTPUTS[
        "params"
    ]
)

print("\nFinal V3 output root:")
print(V3_ROOT)

print(
    "\nSUCCESS: continue with Analysis 2 onward. "
    "Do NOT rerun old V3 Analysis 1."
)

V3 LONG-PATH RECOVERY

Current/old V3 root:
C:\Users\hasee\Desktop\Haseeb Phd Work\New Updated Phd Papers\Paper 2 Nadeem Sarwar Drafts\Paper 2 Revised with code n Results\outputs_mechxformer_av_v3_final_validation
Path length: 169

New SHORT V3 root:
C:\Users\hasee\V3_AV_OUT
Path length: 24

Copying existing matched_initialization ...

Copying existing models ...

Copying existing tables ...

Copying existing logs ...

Copying existing figures ...

[OK] Pandas write test succeeded in the new short directory.

REFERENCE-SEED RECOVERY STATUS
fold 0: checkpoint=YES | result=PENDING | path_len=123
fold 1: checkpoint=NO | result=PENDING | path_len=123
fold 2: checkpoint=NO | result=PENDING | path_len=123
fold 3: checkpoint=NO | result=PENDING | path_len=123
fold 4: checkpoint=NO | result=PENDING | path_len=123

[OK] Fold-0 trained checkpoint preserved:
C:\Users\hasee\V3_AV_OUT\matched_initialization\seed_88000\models\ckpt_ssl_initialized_window_level_fold0.pt

RESUMING V3 ANALYSIS 1


Building token cache:   0%|          | 0/9522 [00:00<?, ?it/s]

[nested SSL matched_initialization_88000 partition 0] epoch=001 loss=115.5145 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=002 loss=114.4037 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=003 loss=117.0007 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=004 loss=114.0898 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=005 loss=114.8389 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=006 loss=112.3504 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=007 loss=112.3392 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=008 loss=115.2157 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=009 loss=115.9689 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=010 loss=111.5847 skipped=0
[nested SSL matched_initialization_88000 partition 0] epoch=011 loss=112.7765 skipped=0
[nested SSL matched_initializati

Building token cache:   0%|          | 0/8898 [00:00<?, ?it/s]

[nested SSL matched_initialization_88000 partition 1] epoch=001 loss=152.2611 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=002 loss=150.9840 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=003 loss=144.8952 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=004 loss=146.2888 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=005 loss=148.3691 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=006 loss=146.5333 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=007 loss=146.2026 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=008 loss=149.8488 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=009 loss=143.8610 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=010 loss=145.3619 skipped=0
[nested SSL matched_initialization_88000 partition 1] epoch=011 loss=140.8833 skipped=0
[nested SSL matched_initializati

...

Building token cache:   0%|          | 0/7257 [00:00<?, ?it/s]

[nested SSL matched_initialization_88000 partition 4] epoch=001 loss=129.0670 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=002 loss=126.6126 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=003 loss=125.8398 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=004 loss=128.1817 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=005 loss=125.3520 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=006 loss=123.2222 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=007 loss=121.1043 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=008 loss=117.1610 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=009 loss=120.7895 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=010 loss=121.3920 skipped=0
[nested SSL matched_initialization_88000 partition 4] epoch=011 loss=120.4284 skipped=0
[nested SSL matched_initializati

,threshold,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,cohen_kappa,ece,...,f1_AVNRT,support_AVNRT,precision_AVRT,recall_AVRT,f1_AVRT,support_AVRT,n_patients,macro_f1_ci_low,macro_f1_ci_high,seed_base
0,0.5,0.642857,0.625641,0.666667,0.625641,0.611111,0.619048,0.289414,0.259259,0.22553,...,0.5,13.0,0.619048,0.866667,0.722222,15.0,28,0.40256,0.775547,88000


,model,ablation,fold,seed,supervised_lr,initialization,best_epoch,training_time_sec,recovered_from_checkpoint,n_inner_train_patients,...,threshold_status,best_macro_f1,n_tied_maximizers,lowest_tied_maximizer,highest_tied_maximizer,temperature,threshold,val_patient_macro_f1_at_threshold,matched_arm,seed_base
0,ssl_initialized,window_level,0,88077,0.0001,C:\Users\hasee\V3_AV_OUT\ssl_pretraining\match...,2,NaN,True,16,...,optimized_patient_macro_f1,1.0,11,0.12,0.170,1.723611,0.12,1.0,ssl_initialized,88000
1,ssl_initialized,window_level,1,89077,0.0001,C:\Users\hasee\V3_AV_OUT\ssl_pretraining\match...,14,358.281932,False,16,...,optimized_patient_macro_f1,1.0,34,0.31,0.475,0.050002,0.31,1.0,ssl_initialized,88000
2,ssl_initialized,window_level,2,90077,0.0001,C:\Users\hasee\V3_AV_OUT\ssl_pretraining\match...,9,348.043785,False,16,...,optimized_patient_macro_f1,1.0,103,0.05,0.560,0.060055,0.05,1.0,ssl_initialized,88000
3,ssl_initialized,window_level,3,91077,0.0001,C:\Users\hasee\V3_AV_OUT\ssl_pretraining\match...,7,227.429083,False,14,...,optimized_patient_macro_f1,1.0,35,0.09,0.260,1.191138,0.09,1.0,ssl_initialized,88000
4,ssl_initialized,window_level,4,92077,0.0001,C:\Users\hasee\V3_AV_OUT\ssl_pretraining\match...,18,366.548719,False,15,...,optimized_patient_macro_f1,1.0,18,0.59,0.675,0.050002,0.59,1.0,ssl_initialized,88000



Final V3 output root:
C:\Users\hasee\V3_AV_OUT

SUCCESS: continue with Analysis 2 onward. Do NOT rerun old V3 Analysis 1.


## Analysis 2 — Source clinical-diagnosis concordance audit


In [27]:

def v3_normalize_record_id(x:Any)->Optional[str]:
    s=str(x).strip(); m=re.search(r'[xX]?(\d+)',s)
    if not m:return None
    return f'x{int(m.group(1)):03d}'


def v3_map_diagnosis_to_binary(x:Any)->Optional[int]:
    s=str(x).upper().replace('–','-').replace('—','-')
    if 'AVNRT' in s or 'ATRIOVENTRICULAR NODAL' in s:return 0
    if 'AVRT' in s or 'ATRIOVENTRICULAR RE-ENTRANT' in s or 'ATRIOVENTRICULAR REENTRANT' in s:return 1
    return None


def v3_find_diagnosis_metadata(root:Path):
    candidates=[]
    for fp in sorted(root.rglob('*.csv')):
        try:
            d=pd.read_csv(fp,sep=None,engine='python')
        except Exception:
            continue
        norm={c:re.sub(r'[^a-z0-9]','',str(c).lower()) for c in d.columns}
        id_cols=[c for c,n in norm.items() if ('patient' in n or 'subject' in n or n in {'id','record','recordid','patientid','subjectid'}) and 'diagn' not in n]
        dx_cols=[c for c,n in norm.items() if 'diagn' in n or n in {'dx','rhythmdiagnosis','clinicaldiagnosis'}]
        if id_cols and dx_cols:
            candidates.append((fp,d,id_cols[0],dx_cols[0]))
    return candidates


def run_v3_clinical_reference_audit():
    candidates=v3_find_diagnosis_metadata(DATA_DIR)
    included=df_binary[['record','label']].drop_duplicates('record').copy(); included['record_norm']=included.record.map(v3_normalize_record_id)
    audit=None; source_file=None
    for fp,d,idc,dxc in candidates:
        x=d[[idc,dxc]].copy(); x['record_norm']=x[idc].map(v3_normalize_record_id); x['clinical_label']=x[dxc].map(v3_map_diagnosis_to_binary)
        m=included.merge(x[['record_norm',dxc,'clinical_label']].drop_duplicates('record_norm'),on='record_norm',how='left')
        if m.clinical_label.notna().sum()>(0 if audit is None else audit.clinical_label.notna().sum()):
            audit=m; source_file=fp
    if audit is None:
        audit=included.copy(); audit['source_diagnosis']=np.nan; audit['clinical_label']=np.nan; source_file=None
    else:
        # Normalize diagnosis column name for export.
        dxcol=[c for c in audit.columns if c not in {'record','label','record_norm','clinical_label'}][-1]
        audit=audit.rename(columns={dxcol:'source_diagnosis'})
    audit['annotation_reference']=audit.label.map({0:'AVNRT',1:'AVRT'})
    audit['clinical_reference']=audit.clinical_label.map({0:'AVNRT',1:'AVRT'})
    audit['agreement']=np.where(audit.clinical_label.notna(),audit.label.astype(float)==audit.clinical_label.astype(float),np.nan)
    audit.to_csv(V3_TABLE_DIR/'Table_V3B_clinical_reference_concordance.csv',index=False)
    n_available=int(audit.clinical_label.notna().sum()); n_agree=int((audit.agreement==True).sum()) if n_available else 0
    summary=pd.DataFrame([{'metadata_file':str(source_file) if source_file else '', 'n_included_patients':int(len(included)), 'n_with_mappable_clinical_diagnosis':n_available, 'n_agree':n_agree, 'agreement_fraction':float(n_agree/n_available) if n_available else np.nan, 'status':'completed' if n_available else 'clinical_diagnosis_field_not_found_or_not_mappable'}])
    summary.to_csv(V3_TABLE_DIR/'Table_V3B_clinical_reference_concordance_summary.csv',index=False)
    return {'audit':audit,'summary':summary}

V3_CLINICAL_AUDIT=run_v3_clinical_reference_audit() if RUN_V3_CLINICAL_REFERENCE_AUDIT else {}
if V3_CLINICAL_AUDIT:display(V3_CLINICAL_AUDIT['summary'])


,metadata_file,n_included_patients,n_with_mappable_clinical_diagnosis,n_agree,agreement_fraction,status
0,C:\Users\hasee\Desktop\leipzig-heart-centerwit...,28,28,28,1.0,completed


## Analysis 3 — Strict ECG-only and IEGM-only models with paired comparison to the matched multimodal reference


In [28]:

V3_STRICT_CACHE_DIR = V3_CACHE_DIR / 'strict_whole_window'
V3_STRICT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def v3_strict_cache_path(row_id: int) -> Path:
    return V3_STRICT_CACHE_DIR / f'whole_{int(row_id):07d}.npz'


def v3_build_strict_whole_window_cache(df: pd.DataFrame, overwrite: bool = False) -> pd.DataFrame:
    failures=[]
    for _,row in tqdm(df.iterrows(),total=len(df),desc='V3 strict whole-window cache'):
        fp=v3_strict_cache_path(int(row.row_id))
        if fp.exists() and not overwrite: continue
        try:
            ecg,iegm,ecg_mask,iegm_mask,orig_fs,_,_=load_window_streams(row.record,int(row.start),int(row.end))
            ecg=robust_zscore(ecg); iegm=robust_zscore(iegm)
            np.savez_compressed(fp,ecg=ecg.astype(np.float32),iegm=iegm.astype(np.float32),ecg_ch_mask=ecg_mask.astype(bool),iegm_ch_mask=iegm_mask.astype(bool))
        except Exception as e:
            failures.append({'row_id':int(row.row_id),'record':row.record,'error':repr(e)})
    f=pd.DataFrame(failures)
    if len(f): f.to_csv(V3_LOG_DIR/'strict_whole_window_cache_failures.csv',index=False)
    return f


class V3StrictWholeWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, modality: str, train: bool=False):
        self.df=df.reset_index(drop=True).copy(); self.modality=str(modality); self.train=bool(train)
        if self.modality not in {'ecg','iegm','dual'}: raise ValueError(self.modality)
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; z=np.load(v3_strict_cache_path(int(row.row_id)),allow_pickle=False)
        if self.modality=='ecg':
            x=z['ecg'].astype(np.float32); mask=z['ecg_ch_mask'].astype(bool)
        elif self.modality=='iegm':
            x=z['iegm'].astype(np.float32); mask=z['iegm_ch_mask'].astype(bool)
        else:
            x=np.concatenate([z['ecg'],z['iegm']],axis=0).astype(np.float32); mask=np.concatenate([z['ecg_ch_mask'],z['iegm_ch_mask']]).astype(bool)
        if self.train and cfg.use_augmentation:
            scale=np.random.normal(1.0,cfg.aug_scale_std,size=(x.shape[0],1)).astype(np.float32); x=x*scale
            x=x+np.random.normal(0.0,cfg.aug_noise_std,size=x.shape).astype(np.float32)
            shift=int(round(cfg.aug_time_shift_ms*cfg.target_fs/1000.0))
            if shift>0:
                sh=np.random.randint(-shift,shift+1); x=np.roll(x,sh,axis=1)
            drop=(np.random.rand(x.shape[0])<cfg.aug_lead_dropout_p)&mask
            x[drop]=0.0
        x=x*mask[:,None].astype(np.float32)
        return {'x':torch.tensor(x,dtype=torch.float32),'ch_mask':torch.tensor(mask,dtype=torch.bool),'label':torch.tensor(int(row.label),dtype=torch.long),'record':row.record,'row_id':torch.tensor(int(row.row_id),dtype=torch.long)}


def v3_patient_balanced_sampler(df: pd.DataFrame):
    d=df.reset_index(drop=True)
    per_record=d.groupby('record').size().to_dict()
    patient_labels=d.groupby('record')['label'].agg(lambda x:int(x.mode().iloc[0]))
    class_patients=patient_labels.value_counts().to_dict()
    weights=[]
    for _,r in d.iterrows():
        weights.append(1.0/(max(per_record[r.record],1)*max(class_patients[int(r.label)],1)))
    return WeightedRandomSampler(torch.tensor(weights,dtype=torch.double),num_samples=len(weights),replacement=True)


def v3_strict_loader(df: pd.DataFrame, modality: str, train: bool=False):
    ds=V3StrictWholeWindowDataset(df,modality,train=train)
    if train and V3_STRICT_PATIENT_BALANCED_SAMPLING:
        return DataLoader(ds,batch_size=V3_STRICT_BATCH_SIZE,sampler=v3_patient_balanced_sampler(df),num_workers=cfg.num_workers,pin_memory=torch.cuda.is_available())
    return DataLoader(ds,batch_size=V3_STRICT_BATCH_SIZE,shuffle=train,num_workers=cfg.num_workers,pin_memory=torch.cuda.is_available())


class V3Residual1D(nn.Module):
    def __init__(self,c:int):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(c,c,7,padding=3),nn.BatchNorm1d(c),nn.GELU(),nn.Dropout(0.1),nn.Conv1d(c,c,7,padding=3),nn.BatchNorm1d(c))
    def forward(self,x): return F.gelu(x+self.net(x))


class V3StrictWholeWindowNet(nn.Module):
    def __init__(self,n_channels:int):
        super().__init__(); d=cfg.d_model
        self.stem=nn.Sequential(nn.Conv1d(n_channels,d//2,15,stride=5,padding=7),nn.BatchNorm1d(d//2),nn.GELU(),nn.Conv1d(d//2,d,9,stride=2,padding=4),nn.BatchNorm1d(d),nn.GELU())
        self.blocks=nn.Sequential(V3Residual1D(d),nn.Conv1d(d,d,7,stride=2,padding=3),nn.GELU(),V3Residual1D(d))
        self.pool=nn.AdaptiveAvgPool1d(V3_STRICT_N_TOKENS)
        self.cls=nn.Parameter(torch.zeros(1,1,d)); nn.init.trunc_normal_(self.cls,std=0.02)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=cfg.n_heads,dim_feedforward=d*4,dropout=cfg.dropout,activation='gelu',batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=cfg.n_layers)
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Dropout(cfg.dropout),nn.Linear(d,2))
    def forward(self,x):
        z=self.pool(self.blocks(self.stem(x))).transpose(1,2)
        cls=self.cls.expand(z.shape[0],-1,-1); z=torch.cat([cls,z],dim=1); z=self.encoder(z)
        return self.head(z[:,0])


def v3_strict_tta(x: torch.Tensor, rep: int) -> torch.Tensor:
    if rep==0: return x
    y=x.clone(); y=y*(torch.randn((y.shape[0],y.shape[1],1),device=y.device,dtype=y.dtype)*cfg.aug_scale_std+1.0); y=y+torch.randn_like(y)*cfg.aug_noise_std
    shift=int(round(cfg.aug_time_shift_ms*cfg.target_fs/1000.0))
    if shift>0:
        sh=int(torch.randint(-shift,shift+1,(1,),device=y.device).item()); y=torch.roll(y,sh,dims=-1)
    return y


@torch.no_grad()
def v3_predict_strict(model,loader,use_tta=True):
    model.eval(); rows=[]; reps=cfg.n_tta if use_tta else 1
    for batch in loader:
        x=batch['x'].to(cfg.device); labels=batch['label'].numpy(); rids=batch['row_id'].numpy(); records=batch['record']
        acc=None
        for r in range(reps):
            lg=model(v3_strict_tta(x,r) if use_tta else x); acc=lg if acc is None else acc+lg
        lg=(acc/reps).detach().cpu().numpy()
        for i in range(len(labels)):
            rows.append({'record':records[i],'row_id':int(rids[i]),'label':int(labels[i]),'logit_AVNRT':float(lg[i,0]),'logit_AVRT':float(lg[i,1])})
    return pd.DataFrame(rows)


def v3_strict_patient_weights(train_df: pd.DataFrame) -> torch.Tensor:
    p=train_df[['record','label']].drop_duplicates('record'); counts=np.bincount(p.label.to_numpy(int),minlength=2).astype(float); w=counts.sum()/(2*np.maximum(counts,1)); w=w/w.mean(); return torch.tensor(w,dtype=torch.float32,device=cfg.device)


def v3_train_strict_fold(df_task:pd.DataFrame,fold:int,modality:str,force:bool=V3_FORCE_RERUN)->Dict[str,pd.DataFrame]:
    tag=f'strict_{modality}_whole_window'; td=V3_ROOT/tag/'tables'; md=V3_ROOT/tag/'models'; ld=V3_ROOT/tag/'logs'
    for p in [td,md,ld]:p.mkdir(parents=True,exist_ok=True)
    rp=td/f'record_predictions_fold{fold}.csv'; wp=td/f'window_predictions_fold{fold}.csv'; pp=td/f'fold_parameters_fold{fold}.csv'
    if rp.exists() and wp.exists() and pp.exists() and not force:return {'record_pred':pd.read_csv(rp),'window_pred':pd.read_csv(wp),'params':pd.read_csv(pp)}
    outer_train=df_task[df_task.fold!=fold].reset_index(drop=True); outer_test=df_task[df_task.fold==fold].reset_index(drop=True); inner_train,inner_val=make_inner_split(outer_train,seed=cfg.seed+777+fold,n_inner_splits=cfg.n_inner_splits)
    tr=v3_strict_loader(inner_train,modality,True); va=v3_strict_loader(inner_val,modality,False); te=v3_strict_loader(outer_test,modality,False)
    n_ch={'ecg':len(ECG_LEADS),'iegm':len(IEGM_CHANNELS),'dual':len(ECG_LEADS)+len(IEGM_CHANNELS)}[modality]
    seed_everything(120000+fold*100+sum(map(ord,modality))); model=V3StrictWholeWindowNet(n_ch).to(cfg.device); opt=torch.optim.AdamW(model.parameters(),lr=V3_STRICT_LR,weight_decay=cfg.weight_decay); scaler=torch.amp.GradScaler('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda'))); cw=v3_strict_patient_weights(inner_train)
    best=-1.; best_epoch=-1; wait=0; ck=md/f'ckpt_fold{fold}.pt'; hist=[]; t0=time.time()
    for epoch in range(1,V3_STRICT_EPOCHS+1):
        model.train(); ls=[]
        for b in tr:
            x=b['x'].to(cfg.device); y=b['label'].to(cfg.device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(cfg.use_amp and cfg.device.startswith('cuda'))):
                lg=model(x); loss=supervised_loss_per_sample(lg,y,cw,cfg).mean()
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); scaler.step(opt); scaler.update(); ls.append(float(loss.detach().cpu()))
        vl=v3_predict_strict(model,va,use_tta=V3_EARLYSTOP_USE_TTA); vr=v3_patient_probs_from_window_logits(vl,1.0); pred=(vr.prob_AVRT.to_numpy(float)>=0.5).astype(int); vf=float(f1_score(vr.label.to_numpy(int),pred,average='macro',zero_division=0))
        hist.append({'epoch':epoch,'train_loss':float(np.mean(ls)),'val_patient_macro_f1_at_0.5':vf}); print(f'[strict {modality} fold{fold}] epoch={epoch:03d} loss={np.mean(ls):.4f} valPatientF1={vf:.4f}')
        if vf>best+1e-5:best=vf;best_epoch=epoch;wait=0;torch.save({'state_dict':model.state_dict(),'epoch':epoch},ck)
        else:
            wait+=1
            if wait>=V3_STRICT_PATIENCE:break
    pd.DataFrame(hist).to_csv(ld/f'history_fold{fold}.csv',index=False); model.load_state_dict(torch.load(ck,map_location=cfg.device)['state_dict'])
    vl=v3_predict_strict(model,va,use_tta=cfg.use_tta); T,tau,vrec,info=v3_fit_patient_operating_point(vl); tl=v3_predict_strict(model,te,use_tta=cfg.use_tta); win,rec=v3_apply_patient_operating_point(tl,T,tau,model=f'v3_strict_{modality}',ablation='whole_window_single_stream',fold=fold)
    rec.to_csv(rp,index=False);win.to_csv(wp,index=False);pd.DataFrame([{'fold':fold,'modality':modality,'n_params':parameter_count(model),'best_epoch':best_epoch,'training_time_sec':time.time()-t0,**info}]).to_csv(pp,index=False)
    del model;gc.collect();
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return {'record_pred':rec,'window_pred':win,'params':pd.read_csv(pp)}


def run_v3_strict_modalities(force:bool=V3_FORCE_RERUN):
    v3_build_strict_whole_window_cache(df_binary,overwrite=False); frames=[]; pars=[]
    for modality in V3_STRICT_MODALITIES:
        for fold in sorted(map(int,df_binary.fold.unique())):
            o=v3_train_strict_fold(df_binary,fold,modality,force=force); x=o['record_pred'].copy();x['strict_modality']=modality;frames.append(x);pars.append(o['params'])
    pred=pd.concat(frames,ignore_index=True); pred.to_csv(V3_TABLE_DIR/'Table_V3C_strict_modality_patient_predictions.csv',index=False)
    rows=[]
    for mod,g in pred.groupby('strict_modality'):
        m=v3_pooled_record_metrics(g);lo,hi=v3_record_bootstrap_ci(g,'macro_f1',1000,seed=cfg.seed+sum(map(ord,mod)));rows.append({'modality':mod,**m,'macro_f1_ci_low':lo,'macro_f1_ci_high':hi})
    summ=pd.DataFrame(rows);summ.to_csv(V3_TABLE_DIR/'Table_V3C_strict_modality_summary.csv',index=False)
    return {'record_pred':pred,'summary':summ,'fold_params':pd.concat(pars,ignore_index=True)}


In [29]:

def run_v3_strict_modalities_with_paired_tests(force:bool=V3_FORCE_RERUN):
    strict=run_v3_strict_modalities(force=force)
    if not V3_REFERENCE_OUTPUTS:
        ref=run_v3_reference_model(force=force)
    else:
        ref=V3_REFERENCE_OUTPUTS
    reference=ref['record_pred'].copy(); pred=strict['record_pred'].copy()
    rows=[]
    for mod in V3_STRICT_MODALITIES:
        g=pred[pred.strict_modality==mod].copy()
        perm=v3_paired_permutation_delta(reference,g,cfg.n_permutations,cfg.seed+sum(map(ord,mod)))
        mc=v3_mcnemar(reference,g)
        rows.append({'comparison':f'multimodal_reference_vs_strict_{mod}','strict_modality':mod,
                     'reference_macro_f1':v3_pooled_record_metrics(reference).get('macro_f1'),
                     'strict_macro_f1':v3_pooled_record_metrics(g).get('macro_f1'),
                     'delta_macro_f1_reference_minus_strict':perm['delta_macro_f1'],
                     'permutation_p':perm['p_value'],'mcnemar_p':mc['p_value'],
                     'reference_correct_strict_wrong':mc['a_correct_b_wrong'],'reference_wrong_strict_correct':mc['a_wrong_b_correct'],'n':perm['n']})
    tests=pd.DataFrame(rows)
    tests['permutation_p_holm']=v3_holm(tests.permutation_p.tolist()); tests['mcnemar_p_holm']=v3_holm(tests.mcnemar_p.tolist())
    tests.to_csv(V3_TABLE_DIR/'Table_V3C_strict_modality_paired_tests.csv',index=False)
    strict['summary'].to_csv(V3_TABLE_DIR/'Table_V3C_strict_modality_summary.csv',index=False)
    pred.to_csv(V3_TABLE_DIR/'Table_V3C_strict_modality_patient_predictions.csv',index=False)
    return {**strict,'paired_tests':tests}

V3_STRICT_OUTPUTS=run_v3_strict_modalities_with_paired_tests() if RUN_V3_STRICT_MODALITIES else {}
if V3_STRICT_OUTPUTS:
    display(V3_STRICT_OUTPUTS['summary']); display(V3_STRICT_OUTPUTS['paired_tests'])


V3 strict whole-window cache:   0%|          | 0/1410 [00:00<?, ?it/s]

[strict ecg fold0] epoch=001 loss=0.2248 valPatientF1=0.7083
[strict ecg fold0] epoch=002 loss=0.1009 valPatientF1=0.7083
[strict ecg fold0] epoch=003 loss=0.0674 valPatientF1=0.8444
[strict ecg fold0] epoch=004 loss=0.0594 valPatientF1=0.7083
[strict ecg fold0] epoch=005 loss=0.0672 valPatientF1=0.5714
[strict ecg fold0] epoch=006 loss=0.0582 valPatientF1=0.3636
[strict ecg fold0] epoch=007 loss=0.0407 valPatientF1=0.5714
[strict ecg fold0] epoch=008 loss=0.0140 valPatientF1=0.7083
[strict ecg fold0] epoch=009 loss=0.0523 valPatientF1=0.7083
[strict ecg fold0] epoch=010 loss=0.0209 valPatientF1=0.7083
[strict ecg fold0] epoch=011 loss=0.0193 valPatientF1=0.7083
[strict ecg fold0] epoch=012 loss=0.0287 valPatientF1=0.6500
[strict ecg fold0] epoch=013 loss=0.0281 valPatientF1=0.8444
[strict ecg fold0] epoch=014 loss=0.0087 valPatientF1=0.6500
[strict ecg fold0] epoch=015 loss=0.0197 valPatientF1=0.6500
[strict ecg fold0] epoch=016 loss=0.0076 valPatientF1=0.8444
[strict ecg fold0] epoch

,modality,threshold,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,cohen_kappa,...,recall_AVNRT,f1_AVNRT,support_AVNRT,precision_AVRT,recall_AVRT,f1_AVRT,support_AVRT,n_patients,macro_f1_ci_low,macro_f1_ci_high
0,ecg,0.5,0.535714,0.525641,0.527778,0.525641,0.520422,0.526539,0.053376,0.052083,...,0.384615,0.434783,13.0,0.555556,0.666667,0.606061,15.0,28,0.320387,0.700535
1,iegm,0.5,0.714286,0.702564,0.730994,0.702564,0.700535,0.705118,0.432625,0.413613,...,0.538462,0.636364,13.0,0.684211,0.866667,0.764706,15.0,28,0.535077,0.856410


,comparison,strict_modality,reference_macro_f1,strict_macro_f1,delta_macro_f1_reference_minus_strict,permutation_p,mcnemar_p,reference_correct_strict_wrong,reference_wrong_strict_correct,n,permutation_p_holm,mcnemar_p_holm
0,multimodal_reference_vs_strict_ecg,ecg,0.611111,0.520422,0.090690,0.478704,0.453125,5,2,28,0.957409,0.90625
1,multimodal_reference_vs_strict_iegm,iegm,0.611111,0.700535,-0.089424,0.558088,0.726562,3,5,28,0.957409,0.90625


## Analysis 4 — Automatic ECG-based beat localization rebuilt from raw data, including fallback provenance and tolerance sensitivity


In [30]:

# Independent automatic-localization cache. The wrapper below records whether XQRS or the fallback detector supplied each window's peaks.
_V3_ORIGINAL_DETECT_XQRS = detect_xqrs_rpeaks
V3_AUTO_DETECT_CONTEXT={'row_id':None}
V3_AUTO_DETECT_LOG={}


def v3_detect_xqrs_logged(ecg_scaled:np.ndarray,fs:int,ecg_present:Optional[np.ndarray]=None)->np.ndarray:
    preferred=[ECG_LEADS.index('II'),ECG_LEADS.index('V5'),ECG_LEADS.index('V2'),ECG_LEADS.index('I')]
    if ecg_present is not None:
        candidates=[i for i in preferred if i<len(ecg_present) and bool(ecg_present[i])]
        candidates += [i for i,p in enumerate(ecg_present) if p and i not in candidates]
    else:candidates=preferred+list(range(ecg_scaled.shape[0]))
    lead_idx=candidates[0] if candidates else 0; sig=np.nan_to_num(np.asarray(ecg_scaled[lead_idx],dtype=np.float32),nan=0.0)
    peaks=np.asarray([],dtype=np.int64); method='xqrs'
    if processing is not None:
        try:peaks=processing.xqrs_detect(sig=sig,fs=fs,verbose=False).astype(np.int64)
        except TypeError:
            try:peaks=processing.xqrs_detect(sig=sig,fs=fs).astype(np.int64)
            except Exception:peaks=np.asarray([],dtype=np.int64)
        except Exception:peaks=np.asarray([],dtype=np.int64)
    if len(peaks)<2:
        method='scipy_find_peaks_fallback'
        try:
            distance=int(max(0.20*fs,1)); prominence=max(0.5,float(np.nanstd(sig))*0.35);peaks,_=sps.find_peaks(sig,distance=distance,prominence=prominence);peaks=peaks.astype(np.int64)
        except Exception:peaks=np.asarray([],dtype=np.int64);method='detector_failure'
    rid=V3_AUTO_DETECT_CONTEXT.get('row_id')
    if rid is not None:V3_AUTO_DETECT_LOG[int(rid)]={'row_id':int(rid),'detector_method':method,'lead_index':int(lead_idx),'n_detected_raw':int(len(peaks))}
    return peaks


@contextmanager
def v3_automatic_detection_context(cache_dir:Path=V3_AUTO_CACHE_DIR):
    global detect_xqrs_rpeaks
    old_func=detect_xqrs_rpeaks; old_cache=cfg.cache_dir; old_source=cfg.rpeak_source
    detect_xqrs_rpeaks=v3_detect_xqrs_logged; cfg.cache_dir=str(cache_dir); cfg.rpeak_source='xqrs_only'
    try:yield
    finally:
        detect_xqrs_rpeaks=old_func; cfg.cache_dir=old_cache; cfg.rpeak_source=old_source; V3_AUTO_DETECT_CONTEXT['row_id']=None


def v3_build_automatic_binary_cache(df:pd.DataFrame,overwrite:bool=False):
    V3_AUTO_CACHE_DIR.mkdir(parents=True,exist_ok=True); failures=[]
    existing_log=V3_LOG_DIR/'automatic_peak_detector_provenance.csv'
    oldlog=pd.read_csv(existing_log) if existing_log.exists() and not overwrite else pd.DataFrame()
    known=set(oldlog.row_id.astype(int)) if len(oldlog) and 'row_id' in oldlog else set()
    with v3_automatic_detection_context():
        for _,row in tqdm(df.iterrows(),total=len(df),desc='Automatic ECG R-peak cache'):
            fp=cache_file_for_row(int(row.row_id)); need_cache=overwrite or not fp.exists(); need_log=int(row.row_id) not in known
            if not need_cache and not need_log:continue
            try:
                V3_AUTO_DETECT_CONTEXT['row_id']=int(row.row_id); z=process_window(row)
                if need_cache:np.savez_compressed(fp,**z)
            except Exception as e:failures.append({'row_id':int(row.row_id),'record':row.record,'error':repr(e)})
    lognew=pd.DataFrame(list(V3_AUTO_DETECT_LOG.values()))
    log=pd.concat([oldlog,lognew],ignore_index=True).drop_duplicates('row_id',keep='last') if len(oldlog) or len(lognew) else pd.DataFrame()
    if len(log):log.to_csv(existing_log,index=False)
    if failures:pd.DataFrame(failures).to_csv(V3_LOG_DIR/'automatic_cache_failures.csv',index=False)
    return log,pd.DataFrame(failures)


def v3_match_peaks(ref:np.ndarray,det:np.ndarray,tol_samples:int)->Dict[str,Any]:
    ref=np.sort(np.asarray(ref,int));det=np.sort(np.asarray(det,int));used=set();errs=[];matched=0
    for r in ref:
        if not len(det):break
        order=np.argsort(np.abs(det-r));j=next((int(k) for k in order if int(k) not in used and abs(int(det[k])-int(r))<=tol_samples),None)
        if j is not None:used.add(j);matched+=1;errs.append(int(det[j])-int(r))
    return {'n_ref':int(len(ref)),'n_det':int(len(det)),'matched':int(matched),'sensitivity':float(matched/len(ref)) if len(ref) else np.nan,'ppv':float(matched/len(det)) if len(det) else np.nan,'median_abs_error_samples':float(np.median(np.abs(errs))) if errs else np.nan}


def run_v3_localization_audit():
    prov,fail=v3_build_automatic_binary_cache(df_binary,overwrite=V3_FORCE_RERUN)
    rows=[]
    for _,row in tqdm(df_binary.iterrows(),total=len(df_binary),desc='Peak matching'):
        refp=BASE_ANNOTATION_CACHE/f"row_{int(row.row_id):07d}.npz"; autop=V3_AUTO_CACHE_DIR/f"row_{int(row.row_id):07d}.npz"
        # cache_file_for_row naming can differ; resolve through the helper under each cache context.
        with v3_cache_context(BASE_ANNOTATION_CACHE):refp=cache_file_for_row(int(row.row_id))
        with v3_cache_context(V3_AUTO_CACHE_DIR):autop=cache_file_for_row(int(row.row_id))
        if not refp.exists() or not autop.exists():continue
        rz=np.load(refp,allow_pickle=False);az=np.load(autop,allow_pickle=False);ref=rz['rpeaks'];det=az['rpeaks']
        for tol_ms in V3_PEAK_MATCH_TOLERANCES_MS:
            m=v3_match_peaks(ref,det,int(round(tol_ms*cfg.target_fs/1000.0)))
            rows.append({'row_id':int(row.row_id),'record':row.record,'label':int(row.label),'tolerance_ms':float(tol_ms),**m,'median_abs_error_ms':float(m['median_abs_error_samples']*1000.0/cfg.target_fs) if pd.notna(m['median_abs_error_samples']) else np.nan})
    win=pd.DataFrame(rows);win=win.merge(prov,on='row_id',how='left') if len(prov) else win
    win.to_csv(V3_TABLE_DIR/'Table_V3D_localization_window_metrics.csv',index=False)
    pats=[]
    for (tol,record),g in win.groupby(['tolerance_ms','record']):
        pats.append({'tolerance_ms':tol,'record':record,'label':int(g.label.mode().iloc[0]),'n_windows':len(g),'peak_sensitivity':float(np.average(g.sensitivity.fillna(0),weights=np.maximum(g.n_ref,1))),'peak_ppv':float(np.average(g.ppv.fillna(0),weights=np.maximum(g.n_det,1))),'median_window_abs_error_ms':float(g.median_abs_error_ms.median()),'fallback_window_fraction':float(np.mean(g.detector_method!='xqrs')) if 'detector_method' in g else np.nan})
    pat=pd.DataFrame(pats);pat.to_csv(V3_TABLE_DIR/'Table_V3D_localization_patient_metrics.csv',index=False)
    summary=[]
    for tol,g in win.groupby('tolerance_ms'):
        nref=int(g.n_ref.sum());ndet=int(g.n_det.sum());mat=int(g.matched.sum())
        summary.append({'tolerance_ms':float(tol),'n_windows':int(len(g)),'n_reference_peaks':nref,'n_detected_peaks':ndet,'matched_peaks':mat,'pooled_sensitivity':float(mat/nref) if nref else np.nan,'pooled_ppv':float(mat/ndet) if ndet else np.nan,'median_window_abs_error_ms':float(g.median_abs_error_ms.median()),'fallback_window_fraction':float(np.mean(g.detector_method!='xqrs')) if 'detector_method' in g else np.nan})
    summ=pd.DataFrame(summary);summ.to_csv(V3_TABLE_DIR/'Table_V3D_localization_summary_by_tolerance.csv',index=False)
    return {'provenance':prov,'window':win,'patient':pat,'summary':summ}

V3_LOCALIZATION_OUTPUTS=run_v3_localization_audit() if RUN_V3_AUTOMATIC_LOCALIZATION else {}
if V3_LOCALIZATION_OUTPUTS:display(V3_LOCALIZATION_OUTPUTS['summary'])


Automatic ECG R-peak cache:   0%|          | 0/1410 [00:00<?, ?it/s]

Peak matching:   0%|          | 0/1410 [00:00<?, ?it/s]

,tolerance_ms,n_windows,n_reference_peaks,n_detected_peaks,matched_peaks,pooled_sensitivity,pooled_ppv,median_window_abs_error_ms,fallback_window_fraction
0,50.0,1410,40287,39948,39145,0.971653,0.979899,6.0,0.0
1,100.0,1410,40287,39948,39570,0.982203,0.990538,6.0,0.0
2,150.0,1410,40287,39948,39700,0.985430,0.993792,6.0,0.0


## Analysis 5 — Automatic-localization classification with and without annotation-derived pacing features


In [31]:

# Extend the input configuration with a pacing-removal condition.
_V3_BASE_ABLATION_FN=apply_input_ablation_np

def apply_input_ablation_np(ecg,iegm,beat_mask,ecg_ch_mask,iegm_ch_mask,features,mech,mode='window_level'):
    if str(mode)!='no_annotation_pacing':return _V3_BASE_ABLATION_FN(ecg,iegm,beat_mask,ecg_ch_mask,iegm_ch_mask,features,mech,mode)
    ecg=np.copy(ecg).astype(np.float32);iegm=np.copy(iegm).astype(np.float32);ecg_ch_mask=np.copy(ecg_ch_mask).astype(bool);iegm_ch_mask=np.copy(iegm_ch_mask).astype(bool);features=np.copy(features).astype(np.float32);mech=np.copy(mech).astype(np.float32)
    _zero_feature_index(features,1); pace_idx=MECH_IDX.get('pace_frac_window')
    if pace_idx is not None:
        mech[...,pace_idx]=0.0; s=MECH_FEAT_SL.start+4*pace_idx;features[...,s:s+4]=0.0
    return ecg,iegm,ecg_ch_mask,iegm_ch_mask,features,mech


def run_v3_automatic_pacing_comparison(force:bool=V3_FORCE_RERUN):
    # Ensure automatic binary cache exists and then keep every loader/pretraining call inside the automatic-cache context.
    v3_build_automatic_binary_cache(df_binary,overwrite=False)
    frames=[];pars=[]
    with v3_automatic_detection_context():
        for fold in sorted(map(int,df_binary.fold.unique())):
            outer_train=df_binary[df_binary.fold!=fold].reset_index(drop=True);outer_test=df_binary[df_binary.fold==fold].reset_index(drop=True);inner_train,inner_val=make_inner_split(outer_train,seed=cfg.seed+777+fold,n_inner_splits=cfg.n_inner_splits)
            seed_value=150000+fold*1000+77
            ssl_ck=v3_nested_ssl_pretrain(fold,inner_train,inner_val,outer_test,seed_value=seed_value,tag='automatic_localization',force=force)
            for abl in [PROPOSED_ABLATION,'no_annotation_pacing']:
                o=v3_train_patient_endpoint_fold(df_binary,fold,model_name='automatic_localization',ablation=abl,init_ckpt=ssl_ck,supervised_lr=V3_MATCHED_SUPERVISED_LR,experiment_tag='automatic_localization_pacing',seed_value=seed_value,force=force)
                x=o['record_pred'].copy();x['analysis_ablation']=abl;frames.append(x);pars.append(o['params'].assign(analysis_ablation=abl))
    pred=pd.concat(frames,ignore_index=True);pred.to_csv(V3_TABLE_DIR/'Table_V3E_automatic_pacing_patient_predictions.csv',index=False)
    summ=[]
    for abl,g in pred.groupby('analysis_ablation'):
        m=v3_pooled_record_metrics(g);lo,hi=v3_record_bootstrap_ci(g,'macro_f1',cfg.n_bootstrap,cfg.seed+sum(map(ord,abl)));summ.append({'configuration':abl,**m,'macro_f1_ci_low':lo,'macro_f1_ci_high':hi})
    summ=pd.DataFrame(summ);summ.to_csv(V3_TABLE_DIR/'Table_V3E_automatic_pacing_summary.csv',index=False)
    a=pred[pred.analysis_ablation==PROPOSED_ABLATION];b=pred[pred.analysis_ablation=='no_annotation_pacing'];perm=v3_paired_permutation_delta(a,b,cfg.n_permutations,cfg.seed);mc=v3_mcnemar(a,b)
    paired=pd.DataFrame([{'comparison':'automatic_full_input_minus_no_annotation_pacing','delta_macro_f1':perm['delta_macro_f1'],'permutation_p':perm['p_value'],'mcnemar_p':mc['p_value'],'full_correct_no_pacing_wrong':mc['a_correct_b_wrong'],'full_wrong_no_pacing_correct':mc['a_wrong_b_correct'],'n':perm['n']}]);paired.to_csv(V3_TABLE_DIR/'Table_V3E_automatic_pacing_paired_tests.csv',index=False)
    pd.concat(pars,ignore_index=True).to_csv(V3_TABLE_DIR/'Table_V3E_automatic_pacing_fold_parameters.csv',index=False)
    return {'record_pred':pred,'summary':summ,'paired':paired}

V3_PACING_OUTPUTS=run_v3_automatic_pacing_comparison() if RUN_V3_PACING_REMOVAL else {}
if V3_PACING_OUTPUTS:
    display(V3_PACING_OUTPUTS['summary']);display(V3_PACING_OUTPUTS['paired'])


Automatic ECG R-peak cache:   0%|          | 0/1410 [00:00<?, ?it/s]

Building token cache:   0%|          | 0/9522 [00:00<?, ?it/s]

[nested SSL automatic_localization partition 0] epoch=001 loss=120.0924 skipped=0
[nested SSL automatic_localization partition 0] epoch=002 loss=119.7932 skipped=0
[nested SSL automatic_localization partition 0] epoch=003 loss=117.6496 skipped=0
[nested SSL automatic_localization partition 0] epoch=004 loss=119.7258 skipped=0
[nested SSL automatic_localization partition 0] epoch=005 loss=115.0459 skipped=0
[nested SSL automatic_localization partition 0] epoch=006 loss=120.2632 skipped=0
[nested SSL automatic_localization partition 0] epoch=007 loss=117.7268 skipped=0
[nested SSL automatic_localization partition 0] epoch=008 loss=115.9899 skipped=0
[nested SSL automatic_localization partition 0] epoch=009 loss=111.8060 skipped=0
[nested SSL automatic_localization partition 0] epoch=010 loss=113.5472 skipped=0
[nested SSL automatic_localization partition 0] epoch=011 loss=113.4486 skipped=0
[nested SSL automatic_localization partition 0] epoch=012 loss=113.1804 skipped=0
[nested SSL auto

Building token cache:   0%|          | 0/8898 [00:00<?, ?it/s]

[nested SSL automatic_localization partition 1] epoch=001 loss=152.4107 skipped=0
[nested SSL automatic_localization partition 1] epoch=002 loss=146.4540 skipped=0
[nested SSL automatic_localization partition 1] epoch=003 loss=145.4110 skipped=0
[nested SSL automatic_localization partition 1] epoch=004 loss=151.5982 skipped=0
[nested SSL automatic_localization partition 1] epoch=005 loss=143.4727 skipped=0
[nested SSL automatic_localization partition 1] epoch=006 loss=148.9013 skipped=0
[nested SSL automatic_localization partition 1] epoch=007 loss=147.8862 skipped=0
[nested SSL automatic_localization partition 1] epoch=008 loss=148.0837 skipped=0
[nested SSL automatic_localization partition 1] epoch=009 loss=146.7950 skipped=0
[nested SSL automatic_localization partition 1] epoch=010 loss=145.6571 skipped=0
[nested SSL automatic_localization partition 1] epoch=011 loss=141.2840 skipped=0
[nested SSL automatic_localization partition 1] epoch=012 loss=144.5916 skipped=0
[nested SSL auto

...

[nested SSL automatic_localization partition 3] epoch=001 loss=125.8400 skipped=0
[nested SSL automatic_localization partition 3] epoch=002 loss=125.0485 skipped=0
[nested SSL automatic_localization partition 3] epoch=003 loss=123.2920 skipped=0
[nested SSL automatic_localization partition 3] epoch=004 loss=122.6538 skipped=0
[nested SSL automatic_localization partition 3] epoch=005 loss=122.8699 skipped=0
[nested SSL automatic_localization partition 3] epoch=006 loss=120.1995 skipped=0
[nested SSL automatic_localization partition 3] epoch=007 loss=119.5446 skipped=0
[nested SSL automatic_localization partition 3] epoch=008 loss=116.4917 skipped=0
[nested SSL automatic_localization partition 3] epoch=009 loss=119.9703 skipped=0
[nested SSL automatic_localization partition 3] epoch=010 loss=118.9201 skipped=0
[nested SSL automatic_localization partition 3] epoch=011 loss=118.1725 skipped=0
[nested SSL automatic_localization partition 3] epoch=012 loss=117.7046 skipped=0
[nested SSL auto

Building token cache:   0%|          | 0/7257 [00:00<?, ?it/s]

[nested SSL automatic_localization partition 4] epoch=001 loss=125.7483 skipped=0
[nested SSL automatic_localization partition 4] epoch=002 loss=128.3012 skipped=0
[nested SSL automatic_localization partition 4] epoch=003 loss=125.8323 skipped=0
[nested SSL automatic_localization partition 4] epoch=004 loss=122.2651 skipped=0
[nested SSL automatic_localization partition 4] epoch=005 loss=116.9219 skipped=0
[nested SSL automatic_localization partition 4] epoch=006 loss=124.2078 skipped=0
[nested SSL automatic_localization partition 4] epoch=007 loss=127.5436 skipped=0
[nested SSL automatic_localization partition 4] epoch=008 loss=120.4420 skipped=0
[nested SSL automatic_localization partition 4] epoch=009 loss=119.7995 skipped=0
[nested SSL automatic_localization partition 4] epoch=010 loss=119.5751 skipped=0
[nested SSL automatic_localization partition 4] epoch=011 loss=119.2166 skipped=0
[nested SSL automatic_localization partition 4] epoch=012 loss=120.3018 skipped=0
[nested SSL auto

,configuration,threshold,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,cohen_kappa,...,recall_AVNRT,f1_AVNRT,support_AVNRT,precision_AVRT,recall_AVRT,f1_AVRT,support_AVRT,n_patients,macro_f1_ci_low,macro_f1_ci_high
0,no_annotation_pacing,0.5,0.714286,0.702564,0.730994,0.702564,0.700535,0.705118,0.432625,0.413613,...,0.538462,0.636364,13.0,0.684211,0.866667,0.764706,15.0,28,0.504762,0.863415
1,window_level,0.5,0.714286,0.702564,0.730994,0.702564,0.700535,0.705118,0.432625,0.413613,...,0.538462,0.636364,13.0,0.684211,0.866667,0.764706,15.0,28,0.504643,0.856410


,comparison,delta_macro_f1,permutation_p,mcnemar_p,full_correct_no_pacing_wrong,full_wrong_no_pacing_correct,n
0,automatic_full_input_minus_no_annotation_pacing,0.0,1.0,1.0,0,0,28


## Analysis 6 — Repeated controlled self-supervised initialization versus random initialization


In [32]:

def run_v3_repeated_matched_initialization(force:bool=V3_FORCE_RERUN):
    frames=[];params=[]
    for seed_base in V3_MATCHED_SEED_BASES:
        pred,par=v3_run_matched_seed(int(seed_base),include_random_arm=True,tag_prefix='matched_initialization',force=force);frames.append(pred);params.append(par)
    allp=pd.concat(frames,ignore_index=True);allp.to_csv(V3_TABLE_DIR/'Table_V3F_matched_initialization_all_patient_predictions.csv',index=False)
    pd.concat(params,ignore_index=True).to_csv(V3_TABLE_DIR/'Table_V3F_matched_initialization_fold_parameters.csv',index=False)
    rows=[];tests=[]
    for seed_base,gseed in allp.groupby('seed_base'):
        for arm,g in gseed.groupby('matched_arm'):
            m=v3_pooled_record_metrics(g);lo,hi=v3_record_bootstrap_ci(g,'macro_f1',cfg.n_bootstrap,seed=cfg.seed+int(seed_base)+sum(map(ord,arm)));rows.append({'seed_base':int(seed_base),'arm':arm,**m,'macro_f1_ci_low':lo,'macro_f1_ci_high':hi})
        a=gseed[gseed.matched_arm=='ssl_initialized'];b=gseed[gseed.matched_arm=='random_initialized'];perm=v3_paired_permutation_delta(a,b,cfg.n_permutations,cfg.seed+int(seed_base));mc=v3_mcnemar(a,b);tests.append({'seed_base':int(seed_base),'delta_macro_f1_ssl_minus_random':perm['delta_macro_f1'],'permutation_p':perm['p_value'],'mcnemar_p':mc['p_value'],'ssl_correct_random_wrong':mc['a_correct_b_wrong'],'ssl_wrong_random_correct':mc['a_wrong_b_correct'],'n':perm['n']})
    summary=pd.DataFrame(rows);testdf=pd.DataFrame(tests)
    summary.to_csv(V3_TABLE_DIR/'Table_V3F_matched_initialization_by_seed.csv',index=False);testdf.to_csv(V3_TABLE_DIR/'Table_V3F_matched_initialization_paired_tests_by_seed.csv',index=False)
    pivot=summary.pivot(index='seed_base',columns='arm',values='macro_f1').reset_index();
    if {'ssl_initialized','random_initialized'}.issubset(pivot.columns):pivot['delta_macro_f1']=pivot.ssl_initialized-pivot.random_initialized
    aggregate=pd.DataFrame([{'n_seed_schedules':int(len(pivot)),'mean_ssl_macro_f1':float(pivot.ssl_initialized.mean()) if 'ssl_initialized' in pivot else np.nan,'sd_ssl_macro_f1':float(pivot.ssl_initialized.std(ddof=1)) if len(pivot)>1 and 'ssl_initialized' in pivot else np.nan,'mean_random_macro_f1':float(pivot.random_initialized.mean()) if 'random_initialized' in pivot else np.nan,'sd_random_macro_f1':float(pivot.random_initialized.std(ddof=1)) if len(pivot)>1 and 'random_initialized' in pivot else np.nan,'mean_delta_macro_f1_ssl_minus_random':float(pivot.delta_macro_f1.mean()) if 'delta_macro_f1' in pivot else np.nan,'sd_delta_macro_f1':float(pivot.delta_macro_f1.std(ddof=1)) if len(pivot)>1 and 'delta_macro_f1' in pivot else np.nan}])
    aggregate.to_csv(V3_TABLE_DIR/'Table_V3F_matched_initialization_across_seed_schedules.csv',index=False)
    return {'predictions':allp,'summary':summary,'paired_tests':testdf,'aggregate':aggregate}

V3_MATCHED_OUTPUTS=run_v3_repeated_matched_initialization() if RUN_V3_REPEATED_MATCHED_SSL else {}
if V3_MATCHED_OUTPUTS:
    display(V3_MATCHED_OUTPUTS['summary']);display(V3_MATCHED_OUTPUTS['paired_tests']);display(V3_MATCHED_OUTPUTS['aggregate'])


[fold 0] Reusing completed outputs: ssl_initialized/window_level
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=001 loss=0.2282 valPatientF1=0.3000
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=002 loss=0.1013 valPatientF1=0.3000
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=003 loss=0.0410 valPatientF1=0.7083
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=004 loss=0.0386 valPatientF1=0.8571
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=005 loss=0.0156 valPatientF1=0.8571
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=006 loss=0.0219 valPatientF1=0.5333
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=007 loss=0.0199 valPatientF1=0.8571
[matched_initialization/seed_88000/random_initialized/window_level/fold0] epoch=008 loss=0.0201 valPatientF1=0.8571
[matche

Building token cache:   0%|          | 0/9522 [00:00<?, ?it/s]

[nested SSL matched_initialization_99000 partition 0] epoch=001 loss=119.8178 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=002 loss=117.6724 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=003 loss=115.5999 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=004 loss=114.5382 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=005 loss=112.4812 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=006 loss=117.5349 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=007 loss=114.7338 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=008 loss=112.8729 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=009 loss=110.1736 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=010 loss=111.7047 skipped=0
[nested SSL matched_initialization_99000 partition 0] epoch=011 loss=112.4011 skipped=0
[nested SSL matched_initializati

Building token cache:   0%|          | 0/8898 [00:00<?, ?it/s]

[nested SSL matched_initialization_99000 partition 1] epoch=001 loss=149.9498 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=002 loss=148.9417 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=003 loss=151.6981 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=004 loss=153.5578 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=005 loss=144.4357 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=006 loss=146.4388 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=007 loss=148.2126 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=008 loss=142.3208 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=009 loss=145.3113 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=010 loss=148.1745 skipped=0
[nested SSL matched_initialization_99000 partition 1] epoch=011 loss=148.1139 skipped=0
[nested SSL matched_initializati

...

Building token cache:   0%|          | 0/7257 [00:00<?, ?it/s]

[nested SSL matched_initialization_110000 partition 4] epoch=001 loss=127.9504 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=002 loss=126.6032 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=003 loss=125.3211 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=004 loss=120.7773 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=005 loss=118.1387 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=006 loss=122.8022 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=007 loss=124.0132 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=008 loss=122.0187 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=009 loss=122.5802 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=010 loss=117.4657 skipped=0
[nested SSL matched_initialization_110000 partition 4] epoch=011 loss=121.8258 skipped=0
[nested SSL matched_i

,seed_base,arm,threshold,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mcc,...,recall_AVNRT,f1_AVNRT,support_AVNRT,precision_AVRT,recall_AVRT,f1_AVRT,support_AVRT,n_patients,macro_f1_ci_low,macro_f1_ci_high
0,88000,random_initialized,0.5,0.607143,0.592308,0.612500,0.592308,0.580952,0.588435,0.203810,...,0.384615,0.476190,13.0,0.600000,0.800000,0.685714,15.0,28,0.372859,0.766809
1,88000,ssl_initialized,0.5,0.642857,0.625641,0.666667,0.625641,0.611111,0.619048,0.289414,...,0.384615,0.500000,13.0,0.619048,0.866667,0.722222,15.0,28,0.416453,0.788839
2,99000,random_initialized,0.5,0.607143,0.587179,0.628788,0.587179,0.561878,0.571937,0.211921,...,0.307692,0.421053,13.0,0.590909,0.866667,0.702703,15.0,28,0.366516,0.741899
3,99000,ssl_initialized,0.5,0.642857,0.625641,0.666667,0.625641,0.611111,0.619048,0.289414,...,0.384615,0.500000,13.0,0.619048,0.866667,0.722222,15.0,28,0.426901,0.781334
4,110000,random_initialized,0.5,0.607143,0.597436,0.605556,0.597436,0.594203,0.599379,0.202829,...,0.461538,0.521739,13.0,0.611111,0.733333,0.666667,15.0,28,0.387458,0.775401
5,110000,ssl_initialized,0.5,0.750000,0.741026,0.761111,0.741026,0.741765,0.745059,0.501735,...,0.615385,0.695652,13.0,0.722222,0.866667,0.787879,15.0,28,0.569063,0.891613


,seed_base,delta_macro_f1_ssl_minus_random,permutation_p,mcnemar_p,ssl_correct_random_wrong,ssl_wrong_random_correct,n
0,88000,0.030159,0.920816,1.00000,4,3,28
1,99000,0.049233,0.794441,1.00000,4,3,28
2,110000,0.147563,0.231354,0.21875,5,1,28


,n_seed_schedules,mean_ssl_macro_f1,sd_ssl_macro_f1,mean_random_macro_f1,sd_random_macro_f1,mean_delta_macro_f1_ssl_minus_random,sd_delta_macro_f1
0,3,0.654663,0.075433,0.579011,0.01625,0.075652,0.063003


## Analysis 7 — External-validation readiness check


In [33]:

def run_v3_external_validation_check():
    status={'external_data_dir':V3_EXTERNAL_DATA_DIR,'status':'not_available','message':'A genuinely independent synchronized cohort was not supplied; no external-validation estimate is generated.'}
    if V3_EXTERNAL_DATA_DIR:
        p=Path(V3_EXTERNAL_DATA_DIR).expanduser()
        if p.exists():
            status={'external_data_dir':str(p.resolve()),'status':'data_path_present_requires_schema_review','message':'An external path is present. Before inference, verify channel naming, annotation/reference definitions, sampling metadata, and cohort independence. This notebook does not silently treat an unverified folder as an external validation cohort.'}
        else:
            status={'external_data_dir':V3_EXTERNAL_DATA_DIR,'status':'path_not_found','message':'Configured external cohort path does not exist.'}
    out=pd.DataFrame([status]);out.to_csv(V3_TABLE_DIR/'Table_V3G_external_validation_status.csv',index=False);return out

V3_EXTERNAL_STATUS=run_v3_external_validation_check() if RUN_V3_EXTERNAL_VALIDATION_CHECK else pd.DataFrame()
display(V3_EXTERNAL_STATUS)


,external_data_dir,status,message
0,,not_available,A genuinely independent synchronized cohort wa...


## Consolidated manuscript-ready export and completeness audit


In [34]:

def v3_read(path:Path)->pd.DataFrame:
    return pd.read_csv(path) if path.exists() and path.stat().st_size>0 else pd.DataFrame()


def export_v3_results():
    table_map={
        'ReferenceSummary':V3_TABLE_DIR/'Table_V3A_reference_patient_level_operating_point_summary.csv',
        'ReferenceFoldParams':V3_TABLE_DIR/'Table_V3A_reference_fold_parameters.csv',
        'ClinicalAudit':V3_TABLE_DIR/'Table_V3B_clinical_reference_concordance.csv',
        'ClinicalAuditSummary':V3_TABLE_DIR/'Table_V3B_clinical_reference_concordance_summary.csv',
        'StrictSummary':V3_TABLE_DIR/'Table_V3C_strict_modality_summary.csv',
        'StrictPaired':V3_TABLE_DIR/'Table_V3C_strict_modality_paired_tests.csv',
        'LocalizationSummary':V3_TABLE_DIR/'Table_V3D_localization_summary_by_tolerance.csv',
        'LocalizationPatient':V3_TABLE_DIR/'Table_V3D_localization_patient_metrics.csv',
        'AutoPacingSummary':V3_TABLE_DIR/'Table_V3E_automatic_pacing_summary.csv',
        'AutoPacingPaired':V3_TABLE_DIR/'Table_V3E_automatic_pacing_paired_tests.csv',
        'MatchedBySeed':V3_TABLE_DIR/'Table_V3F_matched_initialization_by_seed.csv',
        'MatchedPaired':V3_TABLE_DIR/'Table_V3F_matched_initialization_paired_tests_by_seed.csv',
        'MatchedAcrossSeeds':V3_TABLE_DIR/'Table_V3F_matched_initialization_across_seed_schedules.csv',
        'ExternalStatus':V3_TABLE_DIR/'Table_V3G_external_validation_status.csv',
    }
    checklist=[]
    for name,path in table_map.items():checklist.append({'analysis':name,'file':str(path),'available':bool(path.exists() and path.stat().st_size>0)})
    check=pd.DataFrame(checklist);check.to_csv(V3_TABLE_DIR/'V3_completeness_audit.csv',index=False)
    xlsx=V3_ROOT/'V3_final_validation_results.xlsx'
    with pd.ExcelWriter(xlsx) as w:
        for sheet,path in table_map.items():
            d=v3_read(path)
            if d.empty:d=pd.DataFrame([{'status':'not_available','expected_file':str(path)}])
            d.to_excel(w,sheet_name=sheet[:31],index=False)
        check.to_excel(w,sheet_name='CompletenessAudit',index=False);pd.DataFrame([V3_MANIFEST]).to_excel(w,sheet_name='Manifest',index=False)
    md=['# Final validation results','',f'Output root: `{V3_ROOT}`','']
    for sheet,path in table_map.items():
        d=v3_read(path);md.append(f'## {sheet}');md.append(d.to_markdown(index=False) if len(d) else f'Not available: `{path.name}`');md.append('')
    md+=['## Completeness audit',check.to_markdown(index=False),'','## Interpretation boundary','All analyses are internal unless an independently collected synchronized external cohort is explicitly supplied and verified. Repeated cross-validation and repeated seed schedules do not constitute external validation.','']
    report=V3_ROOT/'V3_final_validation_report.md';report.write_text('\n'.join(md),encoding='utf-8')
    print('Wrote:',xlsx);print('Wrote:',report);display(check)
    return xlsx,report,check

V3_RESULTS_WORKBOOK,V3_REPORT,V3_COMPLETENESS=export_v3_results()


Wrote: C:\Users\hasee\V3_AV_OUT\V3_final_validation_results.xlsx
Wrote: C:\Users\hasee\V3_AV_OUT\V3_final_validation_report.md


,analysis,file,available
0,ReferenceSummary,C:\Users\hasee\V3_AV_OUT\tables\Table_V3A_refe...,True
1,ReferenceFoldParams,C:\Users\hasee\V3_AV_OUT\tables\Table_V3A_refe...,True
2,ClinicalAudit,C:\Users\hasee\V3_AV_OUT\tables\Table_V3B_clin...,True
3,ClinicalAuditSummary,C:\Users\hasee\V3_AV_OUT\tables\Table_V3B_clin...,True
4,StrictSummary,C:\Users\hasee\V3_AV_OUT\tables\Table_V3C_stri...,True
5,StrictPaired,C:\Users\hasee\V3_AV_OUT\tables\Table_V3C_stri...,True
6,LocalizationSummary,C:\Users\hasee\V3_AV_OUT\tables\Table_V3D_loca...,True
7,LocalizationPatient,C:\Users\hasee\V3_AV_OUT\tables\Table_V3D_loca...,True
8,AutoPacingSummary,C:\Users\hasee\V3_AV_OUT\tables\Table_V3E_auto...,True
9,AutoPacingPaired,C:\Users\hasee\V3_AV_OUT\tables\Table_V3E_auto...,True


## Manuscript integration map

Use only values generated by the corresponding completed V3 table.

- **Table_V3A**: patient-level operating-point summary and fold parameters.
- **Table_V3B**: concordance between the annotation-derived patient reference and source clinical diagnosis, when the source metadata supports this audit.
- **Table_V3C**: strict ECG-only / IEGM-only performance plus paired permutation and exact McNemar comparisons versus the matched multimodal reference; Holm-adjusted values are included.
- **Table_V3D**: automatic R-peak localization accuracy, detector-fallback frequency, and tolerance sensitivity at 50/100/150 ms.
- **Table_V3E**: automatic-localization classification with and without annotation-derived pacing, including paired inference.
- **Table_V3F**: repeated matched SSL-versus-random-initialization results across seed schedules.
- **Table_V3G**: external-validation availability status. This is not a performance result unless an independent cohort is supplied and verified.

Do not combine a point estimate from one analysis with a confidence interval or p-value from another analysis.
